# Import Library

In [109]:
import cx_Oracle
import pandas as pd
import numpy as np
import warnings
import json
from datetime import date
import os.path
import html
import reinfdicts

warnings.filterwarnings("ignore")

hostname =  "ahqtpd1" #"akrdbttoptest"
port =   1541 #1586
username = "misdba" # misdba
password = "mmilk" # mmilk
service_name= "ttoper" #"ttoptest"

# SQL runner function

In [110]:
def executeQuery(queryStr):
    try:
        dsn = cx_Oracle.makedsn(hostname,port, service_name)
        con = cx_Oracle.connect(username, password, dsn)
        cursor = con.cursor()
        cursor.execute(queryStr)
        columns = [col[0] for col in cursor.description]
        result = cursor.fetchall()
        df = pd.DataFrame(result, columns=columns)
        cursor.close()
        con.close()
        return df
    except cx_Oracle.Error as error:
        print('Error in db connection.: ', error)

# GPS Extraction logic by DD

In [ ]:
def base_data_code():
    df = executeQuery("""
    select * from 
    (select mst.pgm_id, a.hse_id, a.hse_measure_id, 
    case when a.hse_measure_id = 90 then 'MSDS' when  a.hse_measure_id = 91 then 'TSCA' 
    WHEN a.hse_measure_id = 110 then 'CEPAD'
    WHEN a.hse_measure_id = 111 then 'CEPAN'
    WHEN a.hse_measure_id = 112 then 'ELINCS'
    WHEN a.hse_measure_id = 113 then 'EINECS'
    WHEN a.hse_measure_id = 114 then 'KKDIK'
    WHEN a.hse_measure_id = 116 then 'ENCS'
    WHEN a.hse_measure_id = 203 then 'ISHL'
    WHEN a.hse_measure_id = 204 then 'IECSA'
    WHEN a.hse_measure_id = 207 then 'SERB'
    WHEN a.hse_measure_id = 208 then 'SOAF'
    WHEN a.hse_measure_id = 209 then 'IND'
    WHEN a.hse_measure_id = 210 then 'THAI'
    WHEN a.hse_measure_id = 211 then 'JAPAN'
    end as CODE,  
    a.measure_status,  
    mst.prop_plant_trial_id, 
    mpm.plant_name AS PROP_PLANT_FOR_TRIAL,  
    mrm1.region_name as prop_region,
    api_mst.material_code_id
    --, rdtl.pi_pgm_reg_map_id
    ,case when mrm.region_name = 'NA' then 'North America' else mrm.region_name end as REGION_FOR_SOS_APPROVAL
    from
    (select  core.code_catg,dtl.hse_measure_id, core.code, core.code_desc, dtl.hse_id, dtl.measure_status, dtl.measure_cmnts, dtl.created_by
    from MIS_CORE_CODE_MST core, mis_ap_hse_pgm_dtl dtl where core.code_id= dtl.hse_measure_id) a
    left join mis_ap_hse_pgm_mst mst on mst.hse_id = a.hse_id
    left join MIS_GOODYEAR_PLANT_MST mpm on mpm.gy_plant_id = mst.prop_plant_trial_id
    left join MIS_AP_PI_PGM_MST api_mst on api_mst.pgm_id = mst.pgm_id
    left join MIS_AP_PI_PGM_REG_DTL rdtl on rdtl.ap_pgm_id = api_mst.ap_pgm_id
    left join MIS_REGION_MST mrm on mrm.region_id = rdtl.region_id
    left join MIS_SOS_MAT_MST mat_mst on mat_mst.material_code_id = api_mst.material_code_id
    left join MIS_REGION_MST mrm1 on mrm1.region_id = mpm.region_id
    order by 1,2,3,4)
    WHERE CODE is not null
    """)

    df = pd.DataFrame(df).drop_duplicates(keep='first')
    df.to_csv('Input_CSVs/MIS_CODE.csv', escapechar='\\', doublequote=False)

    return df



def base_data_comment():
    df = executeQuery(
        """
        select mst.pgm_id, mst.hse_id,a.HSE_MEASURE_ID, a.code AS TABLE_CODE, a.code_desc,
        case when a.hse_measure_id = 90 then 'MSDS' when  a.hse_measure_id = 91 then 'TSCA' 
        WHEN a.hse_measure_id = 110 then 'CEPAD'
        WHEN a.hse_measure_id = 111 then 'CEPAN'
        WHEN a.hse_measure_id = 112 then 'ELINCS'
        WHEN a.hse_measure_id = 113 then 'EINECS'
        WHEN a.hse_measure_id = 114 then 'KKDIK'
        WHEN a.hse_measure_id = 202 then 'ENCS'
        WHEN a.hse_measure_id = 205 then 'ISHL'
        WHEN a.hse_measure_id = 211 then 'IECSA'
        WHEN a.hse_measure_id = 115 then 'SERB'
        WHEN a.hse_measure_id = 116 then 'SOAF'
        WHEN a.hse_measure_id = 209 then 'IND'
        WHEN a.hse_measure_id = 210 then 'THAI'
        WHEN a.hse_measure_id = 206 then 'JAPAN'
        end as CODE, a.measure_cmnts
        from 
        (select  core.code_catg,dtl.hse_measure_id, core.code, core.code_desc, dtl.hse_id, dtl.measure_status, dtl.measure_cmnts, dtl.created_by
        from MIS_CORE_CODE_MST core, mis_ap_hse_pgm_dtl dtl where core.code_id= dtl.hse_measure_id) a
        left join mis_ap_hse_pgm_mst mst on mst.hse_id = a.hse_id

        """
    )

    df = pd.DataFrame(df).drop_duplicates(keep='first')
    df.to_csv('Input_CSVs/MIS_CMT.csv', escapechar='\\', doublequote=False)
    return df

def proh_plant():
    df = executeQuery("""    
    select 
    PM.ap_pgm_id, PM.pgm_id
    , rd.region_id
    , RM.region_name
    , pd.gy_plant_id
    , gpm.plant_name
    from MIS_AP_PI_PGM_MST PM
    left join MIS_AP_PI_PGM_REG_DTL RD on pm.ap_pgm_id = rd.ap_pgm_id
    left join MIS_REGION_MST RM on rd.region_id = rm.region_id
    left join MIS_AP_PI_PGM_PLANT_DTL PD on pm.ap_pgm_id = pd.ap_pgm_id
    left join MIS_GOODYEAR_PLANT_MST GPM on pd.gy_plant_id = gpm.gy_plant_id and rd.region_id = gpm.region_id
   
    """)
    df = pd.DataFrame(df)[['AP_PGM_ID', 'PGM_ID', 'REGION_NAME', 'PLANT_NAME']]
    df = df.rename(columns={'REGION_NAME':'REGION_FOR_SOS_APPROVAL','PLANT_NAME':'PROH_PLANT_NAME'})
    df = df.drop_duplicates(keep="first")
    df.to_csv('Raw_Data/PROH_REG_PLANT.csv', escapechar='\\', doublequote=False)
    return df


code_base = base_data_code()[['PGM_ID', 'HSE_ID', 'HSE_MEASURE_ID', 'CODE', 'MEASURE_STATUS', 'PROP_PLANT_FOR_TRIAL',
                              'REGION_FOR_SOS_APPROVAL']]
comment_base = base_data_comment()[['PGM_ID', 'HSE_ID',  'CODE', 'MEASURE_CMNTS']]

desc_base = base_data_comment()[['PGM_ID','CODE', 'CODE_DESC']]


df_code = pd.merge(code_base, comment_base, how = 'left', on =['PGM_ID', 'HSE_ID','CODE'])
df_code =df_code.drop_duplicates()
df_code = df_code.rename(columns={'REGION_FOR_SOS_APPROVAL':'REGION_NAME', 'PROP_PLANT_FOR_TRIAL':'PROH_PLANT_NAME'})
df_code = pd.merge(df_code, desc_base, how = 'left', on =['PGM_ID','CODE'])

df_code.to_csv('Raw_Data/MIS_Infocard_GPS.csv', index= False, header = True, escapechar='\\', doublequote=False)
print("df_code.to_csv('Raw_Data/MIS_Infocard_GPS.csv', index= False, header = True, escapechar='\\', doublequote=False)")
proh_plant()

# Other Data Extraction Logic by SA

In [ ]:
def extractRawData():       
    df = executeQuery("""SELECT m.material_group
        , M.REVISION_NO
        , M.MATERIAL_CODE 
        , M.MATERIAL_DESC
        , CASE WHEN SAD.IS_ACTIVE is null THEN SD.IS_ACTIVE ELSE SAD.IS_ACTIVE END AS IS_ACTIVE
        , SM.SUPPLIER_NAME
        , SPD.PLANT_NAME 
        , SPD.CITY
        , SPD.STATE
        , SC.COUNTRY_NAME
        , STD.TRADENAME
        , SA.SUPPLIER_NAME AS A_NAME
        , SAP.PLANT_NAME AS A_PLANT_NAME
        , SAP.CITY AS A_CITY
        , SAP.STATE AS A_STATE
        , AC.COUNTRY_NAME as A_COUNTRY_NAME
        , x.agent_build_break
        , coalesce(PGM.EXP_CODE, REI.EXP_CODE) AS EXP_CODE
        , psv.approval_process_name
        , CASE WHEN psv.approval_process_name = 'NEW MATERIAL' OR psv.approval_process_name = 'NEW SOURCE' THEN xp.exp_code
        ELSE  M.MATERIAL_CODE end as CONSOLIDATED_EX_CODE
        , sd.pgm_id
        , X.PGM_REASON
        , CONCAT(REQ_USR.first_name,CONCAT(' ', REQ_USR.last_name)) as REQUESTED_BY
        , CONCAT(ASS_USR.first_name,CONCAT(' ',ASS_USR.last_name)) as ASSIGNED_TO
        , SD.IS_ACTIVE as SUP_STATUS
        , SAD.IS_ACTIVE as AGENT_STATUS
        , SD.approval_region as SOS_Approval_Region
        , SD.approval_plants
        , SAD.Approval_region as Agent_Approval_Region
        , SAD.approval_plants as Agent_Approval_Plants
        , MIS_GLOBAL_PKG.MIS_RPT_CSV_CONCAT_FN(sd.MATERIAL_CODE_SUPPLIER_ID, 'SOSPLANT') AS RESTRICTED_PLANTS
        FROM MIS_SOS_SUP_DTL SD
        LEFT JOIN mis_sos_mat_mst M ON sd.material_code_id = m.material_code_id
        LEFT JOIN MIS_SUPPLIER_TRADE_DTL STD ON sd.supplier_tradename_id = std.tradename_id
        LEFT JOIN MIS_SUPPLIER_MST SM ON std.supplier_id = sm.supplier_id
        LEFT JOIN MIS_SUPPLIER_PLANT_DTL SPD ON sd.supplier_plant_id = spd.plant_id
        LEFT JOIN mis_sos_sup_agent_dtl SAD ON sd.material_code_supplier_id = sad.material_code_supplier_id AND sad.record_type = 'D'
        LEFT JOIN MIS_SUPPLIER_MST SA ON sad.approval_agent = sa.supplier_id
        LEFT JOIN MIS_SUPPLIER_PLANT_DTL SAP ON sad.agent_plant_id = sap.plant_id
        LEFT JOIN mis_country_mst SC ON spd.country_id = sc.country_id
        LEFT JOIN mis_country_mst AC ON SAP.country_id = ac.country_id
        LEFT JOIN MIS_AP_PI_PGM_MST X ON sd.pgm_id = x.pgm_id
        LEFT JOIN MIS_AP_RMI_PGM_DTL xp ON sd.pgm_id = xp.pgm_id
        LEFT JOIN MIS_AP_PROGRAM_SUMMARY_VW PSV on sd.pgm_id = psv.pgm_id
        LEFT JOIN MIS_AP_PROGRAM_MST Y on SD.PGM_ID = Y.PGM_ID
        LEFT JOIN MIS_USER_MST REQ_USR on y.requested_by = REQ_USR.user_id
        LEFT JOIN MIS_USER_MST ASS_USR on y.assigned_to = ASS_USR.user_id
        LEFT join MIS_AP_RMI_PGM_DTL PGM on sd.PGM_ID =  PGM.PGM_ID
        LEFT join MIS_AP_REINFO_PGM_DTL REI on sd.PGM_ID =  REI.PGM_ID
    """)
    df['MATERIAL_DESC'] = df['MATERIAL_DESC'].str.slice(0, 80)
    df.to_csv('Raw_Data/Mat_Sup_Agent.csv',index=False, escapechar='\\', doublequote=False)


    df = executeQuery('''select
                    pgm.pgm_id
                    , pgm.hse_id
                    , pgm_dtl.hse_measure_id
                    , pi_pgm.material_code_id
                    , m.material_code
                    , m.revision_no
                    , code_mst.code
                    , code_mst.code_desc
                    , pgm_dtl.measure_status
                    , pgm_dtl.measure_cmnts
                    , creat_user.first_name as created_by
                    , pgm_dtl.created_date
                    , mod_user.first_name as modified_by
                    , pgm_dtl.modified_date
                    , gy_p.region_id
                    , gy_r.region_name as PROPOSED_REGION_FOR_TRIAL
                    , pi_pgm.prop_plant_trial
                    , gy_p.plant_name as PROPOSED_PLANT_FOR_TRIAL
                    from mis_ap_hse_pgm_mst pgm
                    left join MIS_AP_PI_PGM_MST pi_pgm on pgm.pgm_id = pi_pgm.pgm_id
                    left join MIS_SOS_MAT_MST m on pi_pgm.MATERIAL_CODE_ID = m.material_code_id
                    left join mis_ap_hse_pgm_dtl pgm_dtl on pgm.hse_id = pgm_dtl.hse_id
                    left join MIS_CORE_CODE_MST code_mst on pgm_dtl.hse_measure_id = code_mst.code_id
                    left join mis_user_mst creat_user on pgm_dtl.created_by = creat_user.user_id
                    left join mis_user_mst mod_user on pgm_dtl.modified_by = mod_user.user_id
                    left join MIS_GOODYEAR_PLANT_MST gy_p on pi_pgm.prop_plant_trial = gy_p.gy_plant_id
                    left Join MIS_REGION_MST gy_r on gy_p.region_id = gy_r.region_id
                    ''')
    df.to_csv('Raw_Data/GPS_questions_data.csv',index=False, escapechar='\\', doublequote=False)

    plants = executeQuery("""select 
    PM.ap_pgm_id
    , PM.pgm_id
    , rd.region_id
    , RM.region_name
    , gpm.gy_plant_id
    , gpm.plant_name as PROH_PLANT_NAME
    from MIS_AP_PI_PGM_MST PM
    left join MIS_AP_PI_PGM_REG_DTL RD on pm.ap_pgm_id = rd.ap_pgm_id
    left join MIS_REGION_MST RM on rd.region_id = rm.region_id
    left join MIS_AP_PI_PGM_PLANT_DTL PD on pm.ap_pgm_id = pd.ap_pgm_id
    left join MIS_GOODYEAR_PLANT_MST GPM on pd.gy_plant_id = gpm.gy_plant_id and rd.region_id = gpm.region_id
                          """)
    
    plants.to_csv('Raw_Data/GPS_Regions_Plants.csv',index=False, escapechar='\\', doublequote=False)

    df = executeQuery('select * from MIS_SOS_MAT_MST')
    df['MATERIAL_DESC'] = df['MATERIAL_DESC'].str.slice(0, 80)
    df.to_csv('Raw_Data/MIS_SOS_MAT_MST.csv',index=False, escapechar='\\', doublequote=False)

    df = executeQuery("""SELECT MCAD.pgm_id, MCAD.attachment_id, MCAD.file_path, MCAD.cycle_id, MCAD.screen_section_id, MSSM.screen_section, MCAD.att_url_flag FROM MIS_CORE_ATTACHMENT_DTL MCAD, MIS_SCREEN_SECTION_MST  MSSM WHERE MCAD.screen_section_id = MSSM.screen_section_id ORDER BY 4,3 ASC""")
    df.to_csv('Raw_Data/MIS_PGM_DOCUMENTS.csv',index=False, escapechar='\\', doublequote=False)
    

    df = executeQuery("""SELECT PGM_v.*
        , coalesce(PGM.EXP_CODE, REI.EXP_CODE) AS EXP_CODE
        , PGM.SPECIFIC_GRAVITY
        , PGM.RHC_POLYMER
        , PGM.RHC_OIL
        , PGM.RHC_FILTER
        , PGM.RMI_DESC
        , PGM.MANET_CLASS
        , PGM.MANET_SUBCLASS
        , PGM.MATERIAL_TYPE
        , PGM.MATERIAL_SUBTYPE
        , PGM.COMPONENT_TYPE
        , PGM.STAINING
        , PGM.COMPATIBLE
        , PGM.MATERIAL_FORM
        , PI.PGM_REASON
        , PI.AGENT_BUILD_BREAK
        , REI.REINGMI_DESC
        , REI.MAT_CLASS
        , REI.MAT_SUBCLASS
        , REI.MAT_DOUBLE_SUBCLASS
        , MIS_GLOBAL_PKG.MIS_RPT_CSV_CONCAT_FN(PGM_v.PGM_ID, 'REGION') AS APPROVAL_REGIONS
        , MIS_GLOBAL_PKG.MIS_RPT_CSV_CONCAT_FN(PGM_v.PGM_ID, 'PLANT') AS RESTRICTED_PLANTS
        FROM MIS_AP_PROGRAM_SUMMARY_VW PGM_v
        LEFT join MIS_AP_RMI_PGM_DTL PGM on PGM_v.PGM_ID =  PGM.PGM_ID
        LEFT join MIS_AP_PI_PGM_MST PI on PGM_v.PGM_ID =  PI.PGM_ID
        LEFT join MIS_AP_REINFO_PGM_DTL REI on PGM_v.PGM_ID =  REI.PGM_ID""")
    
    df2 = executeQuery("""Select * from MIS_NM_PROD_CODE_DTL""")
    
    df2.to_csv('Raw_Data/MIS_NM_PROD_CODE_DTL.csv',index=False, escapechar='\\', doublequote=False)
    
    pgm_mat_map=df2.set_index("PGM_ID")["PROP_NAME_MAT_CODE"].to_dict()
    def update_material_code(row):
        if pd.isna(row['MATERIAL_CODE']) and row['PGM_ID'] in pgm_mat_map:
            return pgm_mat_map[row["PGM_ID"]]
        return row["MATERIAL_CODE"]
        
    df.loc[df['MATERIAL_GROUP']=='REINFORCEMENT', 'MATERIAL_CODE']=df[df['MATERIAL_GROUP']=='REINFORCEMENT'].apply(update_material_code,axis=1)

    df.to_csv('Raw_Data/MIS_PGM_DATA.csv',index=False, escapechar='\\', doublequote=False)

print('extractRawData()')
extractRawData()

In [113]:
def shortencontext(df):
    df['SP_KEY1'] = df['SP_KEY1'].str[:40]
    df['SP_KEY2'] = df['SP_KEY2'].str[:40]
    df['SP_KEY3'] = df['SP_KEY3'].str[:40]
    df['SP_KEY4'] = df['SP_KEY4'].str[:40]
    df['SP_KEY5'] = df['SP_KEY5'].str[:40]
    return df

# Get Supplier Df merged with SIS plants

In [ ]:
def GetSISSupplierDf():    
    supplier_df = pd.read_csv('Raw_Data/Mat_Sup_Agent.csv', na_filter=False, escapechar='\\')
    plant_proh_reg_df = pd.read_csv('Raw_Data/PROH_REG_PLANT.csv', na_filter=False, escapechar='\\')

    plant_code_df = pd.read_excel('Input_excels/Supplier_Plant_SISCode-241122.xlsx', na_filter=False, engine='openpyxl')

    

    supplier_df = pd.merge(supplier_df, plant_code_df, left_on=["SUPPLIER_NAME", "PLANT_NAME", "CITY"], right_on=["Supplier MIS", "Plant MIS", "City MIS"], how="left")
    supplier_df = supplier_df.rename(columns={"Supplier MIS":"sis_Supplier", "Plant MIS":"sis_Plant", "SIS Plant code":"sis_SIS_Plant_code"})
 
    supplier_df = pd.merge(supplier_df, plant_code_df, left_on=["A_NAME", "A_PLANT_NAME", "A_CITY"], right_on=["Supplier MIS", "Plant MIS", "City MIS"], how="left")
    supplier_df = supplier_df.rename(columns={"Supplier MIS":"sis_Agent", "Plant MIS":"sis_A_Plant", "SIS Plant code":"sis_Agent_Plant_code"})


    merge_df3 = pd.merge(plant_proh_reg_df, plant_code_df, left_on= 'PROH_PLANT_NAME', right_on='Plant MIS')
    plant_proh_reg_df['PROH_PLANT_NAME'] = merge_df3['Plant MIS'].combine_first(plant_proh_reg_df['PROH_PLANT_NAME'])

    supplier_df['SUPPLIER_NAME'] = supplier_df['Supplier Opcenter_x'].combine_first(supplier_df['SUPPLIER_NAME'])
    supplier_df['PLANT_NAME'] = supplier_df['Plant Opcenter_x'].combine_first(supplier_df['PLANT_NAME'])
    supplier_df['A_NAME'] = supplier_df['Supplier Opcenter_y'].combine_first(supplier_df['A_NAME'])
    supplier_df['A_PLANT_NAME'] = supplier_df['Plant Opcenter_y'].combine_first(supplier_df['A_PLANT_NAME'])

    supplier_df['MATERIAL_DESC'] = supplier_df['MATERIAL_DESC'].str.slice(0,80)


    supplier_df['REVISION_NO'] = supplier_df['REVISION_NO'] + 1
    supplier_df = supplier_df.drop_duplicates()
    supplier_df['PGM_REASON'] = (
    supplier_df['PGM_REASON']
    .apply(lambda x: html.unescape(x) if isinstance(x, str) else x)
    .str.replace("\n", " ")
    .str.replace("\r", "")
    .str.strip()
)
    supplier_df = supplier_df[supplier_df['MATERIAL_GROUP'] == 'REINFORCEMENT'] 
    supplier_df['MATERIAL_CODE'] = supplier_df['MATERIAL_CODE'].apply(lambda x: x if x.startswith('RF') else 'RF'+x)
    supplier_df.to_csv('Raw_Data/supplier_df.csv', index=False, escapechar='\\', doublequote=False)
    return supplier_df
print('GetSISSupplierDf()')
GetSISSupplierDf()

# Code to populate RF-GenInfo and RF-Basic

In [115]:
def RM_GenInfo():
    RM_GenInfo = executeQuery("""select a.*, b.pgm_id from
    (select material_code_id, material_code, class, subclass, material_desc, is_active as SAP_STATUS,
    material_type as GBS_Material_Type, material_subtype as GBS_Secondary_Material_Type, 
    component_type as GBS_Component_Type, STAINING, component_type as SAP_Type_of_Component,
    material_form, material_code as Initial_Inspection_Code
    from  MIS_SOS_MAT_MST where material_group = 'REINFORCEMENT') a 
    left join MIS_AP_PI_PGM_MST b on a.material_code_id = b.MATERIAL_CODE_ID""")

    RM_GenInfo = pd.DataFrame(RM_GenInfo)
    RM_GenInfo = RM_GenInfo.dropna(subset=['PGM_ID'])
    RM_GenInfo['PGM_ID'] = RM_GenInfo['PGM_ID'].astype(int)
    RM_GenInfo.to_csv('Input_CSVs/GenInfo_ReInforcements.csv', index=False, escapechar='\\', doublequote=False)
    return RM_GenInfo


def RM_BasicInfo():
    RM_BasicInfo = executeQuery(""" select a.*, b.pgm_id from
                            (select material_code_id, material_code as CONSTRUCTION_CODE_FABRIC, 
                            material_code as CONSTRUCTION_CODE_SQUARE_OVEN,
                            material_code as CONSTRUCTION_CODE_WIRE_CABLE, MATERIAL_DESC as SAP_SOS_DESCRIPTION
                            FROM MIS_SOS_MAT_MST where material_group = 'REINFORCEMENT') a
                            left join MIS_AP_PI_PGM_MST b on a.material_code_id = b.MATERIAL_CODE_ID
                            """)

    RM_BasicInfo = pd.DataFrame(RM_BasicInfo)
    RM_BasicInfo = RM_BasicInfo.dropna(subset=['PGM_ID'])
    RM_BasicInfo['PGM_ID'] = RM_BasicInfo['PGM_ID'].astype(int)
    RM_BasicInfo.to_csv('Input_CSVs/BasicInfo_ReInforcements.csv', index=False, escapechar='\\', doublequote=False)
    return RM_BasicInfo

In [116]:
# # # mat_code_list = ['TV27CN', 'NN', 'TU01CU', 'BH', 'WY', '139N', 'RF0047828', 'RF0047235', 'RF396D33', 'RD4024']
# # mat_code_list = ['NN',
# #  'BH',
# #  '139N',
# #  'WY',
# #  'TV27CN',
# #  'RF396D33',
# #  'TU01CU',
# #  'RF0047235',
# #  'RF0040259',
# #  'RFNC0005',
# #  'RFY00040',
# #  'RF0041806',
# #  'RF0041819',
# #  'RF0041921',
# #  'RF0045665',
# #  'RF0044055',
# #  'RF0043057',
# #  'RFV00144',
# #  'RFW00012',
# #  'RFV00141',
# #  'RF0044054',
# #  'RF0044051',
# #  'RF0044052',
# #  'RF0044053',
# #  'RFV00159',
# #  'RFV00160',
# #  'RFGL3310',
# #  'RFGL3332',
# #  'RFGL2127',
# #  'RFGL3329',
# #  'RFV00163',
# #  'RFV00166',
# #  'RF0047758']

# mat_code_list = [ 'P01L', 'NT26QA', 'J35ZS', 'B05L', '9496', 'LH23MA', 'KC15JR', 'F07Q', 'CH', 'NC16KR', 'NT30UA', 'NJ27JF', 'LV19JA', 'FP', 'QV28', 'C17GAZ', 'S01Q', 'E09A', 'BL01WM', 'QU26PF', 'Q09R', 'Z02BI30GF', 'U07L', 'QC32PF', 'LC16SD', 'S03N', 'M14LP19JA', 'TQ26HF', 'JOAX_FDC', 'TEST1', 'RW31SH', '40HK2B', 'Z01NG30GA', 'FA01WY', 'FC01FR', 'RFN00001', 'S00004', '0040084', 'GL2060', '0040253', '0040652', 'S00047', '0041590', '0041619', '0041582', '0041826', '0041817', '0045664', '0043059', '0044389', 'S00090', '0046962', 'W00044' ]

# Populate Material.csv

In [ ]:
def PopulateMaterialCsv():
    #code to populate material.csv
    material_df = pd.read_csv('Raw_Data/MIS_SOS_MAT_MST.csv', na_filter=False, escapechar='\\')
    raw_material_df = material_df[material_df['MATERIAL_GROUP'] == 'REINFORCEMENT']
    raw_material_df['REVISION_NO'] = raw_material_df['REVISION_NO']+1

    raw_material_df = raw_material_df.sort_values(['MATERIAL_CODE', 'REVISION_NO'], ascending=[True, False])
    raw_material_df = raw_material_df.drop_duplicates(subset='MATERIAL_CODE', keep='first')

    MA_OP_DF = pd.DataFrame()
    with open('Input_jsons/Material.json', 'r') as file:
        master_data = json.load(file)

    MA_OP_DF['MA_VALUE'] = raw_material_df['MATERIAL_CODE']
    MA_OP_DF['DESCRIPTION'] = raw_material_df['MATERIAL_DESC'].str.slice(0,80)
    MA_OP_DF['DESCRIPTION'] = MA_OP_DF['DESCRIPTION'].str.rstrip('\r\n')
    MA_OP_DF['ACTIVE'] = raw_material_df['IS_ACTIVE']
    MA_OP_DF['ACTIVE'] = 1
    MA_OP_DF['DATE_IMPORTED'] = date.today()

    MA_OP_Columns = list(master_data.keys())

    master_data.pop('MA_VALUE', None)
    master_data.pop('DESCRIPTION', None)
    master_data.pop('DATE_IMPORTED', None)
    master_data.pop('ACTIVE', None)

    for key, value in master_data.items():
        MA_OP_DF[key] = value if value is not None else None

    MA_OP_DF = MA_OP_DF[MA_OP_Columns]
    # MA_OP_DF = MA_OP_DF[MA_OP_DF['MA_VALUE'].isin(mat_code_list)]
    MA_OP_DF.drop_duplicates()
    MA_OP_DF['MA_VALUE'] = MA_OP_DF['MA_VALUE'].astype(str)
    MA_OP_DF['MA_VALUE'] = MA_OP_DF['MA_VALUE'].apply(lambda x: x if x.startswith('RF') else 'RF'+x)
    MA_OP_DF = MA_OP_DF.drop_duplicates()
    MA_OP_DF.to_csv('Output_CSVs/Material.csv', header=True, index=False, escapechar='\\', doublequote=False)
    return MA_OP_DF
print('PopulateMaterialCsv()')
PopulateMaterialCsv()

# Populate SP.csv

In [118]:
def add_new_Spec():
        supplier_df = pd.read_csv('Raw_Data/supplier_df.csv', na_filter=False, escapechar='\\')
        with open('Input_jsons/SP.json', 'r') as file:
                master_data = json.load(file)

        SP_DF_3 = pd.DataFrame(columns=['SP_VALUE','SP_VERSION','FR_SHORT_DESC','FR_VERSION','CONTEXT','SP_KEY1','SP_KEY2','SP_KEY3','SP_KEY4','SP_KEY5','STYPE_VALUE','CREATED_BY','LC_DESC','LC_VERSION','SS_DESC','CREATED_ON','EFFECTIVE_FROM','EFFECTIVE_TILL','HAS_ADHOC_APPROVAL' ])
                
        for index, sup in supplier_df.iterrows():
                Mat_code = sup['MATERIAL_CODE']
                Version = sup['REVISION_NO']
                temp = supplier_df[(supplier_df['MATERIAL_CODE'] == Mat_code)
                                & (supplier_df['REVISION_NO'] == Version)
                                & (supplier_df['SUPPLIER_NAME'] == sup['SUPPLIER_NAME'])
                                & (supplier_df['PLANT_NAME'] == sup['PLANT_NAME'])
                                & (supplier_df['TRADENAME'] == sup['TRADENAME'])
                                ]
                
                result = ((temp['AGENT_STATUS'] == 'N').all() and (temp['SUP_STATUS'] == 'Y').all())

                if result:
                        
                        new_sp = [
                                sup['MATERIAL_CODE']
                                , sup['REVISION_NO']
                                , master_data["FR_SHORT_DESC"][1]
                                , None
                                , 'Supplier'
                                , sup['SUPPLIER_NAME']
                                , sup['PLANT_NAME']
                                , ''
                                , ''
                                , sup['TRADENAME']
                                , 'SRM'
                                , 'EventManager'
                                , 'Goodyear LC for Material Specifications'
                                , None
                                , '@U'
                                , date.today()
                                , None
                                , None
                                , None
                        ]
                        
                        SP_DF_3.loc[len(SP_DF_3)] = new_sp
        SP_DF_3['SP_VALUE'] = SP_DF_3['SP_VALUE'].apply(lambda x: x if x.startswith('RF') else 'RF'+x)
        return SP_DF_3.drop_duplicates()

# add_new_Spec()

In [ ]:
def PopulateSpCsv():
    # code to popultae SP.csv
    material_df = pd.read_csv('Raw_Data/MIS_SOS_MAT_MST.csv', na_filter=False, escapechar='\\')
    material_df = material_df[material_df['MATERIAL_GROUP'] == 'REINFORCEMENT']
    material_df['MATERIAL_CODE'] = material_df['MATERIAL_CODE'].apply(lambda x: x if x.startswith('RF') else 'RF'+x)

    material_df['STATUS'] = material_df['CLASS'].apply(
        lambda x: 'IA' if x == 'YARN' else None
    )
    material_df['STATUS'] = material_df['STATUS'].fillna(
        material_df['IS_ACTIVE'].map({'Y': '@U', 'N': 'IA'})
    )
    mat_df = material_df[['MATERIAL_CODE', 'CLASS', 'REVISION_NO', 'STATUS']]
    mat_df = mat_df .drop_duplicates(keep='first')
    mat_df['CLASS'] = mat_df['CLASS'].apply(lambda x: 'ReinforcementSteel' if x == 'WIRE' else 'ReinforcementFabric')
    mat_df = mat_df.rename(columns= {"MATERIAL_CODE":"MA_VALUE"})
    mat_df['REVISION_NO'] = mat_df['REVISION_NO'] + 1

    supplier_df = pd.read_csv('Raw_Data/supplier_df.csv', na_filter=False, escapechar='\\')
    MA_OP_DF = pd.read_csv('Output_CSVs/Material.csv', na_filter=False, escapechar='\\')
    MA_OP_DF = pd.merge(MA_OP_DF, mat_df, on =['MA_VALUE'], how='left')
    SP_OP_DF = pd.DataFrame()
    SP_DF = pd.DataFrame()
    SP_DF_2 = pd.DataFrame()
    SP_DF_3 = pd.DataFrame()

    with open('Input_jsons/SP.json', 'r') as file:
        master_data = json.load(file)

    # SP_DF['SP_VALUE'] = MA_OP_DF['MA_VALUE']
    SP_DF['SP_VALUE'] = material_df['MATERIAL_CODE']
    SP_DF['CREATED_ON'] = date.today()
    SP_DF['SP_VERSION'] = MA_OP_DF['REVISION_NO']
    SP_DF['STYPE_VALUE'] = 'RF'
    SP_DF['SS_DESC'] = MA_OP_DF['STATUS']
    SP_DF["FR_SHORT_DESC"] = MA_OP_DF['CLASS']
    SP_OP_DF = pd.concat([SP_OP_DF, SP_DF], ignore_index=True)
    SP_DF_2['SP_VALUE'] = supplier_df['MATERIAL_CODE']
    SP_DF_2['CREATED_ON'] = date.today()
    SP_DF_2['STYPE_VALUE'] = 'SRF'
    SP_DF_2['CONTEXT'] = "Supplier"
    SP_DF_2['SP_KEY1'] = supplier_df['SUPPLIER_NAME']
    SP_DF_2['SP_KEY2'] = supplier_df['PLANT_NAME']
    SP_DF_2['SP_KEY3'] = supplier_df['A_NAME']
    SP_DF_2['SP_KEY4'] = supplier_df['A_PLANT_NAME']
    SP_DF_2['SP_KEY5'] = supplier_df['TRADENAME']
    SP_DF_2['SP_VERSION'] = supplier_df['REVISION_NO']
    yarn_materials = material_df.loc[material_df['CLASS'] == 'YARN', 'MATERIAL_CODE'].values
    SP_DF_2['SS_DESC'] = supplier_df.apply(lambda row: 'IA' if row['MATERIAL_CODE'] in yarn_materials else ('@U' if row['IS_ACTIVE'] == 'Y' else 'IA'),axis=1)
    SP_DF_2["FR_SHORT_DESC"] = 'ReinforcementFabric'

    SP_DF_2['SP_KEY2'] = SP_DF_2['SP_KEY2'].str[:40]
    SP_DF_2['SP_KEY4'] = SP_DF_2['SP_KEY4'].str[:40]
   
    SP_OP_DF = pd.concat([SP_OP_DF, SP_DF_2], ignore_index=True)
    
    SP_DF_3 = add_new_Spec()
    SP_DF_3['SP_KEY2'] = SP_DF_3['SP_KEY2'].str[:40]
    SP_DF_3['SP_KEY4'] = SP_DF_3['SP_KEY4'].str[:40]
    SP_OP_DF = pd.concat([SP_OP_DF, SP_DF_3], ignore_index=True)
   
    SP_OP_Columns = list(master_data.keys())

    master_data.pop('SP_VALUE', None)
    master_data.pop('CREATED_ON', None)
    master_data.pop('FR_SHORT_DESC', None)
    master_data.pop('STYPE_VALUE', None)
    master_data.pop('CONTEXT', None)
    master_data.pop('SS_DESC', None)
    master_data.pop('SP_VERSION', None)
    master_data.pop('SP_KEY1', None)
    master_data.pop('SP_KEY2', None)
    master_data.pop('SP_KEY3', None)
    master_data.pop('SP_KEY4', None)
    master_data.pop('SP_KEY5', None)


    for key, value in master_data.items():
        SP_OP_DF[key] = value if value is not None else None


    SP_OP_DF = SP_OP_DF[SP_OP_Columns]

    SP_OP_DF.loc[SP_OP_DF['CONTEXT'] == 'Supplier', ['FR_SHORT_DESC', 'STYPE_VALUE']] = 'SupplierReinforcement', 'SRF'
    
    try:
        SP_OP_DF.loc[SP_OP_DF['CONTEXT']=='Reinforcement', ['STYPE_VALUE']] = 'RF'
    except:
        pass  
     
    # mat_code_list_normalized = set(mat_code_list)
    # mat_code_list_normalized.update(['RF' + code for code in mat_code_list if not code.startswith('RF')])
    # SP_OP_DF = SP_OP_DF[SP_OP_DF['SP_VALUE'].isin(mat_code_list_normalized)] 

    SP_OP_DF = SP_OP_DF.dropna(subset = ['SP_VERSION'])
   
    SP_OP_DF['SP_VERSION'] = SP_OP_DF['SP_VERSION'].astype(int)
    SP_OP_DF['SP_VALUE'] = SP_OP_DF['SP_VALUE'].astype(str)
    SP_OP_DF['SP_VALUE'] = SP_OP_DF['SP_VALUE'].apply(lambda x: x if x.startswith('RF') else 'RF'+x)
    key_columns = ["SP_VALUE", "SP_VERSION", "FR_SHORT_DESC", "FR_VERSION", "CONTEXT", "SP_KEY1", "SP_KEY2", "SP_KEY3", "SP_KEY4", "SP_KEY5", "STYPE_VALUE", "CREATED_BY"]
    filtered_df_u = SP_OP_DF[SP_OP_DF["SS_DESC"] == "@U"].drop_duplicates(subset=key_columns, keep="first")
    filtered_df_ia = SP_OP_DF[SP_OP_DF["SS_DESC"] == "IA"].drop_duplicates(subset=key_columns, keep=False)
    remaining_df = SP_OP_DF[~SP_OP_DF["SS_DESC"].isin(["@U", "IA"])]
    combined_df = pd.concat([filtered_df_u, filtered_df_ia, remaining_df], ignore_index=True)
    SP_OP_DF = combined_df.drop_duplicates(subset=key_columns, keep="first")
    SP_OP_DF = SP_OP_DF.drop_duplicates()
    SP_OP_DF.to_csv('Output_CSVs/SP.csv', index=False, header=True, escapechar='\\', doublequote=False)   
    SP_OP_DF = SP_OP_DF.drop_duplicates() 
    SP_OP_DF.to_csv('Input_CSVs/SOS_SPECS.csv', header=True, index=False, escapechar='\\', doublequote=False)
    return SP_OP_DF
print('PopulateSpCsv()')
PopulateSpCsv()

# Populate SP-AU.csv with new logic

In [ ]:
def PopulateSpAuCsv():

    sp_df = pd.read_csv('Output_CSVs/SP.csv', na_filter=False, escapechar='\\')

    with open('Input_jsons/SP-AU.json', 'r') as file:
        master_data = json.load(file)

    ex_code_df = pd.read_csv('Raw_Data/supplier_df.csv', na_filter=False, escapechar='\\')
    ex_code_df['MATERIAL_CODE'] = ex_code_df['MATERIAL_CODE'].apply(lambda x: x if x.startswith('RF') else 'RF'+x)
    ex_code_df['PLANT_NAME'] = ex_code_df['PLANT_NAME'].str[:40]
    ex_code_df['A_PLANT_NAME'] = ex_code_df['A_PLANT_NAME'].str[:40]
    SPAU_OP_Columns = list(master_data.keys())
    SPAU_OP_DF = pd.DataFrame(columns=SPAU_OP_Columns)
    sp_df = sp_df.fillna('')
    ex_code_df['CONSOLIDATED_EX_CODE'] = ex_code_df['CONSOLIDATED_EX_CODE'].astype(str)


    for index, row in sp_df.iterrows():
        ex_code_df_f = ex_code_df[(ex_code_df["MATERIAL_CODE"] == row['SP_VALUE']) 
                                & (ex_code_df['REVISION_NO'] == row['SP_VERSION']) 
                                & (ex_code_df['MATERIAL_GROUP']=='REINFORCEMENT')
                                & (ex_code_df['SUPPLIER_NAME']== (row['SP_KEY1']))
                                & (ex_code_df["PLANT_NAME"] == row['SP_KEY2'])
                                & (ex_code_df["A_NAME"] == row['SP_KEY3'])
                                & (ex_code_df["A_PLANT_NAME"] == row['SP_KEY4'])
                                & (ex_code_df["TRADENAME"] == row['SP_KEY5'])
                                ]

        
        for AU_SHORT_DESC in master_data["AU_SHORT_DESC"]:
            if(AU_SHORT_DESC == "auExpSpecCode"):
                df_len = len(ex_code_df_f['EXP_CODE'])
                if df_len >= 1:
                    # TODO: Change following lines in Reinforcements
                    AU_Value = ex_code_df_f.EXP_CODE.iloc[df_len-1]
                else:
                    AU_Value = ''

            elif (AU_SHORT_DESC == "auSpecType"):
                if(row["STYPE_VALUE"] == "RF"):
                    AU_Value = "Reinforcement"
                elif(row["STYPE_VALUE"] == "SRF"):
                    AU_Value = "Supplier Reinforcement"
            elif (AU_SHORT_DESC == "CodeMaskPrefix"):
                AU_Value = "RF"
            value = [
                row['SP_VALUE'],
                row["SP_VERSION"],
                row["CONTEXT"],
                row["SP_KEY1"],
                row["SP_KEY2"],
                row["SP_KEY3"],
                row["SP_KEY4"],
                row["SP_KEY5"],
                AU_SHORT_DESC,
                master_data["AU_VERSION"],
                master_data["AUSEQ"],
                AU_Value
            ]
            values_df = pd.DataFrame([value], columns = SPAU_OP_DF.columns)
            SPAU_OP_DF = pd.concat([SPAU_OP_DF, values_df], ignore_index=True)

    SPAU_OP_DF = SPAU_OP_DF[SPAU_OP_Columns]
    SPAU_OP_DF.loc[SPAU_OP_DF['AU_SHORT_DESC']=='CodeMaskPrefix', 'VALUE'] = 'RF'
    SPAU_OP_DF['SP_VALUE'] = SPAU_OP_DF['SP_VALUE'].apply(lambda x:x if x.startswith('RF') else "RF"+x)
    SPAU_OP_DF = SPAU_OP_DF.drop_duplicates()
    SPAU_OP_DF.to_csv('Output_CSVs/SP-AU.csv', header=True, index=False, escapechar='\\', doublequote=False)
    return SPAU_OP_DF
print('PopulateSpAuCsv()')
PopulateSpAuCsv()

# Populate SP-II.csv for RF Basic, RF GenInfo

In [ ]:
def PopulateSpIiCsv():
    #code to popultae SP-II.csv
    material_df = pd.read_csv('Raw_Data/MIS_SOS_MAT_MST.csv', na_filter=False, escapechar='\\')
    raw_material_df = material_df[material_df['MATERIAL_GROUP'] == 'REINFORCEMENT']
    raw_material_df['REVISION_NO'] = raw_material_df['REVISION_NO'] + 1
    
    supplier_df = pd.read_csv('Raw_Data/supplier_df.csv', na_filter=False, escapechar='\\')

    # raw_material_df = raw_material_df[raw_material_df['MATERIAL_CODE'].isin(mat_code_list)]
    # mat_code_list_normalized = set(mat_code_list)
    # mat_code_list_normalized.update(['RF' + code for code in mat_code_list if not code.startswith('RF')])

    # supplier_df = supplier_df[supplier_df['MATERIAL_CODE'].isin(mat_code_list_normalized)]
    supplier_df['MATERIAL_CODE'] = supplier_df['MATERIAL_CODE'].apply(lambda x: x if x.startswith('RF') else 'RF' + str(x))
    
    with open('Input_jsons/SP_II_new.json', 'r') as file:
        master_data = json.load(file)

    SP_II_OP_Columns = list(master_data.keys())
    SP_II_OP_DF = pd.DataFrame(columns=SP_II_OP_Columns)
    i = 0
    raw_material_df['STAINING'] = raw_material_df['STAINING'].map({'Y':'Yes', 'N':'No'})
    raw_material_df['COMPATIBLE'] = raw_material_df['COMPATIBLE'].map({'Y':'Yes', 'N':'No'})
    raw_material_df['MATERIAL_FORM'] = raw_material_df['MATERIAL_FORM'].map({'S':'Solid', 'L':'Liquid', 'P':'Paste', 'G':'Gas'})

    fabric = pd.read_excel('Input_excels/Fabrics all Fields.XLSX', na_filter=False, engine='openpyxl')
    fabric_new = pd.read_excel('Input_excels/SAP_Output.xlsx', na_filter=False, engine='openpyxl')

    fabric_new = fabric_new[fabric_new['Material'].str.startswith('RF')]
    # fabric_new = fabric_new[fabric_new['Material'].isin(mat_code_list)]
    fabric_new['Cable Bead Coating'] = ''
    fabric_new['Gauge'] = ''
    fabric_new['Bead Inside Diameter'] = ''
    fabric_new['Component Weight'] = ''
    

    cables = pd.read_excel('Input_excels/cables (1).xlsx', na_filter=False, engine='openpyxl')

    df = pd.read_excel('Input_excels/cables new.xlsx', na_filter=False, engine='openpyxl')
    df =  df.drop(['Unnamed: 17', 'Unnamed: 12'], axis =  1)
    fabric_new  = pd.concat([fabric_new, cables, df], axis=0)
    sap_new = pd.read_csv('Input_excels/SAP_Files/SAP_new.csv', na_filter=False, escapechar='\\')
    fabric_new = pd.concat([fabric_new, sap_new], axis=0)
    
    with open('Input_jsons/SP-II_Reinf_new.json', 'r') as file:
        master_data = json.load(file)
    fabric_new['Material'] = fabric_new['Material'].apply(lambda x: x if x.startswith('RF') else 'RF' + str(x))   
    fabric_new = fabric_new.rename(columns={'Material':'MATERIAL_CODE'})
    raw_material_df['MATERIAL_CODE'] = raw_material_df['MATERIAL_CODE'].astype(str)
    raw_material_df['MATERIAL_CODE'] = raw_material_df['MATERIAL_CODE'].apply(lambda x: x if x.startswith('RF') else 'RF' + str(x))
    raw_material_df = pd.merge(raw_material_df, fabric_new, on=['MATERIAL_CODE'], how='left')
    
    final_output_df = pd.read_csv(r'Input_excels/SAP_All_In_One/SAP_new.csv', na_filter=False, escapechar='\\')

    notsap_input_df = pd.read_csv('Input_CSVs/MAT_Not_In_SAP.csv', na_filter=False, escapechar='\\')

    for m_index, row in raw_material_df.iterrows():
        MAT_CODE = row['MATERIAL_CODE']
        REVISION_NO = row['REVISION_NO']
        MATERIAL_TYPE = row['MATERIAL_TYPE']
        MATERIAL_SUBTYPE = row['MATERIAL_SUBTYPE']
        sp_row = supplier_df[supplier_df['MATERIAL_CODE'] == MAT_CODE]
        sp_row = sp_row[sp_row['REVISION_NO'] == REVISION_NO]
        for IC in master_data['IC_CAPTION']:
            try:
                for II in master_data['II_CAPTION'][IC]:
                    IE = master_data['IE_SHORT_DESC'][IC][II]
                    IIVALUE = None

                    if IC == "Reinforcement General Information":
                        if IE == "ieMaterialDescri_old":
                            IIVALUE = row['MATERIAL_DESC']                    
                        elif IE == "ieMaterialClass":
                            IIVALUE = row['CLASS']
                        elif IE == "cbGBSMatType":
                            IIVALUE = MATERIAL_TYPE
                        elif IE == "cbGBSTypeComp":
                            IIVALUE = row['COMPONENT_TYPE']
                        elif IE == "ddlStaining":
                            IIVALUE = row['STAINING']
                        elif IE == "ddlMatForm":
                            IIVALUE = row['MATERIAL_FORM']
                        elif IE == "ieSAPStatusCode":
                            IIVALUE = "Y1" 
                            II = ""
                        elif IE == "ddlSAPCPIValue":
                                CPI_df = pd.read_excel('Input_excels/CPI_logic.xlsx', na_filter=False, engine='openpyxl')
                                CPI_Rec = CPI_df[(CPI_df['Class'] == row['CLASS']) & (CPI_df['Subclass'] == row['SUBCLASS'])]
                                if(len(CPI_Rec)>0):
                                    IIVALUE = CPI_Rec['CPI Value'].iloc[0]
                                else:
                                    IIVALUE = 1
                        elif IE == "ddlSAPStatusDesc":
                            IIVALUE = "Active (Non Development line-up)"
                        elif IE == "ieMaterialSubclass":
                            IIVALUE = row['SUBCLASS']
                        elif IE == "cbGBSMatSecType":
                            IIVALUE = MATERIAL_SUBTYPE
                        elif IE == "ddlCompatible":
                            IIVALUE = row['COMPATIBLE']
                        elif IE=="ieInitialExpSpCode":
                            
                            sp_row = sp_row[sp_row["APPROVAL_PROCESS_NAME"] == "NEW MATERIAL"]
                            count = len(sp_row)
                            if count > 0:
                                IIVALUE = sp_row['EXP_CODE'].iloc[count -1]
                            else:
                                IIVALUE = None
                        elif IE == "cbMaterialCode":
                            IIVALUE = row['MATERIAL_CODE']
                        
                        values = [MAT_CODE,
                                    REVISION_NO,
                                    master_data['CONTEXT'],
                                    master_data['SP_KEY1'],
                                    master_data['SP_KEY2'],
                                    master_data['SP_KEY3'],
                                    master_data['SP_KEY4'],
                                    master_data['SP_KEY5'],
                                    IC,
                                    master_data['IC_ORDER'],
                                    II,
                                    IE,
                                    master_data['IE_VERSION'],
                                    master_data['II_ORDER'],
                                    IIVALUE
                                ]

                        SP_II_OP_DF.loc[len(SP_II_OP_DF)] = values


                    elif IC == "Reinforcement - Basic":
                        if (row['Source'] == 'Fabrics Squarewoven all Fields') or (row['Source'] == 'Wires all Fields'):
                            if IE == "cbConCodeFabric":
                                IIVALUE = row['Fabric Construction Code']
                            elif IE == "cbConCodeSqWov":
                                IIVALUE = row['Fabric Squarewoven Construction Code']
                            elif IE == "cbConCodeWirCab":
                                IIVALUE = row['Cable Construction Code']
                            elif IE == "efSAPSOSDesc":
                                row_data = final_output_df[final_output_df['Material'] == row['MATERIAL_CODE']]
                                if row_data.empty:
                                    IIVALUE = row['Material description']
                                else:
                                    material_desc = row_data['Material description'].values[0]
                                    fabric_processing = row_data['Fabric Processing Code'].values[0]
                                    fabric_construction_code = row_data['Fabric Construction Code'].values[0]
                                    wire_construction_code = row_data['Wire Construction Code'].values[0]
                                    fabric_squarewoven_construction_code = row_data['Fabric Squarewoven Construction Code'].values[0]
                                    cord_density_code = row_data['Cord Density Code'].values[0]

                                    IIVALUE = material_desc
        
                                    if (fabric_construction_code not in reinfdicts.RndatFabricConstructionCodeDict and
                                        wire_construction_code not in reinfdicts.RndatCableConstructionCodeDict and
                                        fabric_squarewoven_construction_code not in reinfdicts.RndatFabricSquarewovenConstructionCodeDict):
                                        IIVALUE = material_desc
                                    else:
                                        if fabric_construction_code in reinfdicts.RndatFabricConstructionCodeDict:
                                            IIVALUE = reinfdicts.RndatFabricConstructionCodeDict[fabric_construction_code]                                        
                                        elif fabric_squarewoven_construction_code in reinfdicts.RndatFabricSquarewovenConstructionCodeDict:
                                            IIVALUE = reinfdicts.RndatFabricSquarewovenConstructionCodeDict[fabric_squarewoven_construction_code]
                                        elif wire_construction_code in reinfdicts.RndatCableConstructionCodeDict:
                                            IIVALUE = reinfdicts.RndatCableConstructionCodeDict[wire_construction_code]
                                    if cord_density_code in reinfdicts.RndatFabricEndCodesDict:
                                        fabric_ends_per_inch = reinfdicts.RndatFabricEndCodesDict.get(cord_density_code)
                                        if fabric_ends_per_inch:
                                            IIVALUE = f"{IIVALUE}, {fabric_ends_per_inch} EPI, {fabric_processing}" 
                                        else:
                                            IIVALUE = f"{IIVALUE} EPI"
                            values = [MAT_CODE,
                                        REVISION_NO,
                                        master_data['CONTEXT'],
                                        master_data['SP_KEY1'],
                                        master_data['SP_KEY2'],
                                        master_data['SP_KEY3'],
                                        master_data['SP_KEY4'],
                                        master_data['SP_KEY5'],
                                        IC,
                                        2,
                                        II,
                                        IE,
                                        master_data['IE_VERSION'],
                                        master_data['II_ORDER'],
                                        IIVALUE
                                    ]
                            SP_II_OP_DF.loc[len(SP_II_OP_DF)] = values 


                        elif row['Source'] == 'Single Cord Fabrics all Fields':
                            if IE == "cbEPI":
                                IIVALUE = 1
                            elif IE == "efSAPSOSDesc":
                                row_data = final_output_df[final_output_df['Material'] == row['MATERIAL_CODE']]
                                if row_data.empty:
                                    IIVALUE = row['Material description']
                                else:
                                    material_desc = row_data['Material description'].values[0]
                                    fabric_processing = row_data['Fabric Processing Code'].values[0]
                                    fabric_construction_code = row_data['Fabric Construction Code'].values[0]
                                    wire_construction_code = row_data['Wire Construction Code'].values[0]
                                    fabric_squarewoven_construction_code = row_data['Fabric Squarewoven Construction Code'].values[0]
                                    cord_density_code = row_data['Cord Density Code'].values[0]

                                    IIVALUE = material_desc
        
                                    if (fabric_construction_code not in reinfdicts.RndatFabricConstructionCodeDict and
                                        wire_construction_code not in reinfdicts.RndatCableConstructionCodeDict and
                                        fabric_squarewoven_construction_code not in reinfdicts.RndatFabricSquarewovenConstructionCodeDict):
                                        IIVALUE = material_desc
                                    else:
                                        if fabric_construction_code in reinfdicts.RndatFabricConstructionCodeDict:
                                            IIVALUE = reinfdicts.RndatFabricConstructionCodeDict[fabric_construction_code]                                        
                                        elif fabric_squarewoven_construction_code in reinfdicts.RndatFabricSquarewovenConstructionCodeDict:
                                            IIVALUE = reinfdicts.RndatFabricSquarewovenConstructionCodeDict[fabric_squarewoven_construction_code]
                                        elif wire_construction_code in reinfdicts.RndatCableConstructionCodeDict:
                                            IIVALUE = reinfdicts.RndatCableConstructionCodeDict[wire_construction_code]
                                    if cord_density_code in reinfdicts.RndatFabricEndCodesDict:
                                        fabric_ends_per_inch = reinfdicts.RndatFabricEndCodesDict.get(cord_density_code)
                                        if fabric_ends_per_inch:
                                            IIVALUE = f"{IIVALUE}, {fabric_ends_per_inch} EPI, {fabric_processing}" 
                                        else:
                                            IIVALUE = f"{IIVALUE} EPI"
                            elif IE == "cbConCodeFabric":
                                IIVALUE = row['Fabric Construction Code']
                            elif IE == 'cbFabricProcCode':
                                IIVALUE = row['Fabric Processing Code']
                                

                            values = [MAT_CODE,
                                        REVISION_NO,
                                        master_data['CONTEXT'],
                                        master_data['SP_KEY1'],
                                        master_data['SP_KEY2'],
                                        master_data['SP_KEY3'],
                                        master_data['SP_KEY4'],
                                        master_data['SP_KEY5'],
                                        IC,
                                        2,
                                        II,
                                        IE,
                                        master_data['IE_VERSION'],
                                        master_data['II_ORDER'],
                                        IIVALUE
                                    ]
                            SP_II_OP_DF.loc[len(SP_II_OP_DF)] = values   

                        elif (row['Source'] == 'Fabrics all Fields'):
                            # print(row['Source'])
                            # print(row['Material description'])
                            if IE == "cbConCodeFabric":
                                IIVALUE = row['Fabric Construction Code']
                            elif IE == "cbConCodeSqWov":
                                IIVALUE = row['Fabric Squarewoven Construction Code']
                            elif IE == "cbConCodeWirCab":
                                IIVALUE = row['Cable Construction Code']
                            elif IE == "cbEPI":
                                IIVALUE = row['Calculated Ends per Inch']
                            elif IE == 'cbCordDensCode':
                                IIVALUE = row['Cord Density Code']
                            elif IE == 'cbFabricProcCode':
                                IIVALUE = row['Fabric Processing Code']
                            elif IE == "efPicksPerMeter":
                                IIVALUE = row['Picks per Meter']
                            elif IE == "ddlFillCord":
                                IIVALUE = row['Fill Cord']
                            elif IE =="efLinCorWt":
                                IIVALUE=  row['Linear Fill Cord Weight DEN']
                            elif IE == "efSAPSOSDesc":
                                row_data = final_output_df[final_output_df['Material'] == row['MATERIAL_CODE']]
                                if row_data.empty:
                                    IIVALUE = row['Material description']
                                else:
                                    material_desc = row_data['Material description'].values[0]
                                    fabric_processing = row_data['Fabric Processing Code'].values[0]
                                    fabric_construction_code = row_data['Fabric Construction Code'].values[0]
                                    wire_construction_code = row_data['Wire Construction Code'].values[0]
                                    fabric_squarewoven_construction_code = row_data['Fabric Squarewoven Construction Code'].values[0]
                                    cord_density_code = row_data['Cord Density Code'].values[0]

                                    IIVALUE = material_desc
        
                                    if (fabric_construction_code not in reinfdicts.RndatFabricConstructionCodeDict and
                                        wire_construction_code not in reinfdicts.RndatCableConstructionCodeDict and
                                        fabric_squarewoven_construction_code not in reinfdicts.RndatFabricSquarewovenConstructionCodeDict):
                                        IIVALUE = material_desc
                                    else:
                                        if fabric_construction_code in reinfdicts.RndatFabricConstructionCodeDict:
                                            IIVALUE = reinfdicts.RndatFabricConstructionCodeDict[fabric_construction_code]                                        
                                        elif fabric_squarewoven_construction_code in reinfdicts.RndatFabricSquarewovenConstructionCodeDict:
                                            IIVALUE = reinfdicts.RndatFabricSquarewovenConstructionCodeDict[fabric_squarewoven_construction_code]
                                        elif wire_construction_code in reinfdicts.RndatCableConstructionCodeDict:
                                            IIVALUE = reinfdicts.RndatCableConstructionCodeDict[wire_construction_code]
                                    if cord_density_code in reinfdicts.RndatFabricEndCodesDict:
                                        fabric_ends_per_inch = reinfdicts.RndatFabricEndCodesDict.get(cord_density_code)
                                        if fabric_ends_per_inch:
                                            IIVALUE = f"{IIVALUE}, {fabric_ends_per_inch} EPI, {fabric_processing}" 
                                        else:
                                            IIVALUE = f"{IIVALUE} EPI"

                            values = [MAT_CODE,
                                        REVISION_NO,
                                        master_data['CONTEXT'],
                                        master_data['SP_KEY1'],
                                        master_data['SP_KEY2'],
                                        master_data['SP_KEY3'],
                                        master_data['SP_KEY4'],
                                        master_data['SP_KEY5'],
                                        IC,
                                        2,
                                        II,
                                        IE,
                                        master_data['IE_VERSION'],
                                        master_data['II_ORDER'],
                                        IIVALUE
                                    ]
                            SP_II_OP_DF.loc[len(SP_II_OP_DF)] = values  
                        elif row['Source'] == 'Cable Beads':
                            if IE == 'efCBInsideDia':
                                IIVALUE = row['Bead Inside Diameter']
                            elif IE =="efCBCompWeight":
                                IIVALUE =  row['Component Weight']
                            elif IE == "efCBGauge":
                                IIVALUE = row['Gauge']
                            elif IE == "ddlCBCoating":
                                IIVALUE = row['Cable Bead Coating']
                            elif IE == "efSAPSOSDesc":
                                row_data = final_output_df[final_output_df['Material'] == row['MATERIAL_CODE']]
                                if row_data.empty:
                                    IIVALUE = row['Material description']
                                else:
                                    material_desc = row_data['Material description'].values[0]
                                    fabric_processing = row_data['Fabric Processing Code'].values[0]
                                    fabric_construction_code = row_data['Fabric Construction Code'].values[0]
                                    wire_construction_code = row_data['Wire Construction Code'].values[0]
                                    fabric_squarewoven_construction_code = row_data['Fabric Squarewoven Construction Code'].values[0]
                                    cord_density_code = row_data['Cord Density Code'].values[0]

                                    IIVALUE = material_desc
        
                                    if (fabric_construction_code not in reinfdicts.RndatFabricConstructionCodeDict and
                                        wire_construction_code not in reinfdicts.RndatCableConstructionCodeDict and
                                        fabric_squarewoven_construction_code not in reinfdicts.RndatFabricSquarewovenConstructionCodeDict):
                                        IIVALUE = material_desc
                                    else:
                                        if fabric_construction_code in reinfdicts.RndatFabricConstructionCodeDict:
                                            IIVALUE = reinfdicts.RndatFabricConstructionCodeDict[fabric_construction_code]                                        
                                        elif fabric_squarewoven_construction_code in reinfdicts.RndatFabricSquarewovenConstructionCodeDict:
                                            IIVALUE = reinfdicts.RndatFabricSquarewovenConstructionCodeDict[fabric_squarewoven_construction_code]
                                        elif wire_construction_code in reinfdicts.RndatCableConstructionCodeDict:
                                            IIVALUE = reinfdicts.RndatCableConstructionCodeDict[wire_construction_code]
                                    if cord_density_code in reinfdicts.RndatFabricEndCodesDict:
                                        fabric_ends_per_inch = reinfdicts.RndatFabricEndCodesDict.get(cord_density_code)
                                        if fabric_ends_per_inch:
                                            IIVALUE = f"{IIVALUE}, {fabric_ends_per_inch} EPI, {fabric_processing}" 
                                        else:
                                            IIVALUE = f"{IIVALUE} EPI"
                            
                                    
                            values = [MAT_CODE,
                                        REVISION_NO,
                                        master_data['CONTEXT'],
                                        master_data['SP_KEY1'],
                                        master_data['SP_KEY2'],
                                        master_data['SP_KEY3'],
                                        master_data['SP_KEY4'],
                                        master_data['SP_KEY5'],
                                        IC,
                                        2,
                                        II,
                                        IE,
                                        master_data['IE_VERSION'],
                                        master_data['II_ORDER'],
                                        IIVALUE
                                    ]
                            SP_II_OP_DF.loc[len(SP_II_OP_DF)] = values 

                        elif IC == "Reinforcement - Basic":
                            if IE == "efSAPSOSDesc":
                                try:
                                    IIVALUE = notsap_input_df.loc[notsap_input_df['MATERIAL_CODE'] == row['MATERIAL_CODE'], 'MATERIAL_DESC'].values[0]
                                except IndexError:
                                    IIVALUE = None 

                            values = [MAT_CODE,
                                    REVISION_NO,
                                    master_data['CONTEXT'],
                                    master_data['SP_KEY1'],
                                    master_data['SP_KEY2'],
                                    master_data['SP_KEY3'],
                                    master_data['SP_KEY4'],
                                    master_data['SP_KEY5'],
                                    IC,
                                    2,
                                    II,
                                    IE,
                                    master_data['IE_VERSION'],
                                    master_data['II_ORDER'],
                                    IIVALUE
                                ]
                            SP_II_OP_DF.loc[len(SP_II_OP_DF)] = values


                        elif (row['Source'] == 'Wires all Fields'):
                            # print(row['Source'])
                            # print(row['Material description'])
                            if IE == "cbConCodeFabric":
                                IIVALUE = row['Fabric Construction Code']
                            elif IE == "cbConCodeSqWov":
                                IIVALUE = row['Fabric Squarewoven Construction Code']
                            elif IE == "cbConCodeWirCab":
                                IIVALUE = row['Cable Construction Code']

                            values = [MAT_CODE,
                                        REVISION_NO,
                                        master_data['CONTEXT'],
                                        master_data['SP_KEY1'],
                                        master_data['SP_KEY2'],
                                        master_data['SP_KEY3'],
                                        master_data['SP_KEY4'],
                                        master_data['SP_KEY5'],
                                        IC,
                                        2,
                                        II,
                                        IE,
                                        master_data['IE_VERSION'],
                                        master_data['II_ORDER'],
                                        IIVALUE
                                    ]
                            SP_II_OP_DF.loc[len(SP_II_OP_DF)] = values  


                        if (row['Source'] == 'Not in SAP'):
                            if IE == "efSAPSOSDesc":
                                IIVALUE = row['Material description']
                            values = [MAT_CODE,
                                        REVISION_NO,
                                        master_data['CONTEXT'],
                                        master_data['SP_KEY1'],
                                        master_data['SP_KEY2'],
                                        master_data['SP_KEY3'],
                                        master_data['SP_KEY4'],
                                        master_data['SP_KEY5'],
                                        IC,
                                        2,
                                        II,
                                        IE,
                                        master_data['IE_VERSION'],
                                        master_data['II_ORDER'],
                                        IIVALUE
                                    ]
                            SP_II_OP_DF.loc[len(SP_II_OP_DF)] = values  

            except:
                continue

        
        mat_supplier_df = supplier_df[supplier_df['MATERIAL_CODE'] == MAT_CODE]
        sp_row = mat_supplier_df[mat_supplier_df['REVISION_NO'] == REVISION_NO]
        sp_row['AGENT_BUILD_BREAK'] = sp_row['AGENT_BUILD_BREAK'].map({'Y':1, 'N':0, '':0})
        for s_index, s_row in sp_row.iterrows():
            # print('SP Row :',s_index)
            for IC in master_data['IC_CAPTION']:
                for II in master_data['II_CAPTION'][IC]:
                    IE = master_data['IE_SHORT_DESC'][IC][II]
                    IIVALUE = None
                                        
                    if IC == "Supplier/Agent Info":
                        if IE == "cbMaterialCode":
                            IIVALUE = s_row['MATERIAL_CODE']
                        elif IE=="ieInitialExpSpCode":
                            IIVALUE = s_row['EXP_CODE']
                        elif IE == "ieMatDescSupplier":
                            row_data = final_output_df[final_output_df['Material'] == row['MATERIAL_CODE']]
                            if row_data.empty:
                                IIVALUE = row['Material description']
                            else:
                                material_desc = row_data['Material description'].values[0]
                                fabric_processing = row_data['Fabric Processing Code'].values[0]
                                fabric_construction_code = row_data['Fabric Construction Code'].values[0]
                                wire_construction_code = row_data['Wire Construction Code'].values[0]
                                fabric_squarewoven_construction_code = row_data['Fabric Squarewoven Construction Code'].values[0]
                                cord_density_code = row_data['Cord Density Code'].values[0]

                                IIVALUE = material_desc
    
                                if (fabric_construction_code not in reinfdicts.RndatFabricConstructionCodeDict and
                                    wire_construction_code not in reinfdicts.RndatCableConstructionCodeDict and
                                    fabric_squarewoven_construction_code not in reinfdicts.RndatFabricSquarewovenConstructionCodeDict):
                                    IIVALUE = material_desc
                                else:
                                    if fabric_construction_code in reinfdicts.RndatFabricConstructionCodeDict:
                                        IIVALUE = reinfdicts.RndatFabricConstructionCodeDict[fabric_construction_code]                                        
                                    elif fabric_squarewoven_construction_code in reinfdicts.RndatFabricSquarewovenConstructionCodeDict:
                                        IIVALUE = reinfdicts.RndatFabricSquarewovenConstructionCodeDict[fabric_squarewoven_construction_code]
                                    elif wire_construction_code in reinfdicts.RndatCableConstructionCodeDict:
                                        IIVALUE = reinfdicts.RndatCableConstructionCodeDict[wire_construction_code]
                                if cord_density_code in reinfdicts.RndatFabricEndCodesDict:
                                    fabric_ends_per_inch = reinfdicts.RndatFabricEndCodesDict.get(cord_density_code)
                                    if fabric_ends_per_inch:
                                        IIVALUE = f"{IIVALUE}, {fabric_ends_per_inch} EPI, {fabric_processing}" 
                                    else:
                                        IIVALUE = f"{IIVALUE} EPI"
                        elif IE == "ieSupplierTradeName":
                            IIVALUE = s_row['TRADENAME']
                        elif IE == "ieSupplier":
                            IIVALUE = s_row['SUPPLIER_NAME']
                        elif IE == "ieAgent":
                            IIVALUE = s_row['A_NAME']
                        elif IE == "ieSupplierPlant":
                            IIVALUE = s_row['PLANT_NAME']
                        elif IE == "ieAgentPlant":
                            IIVALUE = s_row['A_PLANT_NAME']
                        elif IE == "ieSupplierPlantLocationCity":
                            IIVALUE = s_row['CITY']
                        elif IE == "ieAgentPlantLocationCity":
                            IIVALUE = s_row['A_CITY']
                        elif IE == "ieSupplierPlantLocationState":
                            IIVALUE = s_row['STATE']
                        elif IE == "ieAgentPlantLocationState":
                            IIVALUE = s_row['A_STATE']
                        elif IE == "ieAgentPlantLocationCountry":
                            IIVALUE = s_row['A_COUNTRY_NAME']
                        elif IE == "ieSupplierPlantLocationCountry":
                            IIVALUE = s_row['COUNTRY_NAME']
                        elif IE == "ieSISPlantCode":
                            IIVALUE = s_row["sis_SIS_Plant_code"]
                        elif IE == "ieAgentPlantCode":
                            IIVALUE = s_row['sis_Agent_Plant_code']
                            II = "Plant Code"
                        elif IE == "cbBreakBulk":
                            IIVALUE = s_row['AGENT_BUILD_BREAK']
                            if(np.isnan(IIVALUE)):
                                IIVALUE = 0
                        elif IE == "ieSupplierContactName":
                            IIVALUE = None
                        elif IE == "ieAgentContactName":
                            IIVALUE = None
                        elif IE == "ddlSupType":
                            IIVALUE = None
                        elif IE == "ddlAgentType":
                            IIVALUE = None
                        elif IE == "ieSupplierEmail":
                            IIVALUE = None
                        elif IE == "ieAgentEmail":
                            IIVALUE = None
                        elif IE == "ieSupplierPhone":
                            IIVALUE = None
                        elif IE == "ieAgentPhone":
                            IIVALUE = None
                        Key1 = s_row['SUPPLIER_NAME']
                        Key2 = str(s_row['PLANT_NAME'])[:40] if s_row['PLANT_NAME'] is not np.nan else s_row['PLANT_NAME']
                        Key3 = s_row['A_NAME']
                        Key4 = str(s_row['A_PLANT_NAME'])[:40] if s_row['A_PLANT_NAME'] is not np.nan else s_row['A_PLANT_NAME']
                        Key5 = s_row['TRADENAME']
                        values = [MAT_CODE,
                                    s_row['REVISION_NO'],
                                    "Supplier",
                                    Key1,
                                    Key2,
                                    Key3,
                                    Key4,
                                    Key5,
                                    IC,
                                    master_data['IC_ORDER'],
                                    II,
                                    IE,
                                    master_data['IE_VERSION'],
                                    master_data['II_ORDER'],
                                    IIVALUE
                                ]
                        SP_II_OP_DF.loc[len(SP_II_OP_DF)] = values
                        
    SP_II_OP_DF = SP_II_OP_DF[SP_II_OP_Columns]
    SP_II_OP_DF = pd.DataFrame(SP_II_OP_DF)

    SP_II_OP_DF = SP_II_OP_DF[SP_II_OP_DF['IE_SHORT_DESC']!='ftHeader']
    SP_II_OP_DF.loc[(SP_II_OP_DF['IE_SHORT_DESC']=='ddlFillCord') & (SP_II_OP_DF['IIVALUE']=='NON'), 'IIVALUE'] = 'Non Extensible'
    SP_II_OP_DF.loc[(SP_II_OP_DF['IE_SHORT_DESC']=='ddlFillCord') & (SP_II_OP_DF['IIVALUE']=='EXT'), 'IIVALUE'] = 'Extensible'
    SP_II_OP_DF = SP_II_OP_DF[SP_II_OP_DF['IE_SHORT_DESC']!='ftHeader']
    SP_II_OP_DF['SP_VALUE'] = SP_II_OP_DF['SP_VALUE'].apply(lambda x: x if x.startswith('RF') else 'RF'+x)
    SP_II_OP_DF.loc[(SP_II_OP_DF['IE_SHORT_DESC']=='cbMaterialCode'), 'IIVALUE'] = SP_II_OP_DF.loc[(SP_II_OP_DF['IE_SHORT_DESC']=='cbMaterialCode'), 'SP_VALUE'] 
    SP_II_OP_DF['SP_VALUE'] = SP_II_OP_DF['SP_VALUE'].apply(lambda x: x if x.startswith('RF') else 'RF'+x)
    SP_II_OP_DF = SP_II_OP_DF.drop_duplicates()
    SP_II_OP_DF.to_csv('Output_CSVs/SP-II.csv', header=True, index=False, encoding='utf-8', escapechar='\\', doublequote=False)
    SP_II_OP_DF.head()
    return SP_II_OP_DF
print('PopulateSpIiCsv()')
PopulateSpIiCsv()

# Populate Document Infocard

In [ ]:
def populateSpIiDocuments():
    CSV_columns = ['SP_VALUE','SP_VERSION','CONTEXT','SP_KEY1','SP_KEY2','SP_KEY3','SP_KEY4','SP_KEY5','IC_CAPTION','IC_ORDER','II_CAPTION','IE_SHORT_DESC','IE_VERSION','II_ORDER','FILE_NAME']
    supplier_df = pd.read_csv('Raw_Data/supplier_df.csv', na_filter=False, escapechar='\\')
    supplier_df['MATERIAL_CODE'] = supplier_df['MATERIAL_CODE'].apply(lambda x: x if x.startswith('RF') else 'RF'+x)
    SP_df = pd.read_csv('Output_CSVs/SP.csv', na_filter=False, escapechar='\\')
    PGM_DOC_DF = pd.read_csv('Raw_Data/MIS_PGM_DOCUMENTS.csv', na_filter=False, escapechar='\\')
    PGM_DOC_DF['FILE_NAME'] = PGM_DOC_DF['FILE_PATH'].str.split('/').str[-1]
    SP_II_DOCUMENT_OP_DF = pd.DataFrame(columns=CSV_columns)

    for index, sp in SP_df.iterrows():
        PGM_DF = supplier_df[(supplier_df['MATERIAL_CODE'] == sp['SP_VALUE'])
                            & (supplier_df['REVISION_NO'] == sp['SP_VERSION'])
                            & (supplier_df['SUPPLIER_NAME'] == sp['SP_KEY1'])
                            & (supplier_df['PLANT_NAME'] == sp['SP_KEY2'])
                            & (supplier_df['A_NAME'] == sp['SP_KEY3'])
                            & (supplier_df['A_PLANT_NAME'] == sp['SP_KEY4'])
                            & (supplier_df['TRADENAME'] == sp['SP_KEY5'])
                            ]
        
        PGM_DF['PGM_ID'] = PGM_DF['PGM_ID'].astype(str).str.replace('.0', '', regex=False)
        PGM_DF['PGM_ID'] = PGM_DF['PGM_ID'].replace('', 0)
        PGM_DF['PGM_ID'] = PGM_DF['PGM_ID'].astype(int)
        pgm_ids = PGM_DF['PGM_ID'].unique()
        for pgm_id in pgm_ids:
            if pgm_id != '' or 0 or None:
                DOC_DF = PGM_DOC_DF[PGM_DOC_DF['PGM_ID'] == pgm_id]
                for file in DOC_DF['FILE_NAME']:
                    # print(sp['SP_VALUE'])
                    # print("file is: ", file)
                    if(len(file) > 80):
                        print(file)
                    values = [sp['SP_VALUE']
                            , sp['SP_VERSION']
                            , sp['CONTEXT']
                            , sp['SP_KEY1']
                            , sp['SP_KEY2']
                            , sp['SP_KEY3']
                            , sp['SP_KEY4']
                            , sp['SP_KEY5']
                            , 'Documents'
                            , None
                            , 'General Documents'
                            , 'docDocuments'
                            , None
                            , None
                            , file
                            ]
                    SP_II_DOCUMENT_OP_DF.loc[len(SP_II_DOCUMENT_OP_DF)] = values
    SP_II_DOCUMENT_OP_DF['SP_VALUE'] = SP_II_DOCUMENT_OP_DF['SP_VALUE'].apply(lambda x: x if x.startswith('RF') else 'RF'+x)
    SP_II_DOCUMENT_OP_DF = SP_II_DOCUMENT_OP_DF.drop_duplicates()
    SP_II_DOCUMENT_OP_DF.to_csv('Output_CSVs/SP-II-Document.csv', index=False, escapechar='\\', doublequote=False)
    return SP_II_DOCUMENT_OP_DF
print('populateSpIiDocuments()')
populateSpIiDocuments()

# Populate Program Info

In [ ]:
def PopulateProgramInfoCard():
    IC_CAPTION = 'Program Info'

    InfoFIelds = [
        {'II_CAPTION':'Reason for Program', 'IE_SHORT_DESC':'mleReasProg', 'Conatins':'Reason for Program'},
        {'II_CAPTION':'Program Type', 'IE_SHORT_DESC':'ddlProgramType', 'Conatins':'Program Type'},
        {'II_CAPTION':'Requested By', 'IE_SHORT_DESC':'cbReqBy', 'Conatins':'Requested By'},
        {'II_CAPTION':'Assigned To', 'IE_SHORT_DESC':'mleAssignedTo', 'Conatins':'Assigned To'},
        {'II_CAPTION':'NA', 'IE_SHORT_DESC':'cbRegionNA', 'Conatins':'NA'},
        {'II_CAPTION':'LA', 'IE_SHORT_DESC':'cbRegionLA', 'Conatins':'LA'},
        {'II_CAPTION':'EMEA', 'IE_SHORT_DESC':'cbRegionEMEA', 'Conatins':'EMEA'},
        {'II_CAPTION':'Cooper Americas', 'IE_SHORT_DESC':'cbRegionCooperNA', 'Conatins':'Cooper Americas'},
        {'II_CAPTION':'AP', 'IE_SHORT_DESC':'cbRegionAP', 'Conatins':'AP'},
        {'II_CAPTION':'Cooper EMEA', 'IE_SHORT_DESC':'cbRegionCooperEMEA', 'Conatins':'Cooper EMEA'},
        {'II_CAPTION':'Cooper AP', 'IE_SHORT_DESC':'cbRegionCooperAP', 'Conatins':'Cooper AP'},
        {'II_CAPTION':'Proposed Plant for Trial', 'IE_SHORT_DESC':'cbProposedPlant', 'Conatins':'Proposed Plant for Trial'},
        {'II_CAPTION':'Akron', 'IE_SHORT_DESC':'cbLabAkron', 'Conatins':'Akron'},
        {'II_CAPTION':'Lux', 'IE_SHORT_DESC':'cbLabLux', 'Conatins':'Lux'},
        {'II_CAPTION':'Kunshan', 'IE_SHORT_DESC':'cbLabKunshan', 'Conatins':'Kunshan'},
        {'II_CAPTION':'EHS Approval Region (hidden)', 'IE_SHORT_DESC':'efEHSApprovalReg', 'Conatins':'EHS Approval Region (hidden)'},
        {'II_CAPTION':'SAP Status', 'IE_SHORT_DESC':'ddlSAPStatusDesc', 'Conatins':'SAP Status'},
        {'II_CAPTION':'', 'IE_SHORT_DESC':'ieSAPStatusCode', 'Conatins':'.'}
                  ]
    InfoFIelds_df = pd.DataFrame(InfoFIelds)
    
    sp_df = pd.read_csv('Output_CSVs/SP.csv', na_filter=False, escapechar='\\')
    supplier_df = pd.read_csv('Raw_Data/supplier_df.csv', na_filter=False, escapechar='\\')
    gps_df = pd.read_csv('Raw_Data/GPS_questions_data.csv', na_filter=False, escapechar='\\')
    plant_proh_reg_df = pd.read_csv('Raw_Data/PROH_REG_PLANT.csv', na_filter=False, escapechar='\\')
    sis_df = pd.read_excel('Input_excels/Supplier_Plant_SISCode-241122.xlsx', na_filter=False, engine='openpyxl')
    gps_df = pd.merge(gps_df, sis_df, how='left', left_on='PROPOSED_PLANT_FOR_TRIAL', right_on='Plant MIS')
    gps_df['PROPOSED_PLANT_FOR_TRIAL'] = gps_df['PROPOSED_PLANT_FOR_TRIAL'].combine_first(gps_df['Plant Opcenter'])

    gps_q_df = pd.read_csv('Raw_Data/MIS_Infocard_GPS.csv', na_filter=False, escapechar='\\')

    with open('Input_jsons/SP-II_Reinf_new.json', 'r') as file:
        master_data = json.load(file)

    SP_II_OP_Columns = list(master_data.keys())
    supplier_df['APPROVAL_PROCESS_NAME'] = supplier_df['APPROVAL_PROCESS_NAME'].fillna('')
    supplier_df['APPROVAL_PROCESS_NAME'] = supplier_df['APPROVAL_PROCESS_NAME'].astype('str')

    SP_II_OP_DF = pd.DataFrame(columns=SP_II_OP_Columns)
    for index, sp in sp_df.iterrows():
        PGM_DF = supplier_df[(supplier_df['MATERIAL_CODE'] == sp['SP_VALUE'])
                             & (supplier_df['REVISION_NO'] == sp['SP_VERSION'])
                             & (supplier_df['SUPPLIER_NAME'] == sp['SP_KEY1'])
                             & (supplier_df['PLANT_NAME'] == sp['SP_KEY2'])
                             & (supplier_df['A_NAME'] == sp['SP_KEY3'])
                             & (supplier_df['A_PLANT_NAME'] == sp['SP_KEY4'])
                             & (supplier_df['TRADENAME'] == sp['SP_KEY5'])
                             ]
        
        # -----------------Filtering the valid record for ProgramInfo-----------------
        PGM_DF.reset_index(drop=True, inplace=True)
        PGM_DF['AGENT_APPROVAL_REGION'] = PGM_DF['AGENT_APPROVAL_REGION'].replace('', None)
        Not_null_index = PGM_DF['AGENT_APPROVAL_REGION'].first_valid_index()

        if len(PGM_DF) > 1 and Not_null_index is not None:
            PGM_DF = PGM_DF.iloc[[Not_null_index]]   
        # -----------------End of Filter-----------------

        PGM_DF['PGM_ID'] = PGM_DF['PGM_ID'].astype(str).str.replace('.0', '', regex=False)
        PGM_DF['PGM_ID'] = PGM_DF['PGM_ID'].replace('', 0)
        PGM_DF['PGM_ID'] = PGM_DF['PGM_ID'].astype(int)
        pgm_ids = PGM_DF['PGM_ID'].unique()


        for pgm_id in pgm_ids:
            
            PGM_DF1 = gps_q_df[gps_q_df['PGM_ID'] == pgm_id]
            PGM_DF3 = gps_df[gps_df['PGM_ID'] == pgm_id]
            PGM_DF2 = plant_proh_reg_df[plant_proh_reg_df['PGM_ID'] == pgm_id]
            Regions = PGM_DF2['REGION_FOR_SOS_APPROVAL'].unique()
            
            # Looping for Regions
            for index, field in InfoFIelds_df.iterrows():
                IIVALUE = None
                if(pgm_id ==0):
                    # print(pgm_id)
                    if field['IE_SHORT_DESC'] == 'ddlProgramType':
                        IIVALUE = ''
                    if field['IE_SHORT_DESC'] == 'cbReqBy':
                        IIVALUE = ''
                    if field['IE_SHORT_DESC'] == 'mleAssignedTo':
                        IIVALUE = ''
                    if field['IE_SHORT_DESC'] == 'ddlSAPStatusDesc':
                        IIVALUE = 'See base specification'
                    if field['IE_SHORT_DESC'] == 'ieSAPStatusCode':
                        IIVALUE = 'ZZ'
                    value = [
                        sp['SP_VALUE'],
                        sp['SP_VERSION'],
                        sp['CONTEXT'],
                        sp['SP_KEY1'],
                        sp['SP_KEY2'],
                        sp['SP_KEY3'],
                        sp['SP_KEY4'],
                        sp['SP_KEY5'],
                        IC_CAPTION,
                        None,
                        field['II_CAPTION'],
                        field['IE_SHORT_DESC'],
                        None,
                        None,
                        IIVALUE
                    ]
                    SP_II_OP_DF.loc[len(SP_II_OP_DF)] = value
                    continue

                if field['Conatins'] in Regions:
                    values = [
                        sp['SP_VALUE'],
                        sp['SP_VERSION'],
                        sp['CONTEXT'],
                        sp['SP_KEY1'],
                        sp['SP_KEY2'],
                        sp['SP_KEY3'],
                        sp['SP_KEY4'],
                        sp['SP_KEY5'],
                        IC_CAPTION,
                        None,
                        field['II_CAPTION'],
                        field['IE_SHORT_DESC'],
                        None,
                        None,
                        '1'
                    ]
                    SP_II_OP_DF.loc[len(SP_II_OP_DF)] = values
                else:
                    # Populate 'Proposed Plant for Trial'
                    IIVALUE = None
                    if field['Conatins'] == 'Proposed Plant for Trial':
                        IIVALUE = PGM_DF3['PROPOSED_PLANT_FOR_TRIAL'].unique()[0] if len(PGM_DF3) > 0 else None
                    if field['IE_SHORT_DESC'] == 'mleReasProg' and len(PGM_DF) > 0:
                        IIVALUE = PGM_DF['PGM_REASON'].iloc[0]
                    if field['IE_SHORT_DESC'] == 'ddlProgramType':
                        IIVALUE = PGM_DF['APPROVAL_PROCESS_NAME'].iloc[0].title()
                    if field['IE_SHORT_DESC'] == 'cbReqBy':
                        IIVALUE = PGM_DF['REQUESTED_BY'].iloc[0]
                    if field['IE_SHORT_DESC'] == 'mleAssignedTo':
                        IIVALUE = PGM_DF['ASSIGNED_TO'].iloc[0]
                    if field['IE_SHORT_DESC'] == 'ddlSAPStatusDesc':
                        IIVALUE = 'See base specification'
                    if field['IE_SHORT_DESC'] == 'ieSAPStatusCode':
                        IIVALUE = 'ZZ'
                    value = [
                        sp['SP_VALUE'],
                        sp['SP_VERSION'],
                        sp['CONTEXT'],
                        sp['SP_KEY1'],
                        sp['SP_KEY2'],
                        sp['SP_KEY3'],
                        sp['SP_KEY4'],
                        sp['SP_KEY5'],
                        IC_CAPTION,
                        None,
                        field['II_CAPTION'],
                        field['IE_SHORT_DESC'],
                        None,
                        None,
                        IIVALUE
                    ]
                    SP_II_OP_DF.loc[len(SP_II_OP_DF)] = value
                

    SP_II_OP_DF = SP_II_OP_DF.drop_duplicates()
    SHRT_DESC = ['cbRegionCooperNA', 'cbRegionCooperEMEA', 'cbRegionCooperAP', 'cbLabAkron', 'cbLabLux', 
                'cbLabKunsha', 'cbLabKunshan',
                'cbRegionLA', 'cbRegionEMEA', 'cbRegionAP', 'cbRegionNA'] 
    ii_value = [np.nan, '', None]
    SP_II_OP_DF.loc[(SP_II_OP_DF['IE_SHORT_DESC'].isin(SHRT_DESC)) & (SP_II_OP_DF['IIVALUE'].isin(ii_value)), ['IIVALUE']]=0 
    SP_II_OP_DF['SP_VALUE'] = SP_II_OP_DF['SP_VALUE'].apply(lambda x: x if x.startswith('RF') else 'RF'+x) 
    SP_II_OP_DF = SP_II_OP_DF.drop_duplicates()
    SP_II_OP_DF.to_csv('Output_CSVs/SP-II(Program Info).csv', index=False, escapechar='\\', doublequote=False)       
    return SP_II_OP_DF

print('PopulateProgramInfoCard()')
PopulateProgramInfoCard()

# Populated SP-II.csv

In [124]:
def CreateICforExtraSpec(IC_OP_DF, supplier_df_inp):
    supplier_df = supplier_df_inp
    with open('Input_jsons/SP.json', 'r') as file:
            master_data = json.load(file)

    SP_DF_3 = pd.DataFrame(columns=['SP_VALUE','SP_VERSION','CONTEXT','SP_KEY1','SP_KEY2','SP_KEY3','SP_KEY4','SP_KEY5','IC_CAPTION','IP_VERSION','IP_SHORT_DESC','IC_ORDER'])
            
    for index, sup in supplier_df.iterrows():
            Mat_code = sup['MATERIAL_CODE']
            Version = sup['REVISION_NO']
            temp = supplier_df[(supplier_df['MATERIAL_CODE'] == Mat_code)
                            & (supplier_df['REVISION_NO'] == Version)
                            & (supplier_df['SUPPLIER_NAME'] == sup['SUPPLIER_NAME'])
                            & (supplier_df['PLANT_NAME'] == sup['PLANT_NAME'])
                            & (supplier_df['TRADENAME'] == sup['TRADENAME'])
                            ]
            
            result = ((temp['AGENT_STATUS'] == 'N').all() and (temp['SUP_STATUS'] == 'Y').all())

            if result:
                    
                    new_ic = [
                            sup['MATERIAL_CODE']
                            , sup['REVISION_NO']
                            , 'Supplier'
                            , sup['SUPPLIER_NAME']
                            , sup['PLANT_NAME']
                            , ''
                            , ''
                            , sup['TRADENAME']
                            , 'GPS Full Supplier Approval Review'
                            , ''
                            , 'ipFullApprovalInformation'
                            , '4'
                    ]
                    # print (new_ic)
                    IC_OP_DF.loc[len(IC_OP_DF)] = new_ic
    return IC_OP_DF

# CreateICforExtraSpec

# Populate SP-IC.csv

In [ ]:
def populate_SPIC():

    supplier_df = pd.read_csv('Raw_Data/supplier_df.csv', na_filter=False, escapechar='\\')

    # mat_code_list_normalized = set(mat_code_list)
    # mat_code_list_normalized.update(['RF' + code for code in mat_code_list if not code.startswith('RF')])
    # supplier_df = supplier_df[supplier_df['MATERIAL_CODE'].isin(mat_code_list_normalized)]

    SP_IC = pd.DataFrame()
    SP_IC['SP_VALUE'] = supplier_df['MATERIAL_CODE']
    SP_IC['SP_VERSION'] = supplier_df['REVISION_NO']
    SP_IC['CONTEXT'] = 'Supplier'
    SP_IC['SP_KEY1'] = supplier_df['SUPPLIER_NAME']
    SP_IC['SP_KEY2'] = supplier_df['PLANT_NAME']
    SP_IC['SP_KEY3'] = supplier_df['A_NAME']
    SP_IC['SP_KEY4'] = supplier_df['A_PLANT_NAME']
    SP_IC['SP_KEY5'] = supplier_df['TRADENAME']
    SP_IC['IC_CAPTION'] = 'GPS Full Supplier Approval Review'
    SP_IC['IP_VERSION'] = None
    SP_IC['IP_SHORT_DESC'] = 'ipFullApprovalInformation'
    SP_IC['IC_ORDER'] = '7'

    SP_IC_AGENT_APPROVAL = pd.DataFrame()
    SP_IC_AGENT_APPROVAL['SP_VALUE'] = supplier_df['MATERIAL_CODE']
    SP_IC_AGENT_APPROVAL['SP_VERSION'] = supplier_df['REVISION_NO']
    SP_IC_AGENT_APPROVAL['CONTEXT'] = 'Supplier'
    SP_IC_AGENT_APPROVAL['SP_KEY1'] = supplier_df['SUPPLIER_NAME']
    SP_IC_AGENT_APPROVAL['SP_KEY2'] = supplier_df['PLANT_NAME']
    SP_IC_AGENT_APPROVAL['SP_KEY3'] = supplier_df['A_NAME']
    SP_IC_AGENT_APPROVAL['SP_KEY4'] = supplier_df['A_PLANT_NAME']
    SP_IC_AGENT_APPROVAL['SP_KEY5'] = supplier_df['TRADENAME']
    SP_IC_AGENT_APPROVAL['IC_CAPTION'] = 'GPS Full Agent Approval Review'
    SP_IC_AGENT_APPROVAL['IP_VERSION'] = None
    SP_IC_AGENT_APPROVAL['IP_SHORT_DESC'] = 'ipFullAgentApprovalInformation'
    SP_IC_AGENT_APPROVAL['IC_ORDER'] = '5'

    SP_IC_AGENT_APPROVAL = SP_IC_AGENT_APPROVAL[(SP_IC_AGENT_APPROVAL['SP_KEY3'] != '') & (SP_IC_AGENT_APPROVAL['SP_KEY4'] != '')]
    
    SP_IC = pd.concat([SP_IC, SP_IC_AGENT_APPROVAL], ignore_index=True)

    # considering only first 40 charecters
    SP_IC['SP_KEY2'] = SP_IC['SP_KEY2'].str[:40]
    SP_IC['SP_KEY4'] = SP_IC['SP_KEY4'].str[:40]

    SP_IC = CreateICforExtraSpec(SP_IC, supplier_df)
    # considering only first 40 charecters
    SP_IC['SP_KEY1'] = SP_IC['SP_KEY1'].str[:40]
    SP_IC['SP_KEY2'] = SP_IC['SP_KEY2'].str[:40]
    SP_IC['SP_KEY3'] = SP_IC['SP_KEY3'].str[:40]
    SP_IC['SP_KEY4'] = SP_IC['SP_KEY4'].str[:40]
    SP_IC['SP_KEY5'] = SP_IC['SP_KEY5'].str[:40]
    SP_IC['SP_VALUE'] = SP_IC['SP_VALUE'].apply(lambda x: x if x.startswith('RF') else 'RF'+x)
    SP_IC = SP_IC.drop_duplicates()
    SP_IC.to_csv('Output_CSVs/SP-IC.csv', index=False, escapechar='\\', doublequote=False)
    return SP_IC
    
print('populate_SPIC()')
populate_SPIC()

# Populate GPS inforcard

In [ ]:
import SP_II_GPS_21_Oct as Constants

def PopulateSPIIForGPS():
    sp_df = pd.read_csv('Output_CSVs/SP.csv', na_filter=False, escapechar='\\')
    sp_df = sp_df[sp_df['CONTEXT'] =='Supplier'].drop_duplicates(['SP_VALUE',
                                                                  'SP_VERSION',
                                                                  'SP_KEY1',
                                                                  'SP_KEY2',
                                                                  'SP_KEY3',
                                                                  'SP_KEY4',
                                                                  'SP_KEY5'])
    supplier_df = pd.read_csv('Raw_Data/supplier_df.csv', na_filter=False, escapechar='\\')
    supplier_df = supplier_df[supplier_df['MATERIAL_GROUP']=='REINFORCEMENT']
    gps_df = pd.read_csv('Raw_Data/MIS_Infocard_GPS.csv', na_filter=False, escapechar='\\')
    plant_proh_reg_df = pd.read_csv('Raw_Data/PROH_REG_PLANT.csv', na_filter=False, escapechar='\\')
 
    gps_df['MEASURE_STATUS'] = gps_df['MEASURE_STATUS'].map({'Y':'Yes', 'N':'No', 'U':'Unspecified'})
    gps_df['PGM_ID'] = gps_df['PGM_ID'].astype(str).str.replace('.0', '', regex=False)
    gps_df['PGM_ID'] = gps_df['PGM_ID'].replace('', 0)
    gps_df['PGM_ID'] = gps_df['PGM_ID'].astype(int)
    
    with open('Input_jsons/SP-II_Reinf_new.json', 'r') as file:
        master_data = json.load(file)

    SP_II_OP_Columns = list(master_data.keys())

    SP_II_OP_DF = pd.DataFrame(columns=SP_II_OP_Columns)
    test = pd.DataFrame()
    for index, sp in sp_df.iterrows():
        PGM_DF = supplier_df[(supplier_df['MATERIAL_CODE'] == sp['SP_VALUE'])
                             & (supplier_df['REVISION_NO'] == sp['SP_VERSION'])
                             & (supplier_df['SUPPLIER_NAME'] == sp['SP_KEY1'])
                             & (supplier_df['PLANT_NAME'] == sp['SP_KEY2'])
                             & (supplier_df['A_NAME'] == sp['SP_KEY3'])
                             & (supplier_df['A_PLANT_NAME'] == sp['SP_KEY4'])
                             & (supplier_df['TRADENAME'] == sp['SP_KEY5'])
                             ]
        if len(PGM_DF) == 0:
            PGM_DF = supplier_df[(supplier_df['MATERIAL_CODE'] == sp['SP_VALUE'])
                             & (supplier_df['REVISION_NO'] == sp['SP_VERSION'])
                             & (supplier_df['SUPPLIER_NAME'] == sp['SP_KEY1'])
                             & (supplier_df['PLANT_NAME'] == sp['SP_KEY2'])
                             & (supplier_df['TRADENAME'] == sp['SP_KEY5'])
                             ]
        
        # -----------------Filtering the valid record for AGENT_APPROVAL_REGION-----------------
        PGM_DF.reset_index(drop=True, inplace=True)
        PGM_DF['AGENT_APPROVAL_REGION'] = PGM_DF['AGENT_APPROVAL_REGION'].replace('', None)
        Not_null_index = PGM_DF['AGENT_APPROVAL_REGION'].first_valid_index()

        if len(PGM_DF) > 1 and Not_null_index is not None:
            PGM_DF = PGM_DF.iloc[[Not_null_index]]  
        # -----------------End of Filter-----------------
        
        
        PGM_DF['PGM_ID'] = PGM_DF['PGM_ID'].astype(str).str.replace('.0', '', regex=False)
        PGM_DF['PGM_ID'] = PGM_DF['PGM_ID'].replace('', 0)
        PGM_DF['PGM_ID'] = PGM_DF['PGM_ID'].astype(int)
        try:
            pgm_id = PGM_DF['PGM_ID'].values[0]
        except:
            pass
       
        PGM_DF1 = gps_df[gps_df['PGM_ID'] == (pgm_id)]
        PGM_DF2 = plant_proh_reg_df[plant_proh_reg_df['PGM_ID'] == pgm_id]
    
        Regions_sup = [reg.replace('Cooper Asia', 'Cooper AP').replace('Cooper Europe', 'Cooper EMEA') for reg in str(PGM_DF['SOS_APPROVAL_REGION'].iloc[0]).split(', ')] if len(PGM_DF) > 0 else []
        Regions_Agent = [reg.replace('Cooper Asia', 'Cooper AP').replace('Cooper Europe', 'Cooper EMEA') for reg in str(PGM_DF['AGENT_APPROVAL_REGION'].iloc[0]).split(', ')] if len(PGM_DF) > 0 else []
            
        # Looping for Regions
        for field in Constants.INFO_FILEDS_FOR_GPS:
            IIVALUE = None
            if field['ii_short_description'] == 'efSOSApprovalReg':
                IIVALUE = ', '.join(Regions_sup)
                values = [
                    sp['SP_VALUE'],
                    sp['SP_VERSION'],
                    sp['CONTEXT'],
                    sp['SP_KEY1'],
                    sp['SP_KEY2'],
                    sp['SP_KEY3'],
                    sp['SP_KEY4'],
                    sp['SP_KEY5'],
                    Constants.IC_CAPTION,
                    Constants.IC_ORDER,
                    field['gui_name'],
                    field['ii_short_description'],
                    None,
                    None,
                    IIVALUE
                ]
                SP_II_OP_DF.loc[len(SP_II_OP_DF)] = values
            elif field['ii_short_description'] == 'efFullAppCode':
                IIVALUE = ''
                values = [
                        sp['SP_VALUE'],
                        sp['SP_VERSION'],
                        sp['CONTEXT'],
                        sp['SP_KEY1'],
                        sp['SP_KEY2'],
                        sp['SP_KEY3'],
                        sp['SP_KEY4'],
                        sp['SP_KEY5'],
                        Constants.IC_CAPTION,
                        Constants.IC_ORDER,
                        field['gui_name'],
                        field['ii_short_description'],
                        None,
                        None,
                        IIVALUE
                    ]
                SP_II_OP_DF.loc[len(SP_II_OP_DF)] = values
              
            if field['gui_name'] in Regions_sup:
                values = [
                    sp['SP_VALUE'],
                    sp['SP_VERSION'],
                    sp['CONTEXT'],
                    sp['SP_KEY1'],
                    sp['SP_KEY2'],
                    sp['SP_KEY3'],
                    sp['SP_KEY4'],
                    sp['SP_KEY5'],
                    Constants.IC_CAPTION,
                    Constants.IC_ORDER,
                    field['gui_name'],
                    field['ii_short_description'],
                    None,
                    None,
                    '1'
                ]
                SP_II_OP_DF.loc[len(SP_II_OP_DF)] = values
                

    # -----------------Populating record for AGENT_GPS_Infocard-----------------

            if field['gui_name'] in Regions_Agent and (sp['SP_KEY3'] and sp['SP_KEY4'] != '' or None):
                # Adding values to Agent GPS infocard
                values_Agent = [
                    sp['SP_VALUE'],
                    sp['SP_VERSION'],
                    sp['CONTEXT'],
                    sp['SP_KEY1'],
                    sp['SP_KEY2'],
                    sp['SP_KEY3'],
                    sp['SP_KEY4'],
                    sp['SP_KEY5'],
                    Constants.IC_CAPTION_AGENT,
                    Constants.IC_ORDER_AGENT,
                    field['gui_name'],
                    field['ii_short_description'],
                    None,
                    None,
                    '1'
                ]
                SP_II_OP_DF.loc[len(SP_II_OP_DF)] = values_Agent
                
    # -----------------Block end AGENT_GPS_Infocard-----------------

            # Code for Mesure status
            m_df = PGM_DF1[(PGM_DF1['CODE'] == field['measure_code'])]
            
            if(len(m_df)>0):
                values = [
                    sp['SP_VALUE'],
                    sp['SP_VERSION'],
                    sp['CONTEXT'],
                    sp['SP_KEY1'],
                    sp['SP_KEY2'],
                    sp['SP_KEY3'],
                    sp['SP_KEY4'],
                    sp['SP_KEY5'],
                    Constants.IC_CAPTION,
                    Constants.IC_ORDER,
                    field['gui_name'],
                    field['ii_short_description'],
                    None,
                    None,
                    m_df['MEASURE_STATUS'].iloc[0]
                ]
                SP_II_OP_DF.loc[len(SP_II_OP_DF)] = values
                fields_df = pd.DataFrame(Constants.INFO_FILEDS_FOR_GPS)
                Comment_field = fields_df[(fields_df['gui_name'] == 'Comment') & (fields_df['measure_code'] == field['measure_code'])]
                values_comment = [
                    sp['SP_VALUE'],
                    sp['SP_VERSION'],
                    sp['CONTEXT'],
                    sp['SP_KEY1'],
                    sp['SP_KEY2'],
                    sp['SP_KEY3'],
                    sp['SP_KEY4'],
                    sp['SP_KEY5'],
                    Constants.IC_CAPTION,
                    Constants.IC_ORDER,
                    Comment_field['gui_name'].iloc[0],
                    Comment_field['ii_short_description'].iloc[0],
                    None,
                    None,
                    m_df['MEASURE_CMNTS'].iloc[0] if m_df['MEASURE_CMNTS'].iloc[0] is not None else None
                ]
                SP_II_OP_DF.loc[len(SP_II_OP_DF)] = values_comment

                # Code for Plant prohibited
                Plant_field = fields_df[(fields_df['contains'] == 'Plant Proh')]
                plants = PGM_DF2['PROH_PLANT_NAME'].unique()
                plants_string = '\n'.join(plants)
                values_plant_proh = [
                    sp['SP_VALUE'],
                    sp['SP_VERSION'],
                    sp['CONTEXT'],
                    sp['SP_KEY1'],
                    sp['SP_KEY2'],
                    sp['SP_KEY3'],
                    sp['SP_KEY4'],
                    sp['SP_KEY5'],
                    Constants.IC_CAPTION,
                    Constants.IC_ORDER,
                    Plant_field['gui_name'].iloc[0],
                    Plant_field['ii_short_description'].iloc[0],
                    None,
                    None,
                    plants_string
                ]
                SP_II_OP_DF.loc[len(SP_II_OP_DF)] = values_plant_proh

    SP_II_OP_DF = SP_II_OP_DF[SP_II_OP_DF['IIVALUE'] != 'Unspecified']
    SP_II_OP_DF = SP_II_OP_DF[SP_II_OP_DF['IE_SHORT_DESC'] != 'ddFullApp-TSCA']
    SP_II_OP_DF = SP_II_OP_DF[SP_II_OP_DF['IE_SHORT_DESC'] != 'mlComment']
    SP_II_OP_DF = SP_II_OP_DF[SP_II_OP_DF['IE_SHORT_DESC'] != 'ddlFullAp-CEPAD']
    SP_II_OP_DF = SP_II_OP_DF[SP_II_OP_DF['IE_SHORT_DESC'] != 'ddlFullAp-REACHEU']
    SP_II_OP_DF = SP_II_OP_DF[SP_II_OP_DF['IE_SHORT_DESC'] != 'ddlFullAp-KKDIK']
    SP_II_OP_DF = SP_II_OP_DF[SP_II_OP_DF['IE_SHORT_DESC'] != 'ddlFullAp-Japan']



    SP_II_OP_DF['II_CAPTION'] = SP_II_OP_DF['II_CAPTION'].replace('North America', 'NA')
    SP_II_OP_DF['IIVALUE'] = SP_II_OP_DF['IIVALUE'].replace(np.nan, 'Unspecified')
    SP_II_OP_DF['SP_VALUE'] = SP_II_OP_DF['SP_VALUE'].apply(lambda x: x if x.startswith('RF') else 'RF'+x)
    SP_II_OP_DF = SP_II_OP_DF.drop_duplicates()
    SP_II_OP_DF.to_csv('Output_CSVs/SP-II(GPS).csv', index=False, escapechar='\\', doublequote=False)
    return SP_II_OP_DF
    
print('PopulateSPIIForGPS()')
PopulateSPIIForGPS()

Merge SP-II, SP-II(Program Info), SP-II(GPS)

In [ ]:
def MergeSpIiCsv():
     df1 = pd.read_csv('Output_CSVs/SP-II.csv', na_filter=False, escapechar='\\')
     df2 = pd.read_csv('Output_CSVs/SP-II(GPS).csv', na_filter=False, escapechar='\\')
     df3 = pd.read_csv('Output_CSVs/SP-II(Program Info).csv', na_filter=False, escapechar='\\') 
     df1 = pd.concat([df1, df2], ignore_index=True) 
     df1 = pd.concat([df1, df3], ignore_index=True)
     df1 = df1.drop_duplicates()
     df1.to_csv('Output_CSVs/SP-II.csv', index=False, escapechar='\\', doublequote=False)
     #-----------------------------------Added folowing code to populate the supplier only specs where all the agenst are incative-----------------------------------
     sp_df = pd.read_csv('Output_CSVs/SP.csv', na_filter=False, escapechar='\\')
     sp_df = sp_df[(sp_df['CONTEXT'] == 'Supplier') & (sp_df['SP_KEY3'] == '') & (sp_df['SP_KEY4'] == '')]
     SUP_II_extra = pd.DataFrame()
     SP_II_OP_DF = pd.read_csv('Output_CSVs/SP-II.csv', na_filter=False, escapechar='\\')
     II_to_be_removed = ['ieAgent', 'ieAgentPlant', 'ieAgentPlantLocationCity', 'ieAgentPlantLocationState', 'ieAgentPlantLocationCountry', 'ieAgentPlantCode', 'cbBreakBulk', 'ieAgentContactName', 'ddlAgentType', 'ieAgentEmail', 'ieAgentPhone']
     for index, sp in sp_df.iterrows():
          SUP_II_extra = SP_II_OP_DF[(SP_II_OP_DF['SP_VALUE'] == sp['SP_VALUE'])
                                        & (SP_II_OP_DF['SP_VERSION'] == sp['SP_VERSION'])
                                        & (SP_II_OP_DF['SP_KEY1'] == sp['SP_KEY1'])
                                        & (SP_II_OP_DF['SP_KEY2'] == sp['SP_KEY2'])
                                        & (SP_II_OP_DF['SP_KEY5'] == sp['SP_KEY5'])
                                        & (SP_II_OP_DF['IC_CAPTION'] != 'GPS Full Agent Approval Review')
                                        ]
          SUP_II_extra['SP_KEY3'] = ''
          SUP_II_extra['SP_KEY4'] = ''
          SUP_II_extra = SUP_II_extra[SUP_II_extra['IC_CAPTION'] != 'GPS Full Agent Approval Review']
          # print(SUP_II_extra[['SP_VALUE', 'SP_VERSION', 'IC_CAPTION']])
          SUP_II_extra.loc[SUP_II_extra['IE_SHORT_DESC'].isin(II_to_be_removed), 'IIVALUE'] = '' 
          SP_II_OP_DF = pd.concat([SP_II_OP_DF, SUP_II_extra], ignore_index=True)
          
     # considering only first 40 charecters
     SP_II_OP_DF['SP_KEY1'] = SP_II_OP_DF['SP_KEY1'].str[:40]
     SP_II_OP_DF['SP_KEY2'] = SP_II_OP_DF['SP_KEY2'].str[:40]
     SP_II_OP_DF['SP_KEY3'] = SP_II_OP_DF['SP_KEY3'].str[:40]
     SP_II_OP_DF['SP_KEY4'] = SP_II_OP_DF['SP_KEY4'].str[:40]
     SP_II_OP_DF['SP_KEY5'] = SP_II_OP_DF['SP_KEY5'].str[:40]
     SP_II_OP_DF = SP_II_OP_DF.drop_duplicates()

     SP_II_OP_DF.to_csv('Output_CSVs/SP-II.csv', index=False, escapechar='\\', doublequote=False)

print('MergeSpIiCsv()')
MergeSpIiCsv()

In [ ]:
proh_plants_new = executeQuery("""SELECT m.material_group
        , M.REVISION_NO
        , M.MATERIAL_CODE 
        , M.MATERIAL_DESC
        , CASE WHEN SAD.IS_ACTIVE is null THEN SD.IS_ACTIVE ELSE SAD.IS_ACTIVE END AS IS_ACTIVE
        , SM.SUPPLIER_NAME
        , SPD.PLANT_NAME 
        , SPD.CITY
        , SPD.STATE
        , SC.COUNTRY_NAME
        , STD.TRADENAME
        , SA.SUPPLIER_NAME AS A_NAME
        , SAP.PLANT_NAME AS A_PLANT_NAME
        , SAP.CITY AS A_CITY
        , SAP.STATE AS A_STATE
        , AC.COUNTRY_NAME as A_COUNTRY_NAME
        , x.agent_build_break
        , xp.exp_code
        , psv.approval_process_name
        , CASE WHEN psv.approval_process_name = 'NEW MATERIAL' OR psv.approval_process_name = 'NEW SOURCE' THEN xp.exp_code
        ELSE  M.MATERIAL_CODE end as CONSOLIDATED_EX_CODE
        , sd.pgm_id
        , X.PGM_REASON
        , CONCAT(REQ_USR.first_name,CONCAT(' ', REQ_USR.last_name)) as REQUESTED_BY
        ,CONCAT(ASS_USR.first_name,CONCAT(' ',ASS_USR.last_name)) as ASSIGNED_TO
        , SD.IS_ACTIVE as SUP_STATUS
        , SAD.IS_ACTIVE as AGENT_STATUS
        , SD.approval_region as SOS_Approval_Region
        , SD.approval_plants
        , SAD.Approval_region as Agent_Approval_Region
        , SAD.approval_plants as Agent_Approval_Plants
        , sd.material_code_supplier_id
        ,MIS_GLOBAL_PKG.MIS_RPT_CSV_CONCAT_FN(sd.MATERIAL_CODE_SUPPLIER_ID, 'SOSPLANT') AS RESTRICTED_PLANTS
        FROM MIS_SOS_SUP_DTL SD  
        LEFT JOIN mis_sos_mat_mst M ON sd.material_code_id = m.material_code_id
        LEFT JOIN MIS_SUPPLIER_TRADE_DTL STD ON sd.supplier_tradename_id = std.tradename_id
        LEFT JOIN MIS_SUPPLIER_MST SM ON std.supplier_id = sm.supplier_id
        LEFT JOIN MIS_SUPPLIER_PLANT_DTL SPD ON sd.supplier_plant_id = spd.plant_id
        LEFT JOIN mis_sos_sup_agent_dtl SAD ON sd.material_code_supplier_id = sad.material_code_supplier_id
        LEFT JOIN MIS_SUPPLIER_MST SA ON sad.approval_agent = sa.supplier_id
        LEFT JOIN MIS_SUPPLIER_PLANT_DTL SAP ON sad.agent_plant_id = sap.plant_id
        LEFT JOIN mis_country_mst SC ON spd.country_id = sc.country_id
        LEFT JOIN mis_country_mst AC ON SAP.country_id = ac.country_id
        LEFT JOIN MIS_AP_PI_PGM_MST X ON sd.pgm_id = x.ap_pgm_id
        LEFT JOIN MIS_AP_RMI_PGM_DTL xp ON sd.pgm_id = xp.pgm_id
        LEFT JOIN MIS_AP_PROGRAM_SUMMARY_VW PSV on sd.pgm_id = psv.pgm_id
        LEFT JOIN MIS_AP_PROGRAM_MST Y on SD.PGM_ID = Y.PGM_ID
        LEFT JOIN MIS_USER_MST REQ_USR on y.requested_by = REQ_USR.user_id
        LEFT JOIN MIS_USER_MST ASS_USR on y.assigned_to = ASS_USR.user_id""")


proh_plants_new[proh_plants_new['MATERIAL_CODE']=='B44LB'][['MATERIAL_CODE', 'RESTRICTED_PLANTS']]

In [ ]:
proh_plants_new = executeQuery("""SELECT m.material_group
        , M.REVISION_NO
        , M.MATERIAL_CODE 
        , M.MATERIAL_DESC
        , CASE WHEN SAD.IS_ACTIVE is null THEN SD.IS_ACTIVE ELSE SAD.IS_ACTIVE END AS IS_ACTIVE
        , SM.SUPPLIER_NAME
        , SPD.PLANT_NAME 
        , SPD.CITY
        , SPD.STATE
        , SC.COUNTRY_NAME
        , STD.TRADENAME
        , SA.SUPPLIER_NAME AS A_NAME
        , SAP.PLANT_NAME AS A_PLANT_NAME
        , SAP.CITY AS A_CITY
        , SAP.STATE AS A_STATE
        , AC.COUNTRY_NAME as A_COUNTRY_NAME
        , x.agent_build_break
        , xp.exp_code
        , psv.approval_process_name
        , CASE WHEN psv.approval_process_name = 'NEW MATERIAL' OR psv.approval_process_name = 'NEW SOURCE' THEN xp.exp_code
        ELSE  M.MATERIAL_CODE end as CONSOLIDATED_EX_CODE
        , sd.pgm_id
        , X.PGM_REASON
        , CONCAT(REQ_USR.first_name,CONCAT(' ', REQ_USR.last_name)) as REQUESTED_BY
        ,CONCAT(ASS_USR.first_name,CONCAT(' ',ASS_USR.last_name)) as ASSIGNED_TO
        , SD.IS_ACTIVE as SUP_STATUS
        , SAD.IS_ACTIVE as AGENT_STATUS
        , SD.approval_region as SOS_Approval_Region
        , SD.approval_plants
        , SAD.Approval_region as Agent_Approval_Region
        , SAD.approval_plants as Agent_Approval_Plants
        , sd.material_code_supplier_id
        ,MIS_GLOBAL_PKG.MIS_RPT_CSV_CONCAT_FN(sd.MATERIAL_CODE_SUPPLIER_ID, 'SOSPLANT') AS RESTRICTED_PLANTS
        FROM MIS_SOS_SUP_DTL SD  
        LEFT JOIN mis_sos_mat_mst M ON sd.material_code_id = m.material_code_id
        LEFT JOIN MIS_SUPPLIER_TRADE_DTL STD ON sd.supplier_tradename_id = std.tradename_id
        LEFT JOIN MIS_SUPPLIER_MST SM ON std.supplier_id = sm.supplier_id
        LEFT JOIN MIS_SUPPLIER_PLANT_DTL SPD ON sd.supplier_plant_id = spd.plant_id
        LEFT JOIN mis_sos_sup_agent_dtl SAD ON sd.material_code_supplier_id = sad.material_code_supplier_id
        LEFT JOIN MIS_SUPPLIER_MST SA ON sad.approval_agent = sa.supplier_id
        LEFT JOIN MIS_SUPPLIER_PLANT_DTL SAP ON sad.agent_plant_id = sap.plant_id
        LEFT JOIN mis_country_mst SC ON spd.country_id = sc.country_id
        LEFT JOIN mis_country_mst AC ON SAP.country_id = ac.country_id
        LEFT JOIN MIS_AP_PI_PGM_MST X ON sd.pgm_id = x.ap_pgm_id
        LEFT JOIN MIS_AP_RMI_PGM_DTL xp ON sd.pgm_id = xp.pgm_id
        LEFT JOIN MIS_AP_PROGRAM_SUMMARY_VW PSV on sd.pgm_id = psv.pgm_id
        LEFT JOIN MIS_AP_PROGRAM_MST Y on SD.PGM_ID = Y.PGM_ID
        LEFT JOIN MIS_USER_MST REQ_USR on y.requested_by = REQ_USR.user_id
        LEFT JOIN MIS_USER_MST ASS_USR on y.assigned_to = ASS_USR.user_id""")


proh_plants_new[proh_plants_new['MATERIAL_CODE']=='F301FN'][['MATERIAL_CODE', 'RESTRICTED_PLANTS']]
    
proh_plants_new.head()

In [130]:
single = ['AE01FA',
 'AR01SNZ',
 'AR01SNS',
 'AW01SN',
 'BK01ZV',
 'BN01ZV',
 'CB01WU',
 'CE01WA',
 'CG01FH',
 'CG01VI',
 'JN01DH',
 'NC01KA',
 'NC01QA',
 'NC01QH',
 'SF01EU',
 'TM01CU',
 'TU01CU']

cable = ['RF213D2A',
 'RF263E2A',
 'RF265K3A',
 'RF288K3A',
 'RF315E24',
 'RF339D23',
 'RF364F3A',
 'RF366E25',
 'RF366G3A',
 'RF366P2A',
 'RF370E25',
 'RF380G33',
 'RF382E33',
 'RF383K2A',
 'RF384G33',
 'RF38CK2B',
 'RF394D33',
 '394D33',
 'RF396D33',
 'RF408K2A',
 'RF40HK2B',
 'RF416E24',
 'RF418E25',
 'RF418G33',
 'RF424G44',
 'RF446G34',
 'RF448G44',
 'RF448M53',
 'RF470R4A',
 'RF472E5A',
 'RF472H45',
 'RF476G43',
 'RF500L24',
 'RF500P34',
 'RF524H45',
 'RF52DL34',
 'RF536H45',
 'RF547L34',
 'RF549L34',
 'RF574T35',
 'RF576G54',
 'RF576G55',
 'RF57HN43',
 'RF581N45',
 'RF60ET44',
 'RF213D2A',
 'RF263E2A',
 'RF265K3A',
 'RF288K3A',
 'RF315E24',
 'RF339D23',
 'RF364F3A',
 'RF366E25',
 'RF366G3A',
 'RF366P2A',
 'RF370E25',
 'RF380G33',
 'RF382E33',
 'RF383K2A',
 'RF384G33',
 'RF38CK2B',
 'RF394D33',
 '394D33',
 'RF396D33',
 'RF408K2A',
 'RF40HK2B',
 'RF416E24',
 'RF418E25',
 'RF418G33',
 'RF424G44',
 'RF446G34',
 'RF448G44',
 'RF448M53',
 'RF470R4A',
 'RF472E5A',
 'RF472H45',
 'RF476G43',
 'RF500L24',
 'RF500P34',
 'RF524H45',
 'RF52DL34',
 'RF536H45',
 'RF547L34',
 'RF549L34',
 'RF574T35',
 'RF576G54',
 'RF576G55',
 'RF57HN43',
 'RF581N45',
 'RF60ET44']

# Code to Migrate MIS-SCREENS

In [131]:
def add_migrated_data_infocard(sp, IC_OP_DF):
    new_ic = [
            sp['SP_VALUE']
            , sp['SP_VERSION']
            , 'Supplier'
            , sp['SP_KEY1']
            , sp['SP_KEY2']
            , sp['SP_KEY3']
            , sp['SP_KEY4']
            , sp['SP_KEY5']
            , 'MIS Migrated Data'
            , ''
            , 'ipMISMigration'
            , '6'
    ]
    IC_OP_DF.loc[len(IC_OP_DF)] = new_ic
    return IC_OP_DF

In [132]:
def map_doc(pgm_id, sp):
    doc_df = pd.read_csv("Input_CSVs/FileList.csv", na_filter=False, escapechar='\\')
    name_split = doc_df['FILE_NAME'].str.split('_').str[1]
    doc_df['PGM_ID'] = name_split
    doc_df['PGM_ID'] = doc_df['PGM_ID'].astype(int)
    

    CSV_columns = ['SP_VALUE','SP_VERSION','CONTEXT','SP_KEY1','SP_KEY2','SP_KEY3','SP_KEY4','SP_KEY5','IC_CAPTION','IC_ORDER','II_CAPTION','IE_SHORT_DESC','IE_VERSION','II_ORDER','FILE_NAME']
    temp_df = pd.DataFrame(columns=CSV_columns)
    pgm_docs = doc_df[doc_df['PGM_ID'] == int(pgm_id)]
    
    

    for index, doc in pgm_docs.iterrows():
        values = [sp['SP_VALUE']
                , sp['SP_VERSION']
                , sp['CONTEXT']
                , sp['SP_KEY1']
                , sp['SP_KEY2']
                , sp['SP_KEY3']
                , sp['SP_KEY4']
                , sp['SP_KEY5']
                , 'MIS Migrated Data'
                , None
                , 'Documents'
                , 'docDocuments'
                , None
                , None
                , doc['FILE_NAME']
                ]
        temp_df.loc[len(temp_df)] = values
    return temp_df

In [ ]:
def migrateMISScreens():
    pgm_df = pd.read_csv("Raw_Data/MIS_PGM_DATA.csv", na_filter=False, escapechar='\\')
    pgm_df = pgm_df.drop_duplicates()
    pgm_df = pgm_df[pgm_df['MATERIAL_GROUP'].isin(['REINFORCEMENT'])]

    # mat_code_list_normalized = set(mat_code_list)
    # mat_code_list_normalized.update(['RF' + code for code in mat_code_list if not code.startswith('RF')])
    # pgm_df = pgm_df[pgm_df['MATERIAL_CODE'].isin(mat_code_list_normalized)]

    SP_IC_DF = pd.read_csv('Output_CSVs/SP-IC.csv', na_filter=False, escapechar='\\')
    SP_IC_DF = SP_IC_DF.drop_duplicates()

    pgm_df['MATERIAL_CODE'] = pgm_df['MATERIAL_CODE'].apply(lambda x: x if x.startswith('RF') or x == '' else 'RF' + x)

    pgm_df['REVISION_NO'] = pgm_df['REVISION_NO'].astype(str).str.replace('.0', '', regex=False)
    pgm_df['REVISION_NO'] = pgm_df['REVISION_NO'].apply(lambda x : int(x)+1 if x is not '' else '1')
 
 
    pgm_df['Computed_MATERIAL_CODE'] = pgm_df.apply(lambda x: x['EXP_CODE'] if (x['CURRENT_STATUS'] != 'Completed') else x['MATERIAL_CODE'], axis=1)
    pgm_df['Computed_MATERIAL_CODE'] = pgm_df['Computed_MATERIAL_CODE'].apply(lambda x: x if x.startswith('RF') else 'RF' + x)

    sp_df = pd.read_csv("Output_CSVs/SP.csv", na_filter=False, escapechar='\\')
    sp_df = sp_df.drop_duplicates()

    plant_code_df = pd.read_excel('Input_excels/Supplier_Plant_SISCode-241122.xlsx', na_filter=False, engine='openpyxl')

    pgm_df = pd.merge(pgm_df, plant_code_df, left_on=["SUPPLIER_NAME", "SUPPLIER_PLANT", "SUPPLIER_CITY"], right_on=["Supplier MIS", "Plant MIS", "City MIS"], how="left")
    pgm_df = pgm_df.rename(columns={"Supplier MIS":"sis_Supplier", "Plant MIS":"sis_Plant", "SIS Plant code":"sis_SIS_Plant_code"})

    pgm_df = pd.merge(pgm_df, plant_code_df, left_on=["AGENT_NAME", "AGENT_PLANT", "AGENT_CITY"], right_on=["Supplier MIS", "Plant MIS", "City MIS"], how="left")
    pgm_df = pgm_df.rename(columns={"Supplier MIS":"sis_Agent", "Plant MIS":"sis_A_Plant", "SIS Plant code":"sis_Agent_Plant_code"})

    pgm_df['SUPPLIER_NAME'] = pgm_df['Supplier Opcenter_x'].combine_first(pgm_df['SUPPLIER_NAME'])
    pgm_df['SUPPLIER_PLANT'] = pgm_df['Plant Opcenter_x'].combine_first(pgm_df['SUPPLIER_PLANT'])
    pgm_df['AGENT_NAME'] = pgm_df['Supplier Opcenter_y'].combine_first(pgm_df['AGENT_NAME'])
    pgm_df['AGENT_PLANT'] = pgm_df['Plant Opcenter_y'].combine_first(pgm_df['AGENT_PLANT'])
   
    pgm_df['SUPPLIER_NAME'] = pgm_df['SUPPLIER_NAME'].str[:40]
    pgm_df['SUPPLIER_PLANT'] = pgm_df['SUPPLIER_PLANT'].str[:40]
    pgm_df['AGENT_NAME'] = pgm_df['AGENT_NAME'].str[:40]
    pgm_df['AGENT_PLANT'] = pgm_df['AGENT_PLANT'].str[:40]
    pgm_df['SUPPLIER_TRADENAME'] = pgm_df['SUPPLIER_TRADENAME'].str[:40]

    pgm_df['SUPPLIER_NAME'] = pgm_df['SUPPLIER_NAME'].replace('N/A', '')
    pgm_df['SUPPLIER_NAME'] = pgm_df['SUPPLIER_NAME'].replace('n/a', '')

    pgm_df['SUPPLIER_PLANT'] = pgm_df['SUPPLIER_PLANT'].replace('N/A', '')
    pgm_df['SUPPLIER_PLANT'] = pgm_df['SUPPLIER_PLANT'].replace('n/a', '')


    pgm_df['SUPPLIER_TRADENAME'] = pgm_df['SUPPLIER_TRADENAME'].replace('N/A', '')
    pgm_df['SUPPLIER_TRADENAME'] = pgm_df['SUPPLIER_TRADENAME'].replace('n/a', '')
    pgm_df['AGENT_NAME'] = pgm_df['AGENT_NAME'].replace('N/A', '')
    pgm_df['AGENT_NAME'] = pgm_df['AGENT_NAME'].replace('n/a', '')
    pgm_df['AGENT_PLANT'] = pgm_df['AGENT_PLANT'].replace('N/A', '')
    pgm_df['AGENT_PLANT'] = pgm_df['AGENT_PLANT'].replace('n/a', '')

    multi_program_specs = pd.DataFrame(columns=sp_df.keys())

    
    CSV_columns = ['SP_VALUE','SP_VERSION','CONTEXT','SP_KEY1','SP_KEY2','SP_KEY3','SP_KEY4','SP_KEY5','IC_CAPTION','IC_ORDER','II_CAPTION','IE_SHORT_DESC','IE_VERSION','II_ORDER','FILE_NAME']
    SP_II_Documents_OP_DF = pd.DataFrame(columns=CSV_columns)
    pgm_not_match = pd.DataFrame(pgm_df)
    PGM_List = list()
    pgm_df['REVISION_NO'] = pgm_df['REVISION_NO'].astype('str')

    for index, sp in sp_df.iterrows():
        pgm = pgm_df[(pgm_df['Computed_MATERIAL_CODE'] == sp['SP_VALUE'])
                     & (str(pgm_df['REVISION_NO'].iloc[0]) == str(sp['SP_VERSION']))
                     & (pgm_df['SUPPLIER_NAME'] == sp['SP_KEY1'])
                     & (pgm_df['SUPPLIER_PLANT'] == sp['SP_KEY2'])
                     & (pgm_df['AGENT_NAME'] == sp['SP_KEY3'])
                     & (pgm_df['AGENT_PLANT'] == sp['SP_KEY4'])
                     & (pgm_df['SUPPLIER_TRADENAME'] == sp['SP_KEY5'])
                     ]
        
        if (len(pgm['PGM_ID']) == 1) and (sp['CONTEXT'] == 'Supplier'):
 
            PGM_List.append(pgm['PGM_ID'].iloc[0])
 
            if str(pgm['APPROVAL_PROCESS_NAME']) in ['REVISIONS', 'SOURCE CHANGE', 'EMERGENCY']:
                sp['PGM_Number'] = 'SC/ER/RE'
                multi_program_specs = pd.concat([multi_program_specs, sp], ignore_index=True)
                continue
 
            pgm['PGM_ID'] = pgm['PGM_ID'].astype('str')
            sp['PGM_Number'] = ', '.join(pgm['PGM_ID'])
            multi_program_specs = pd.concat([multi_program_specs, sp], ignore_index=True)
            PGM_ID = pgm['PGM_ID'].iloc[0]
            SP_IC_DF = add_migrated_data_infocard(sp, SP_IC_DF)
            SP_II_Documents_OP_DF = pd.concat([SP_II_Documents_OP_DF, map_doc(PGM_ID, sp)], ignore_index=True)
 
        elif (len(pgm['PGM_ID']) > 1) and (sp['CONTEXT'] == 'Supplier'):
            for index, pgm_id in pgm.iterrows():
                PGM_List.append(pgm_id['PGM_ID'])
                if (pgm_id['APPROVAL_PROCESS_NAME'] in ['REVISIONS', 'SOURCE CHANGE', 'EMERGENCY']):
                    sp['PGM_Number'] = 'SC/ER/RE_'+str(pgm_id['PGM_ID'])
                    multi_program_specs = pd.concat([multi_program_specs, sp], ignore_index=True)
                    continue
                else:
                    SP_IC_DF = add_migrated_data_infocard(sp, SP_IC_DF)
                    SP_II_Documents_OP_DF = pd.concat([SP_II_Documents_OP_DF, map_doc(pgm_id['PGM_ID'], sp)], ignore_index=True)
                    sp['PGM_Number'] = pgm_id['PGM_ID']
                    multi_program_specs = pd.concat([multi_program_specs, sp], ignore_index=True)
           
        elif(len(pgm['PGM_ID'])== 0):
            sp['PGM_Number'] = 'No PGM found!'
            multi_program_specs = pd.concat([multi_program_specs, sp], ignore_index=True)
        elif (sp['CONTEXT'] == '' or None):
            pgm['PGM_ID'] = pgm['PGM_ID'].astype('str')
            sp['PGM_Number'] = ', '.join(pgm['PGM_ID'])
            multi_program_specs = pd.concat([multi_program_specs, sp], ignore_index=True)
    
    # considering only first 40 charecters
    SP_II_Documents_OP_DF['SP_KEY1'] = SP_II_Documents_OP_DF['SP_KEY1'].str[:40]
    SP_II_Documents_OP_DF['SP_KEY2'] = SP_II_Documents_OP_DF['SP_KEY2'].str[:40]
    SP_II_Documents_OP_DF['SP_KEY3'] = SP_II_Documents_OP_DF['SP_KEY3'].str[:40]
    SP_II_Documents_OP_DF['SP_KEY4'] = SP_II_Documents_OP_DF['SP_KEY4'].str[:40]
    SP_II_Documents_OP_DF['SP_KEY5'] = SP_II_Documents_OP_DF['SP_KEY5'].str[:40]

    multi_program_specs.to_csv('multi_program_specs.csv', index=False, escapechar='\\', doublequote=False)
    SP_IC_DF['SP_VALUE'] = SP_IC_DF['SP_VALUE'].apply(lambda x: x if x.startswith('RF') else 'RF'+x)
    SP_IC_DF = SP_IC_DF.drop_duplicates()
    SP_IC_DF.to_csv('Output_CSVs/SP-IC.csv', index=False, escapechar='\\', doublequote=False)
    SP_II_Documents_OP_DF['SP_VALUE'] = SP_II_Documents_OP_DF['SP_VALUE'].apply(lambda x: x if x.startswith('RF') else 'RF'+x)
    SP_II_Documents_OP_DF = SP_II_Documents_OP_DF.drop_duplicates()
    SP_II_Documents_OP_DF.to_csv('Output_CSVs/SP-II-Documents(PGM).csv', index=False, escapechar='\\', doublequote=False)
    pgm_not_match = pgm_df[~pgm_df['PGM_ID'].isin(PGM_List)]
    pgm_not_match.to_csv('Output_CSVs/ProgramsNotMigrated.csv', index=False)
    return pgm_df

migrateMISScreens()

# Add New Specs for Programs

In [ ]:
# -------------------------------- update the function AddNewSpecforProgramsNotMigrated
def match_prefix(s, prefixes):
    for prefix in prefixes:
        if s.startswith(prefix):
            return prefix
    return s

counter = 40000

def update_material_code(row):
    global counter 
    if row['Computed_MATERIAL_CODE'] not in (None, '', np.nan):
        return row['Computed_MATERIAL_CODE']
    else:
        result = 'RF0' + str(counter) 
        counter += 1
        return result

def AddNewSpecforProgramsNotMigrated():
    ProgramsNotMigrated = pd.read_csv('Raw_Data/MIS_PGM_DATA.csv', na_filter=False, escapechar='\\')
    ProgramsNotMigrated['EXP_CODE'] = ProgramsNotMigrated['EXP_CODE'].apply(lambda x: 'RF' + x if pd.notna(x) and x != '' and not x.startswith('RF') else x)

    NEW_CODES_DF = pd.read_csv('Input_CSVs/new_exp_codes.csv', na_filter=False, escapechar='\\')

    ProgramsNotMigrated = pd.merge(ProgramsNotMigrated, NEW_CODES_DF, left_on='PGM_ID', right_on='PGM_NO', how='left')
    ProgramsNotMigrated['EXP_CODE'] = ProgramsNotMigrated['EXP_CODE'].where(ProgramsNotMigrated['NEW_EXP_CODE'].isnull(), ProgramsNotMigrated['NEW_EXP_CODE'])
    ProgramsNotMigrated.drop(columns=['NEW_EXP_CODE', 'PGM_NO'], inplace=True)
    ProgramsNotMigrated.to_csv('Raw_Data/MIS_PGM_DATA.csv', index=False, escapechar='\\', doublequote=False)
    SP_OP_DF = pd.read_csv('Input_CSVs/SOS_SPECS.csv', escapechar='\\', doublequote=False)
    SP_DF = pd.DataFrame()

    plant_code_df = pd.read_excel('Input_excels/Supplier_Plant_SISCode-241122.xlsx', na_filter=False, engine='openpyxl')

    ProgramsNotMigrated['PGM_TYPE'] = ProgramsNotMigrated['PGM_NAME'].str[:2]

    status_df = pd.read_excel('Input_excels/mis_program_statuses r1.xlsx', na_filter=False, engine='openpyxl')
    status_df = status_df[['PROGRAM_STAGE', 'TYPE', 'Program_Status','Opcenter Lifecycle', 'Opcenter Status', 'Opcenter Status Handle']]
    status_df['PROGRAM_STAGE'] = status_df['PROGRAM_STAGE'].str.split('-').str[0].str.rstrip()
    StageList = status_df['PROGRAM_STAGE'].unique().tolist()
    StageList = [item for item in StageList if item]
    ProgramsNotMigrated['PROGRAM_STAGE'] = ProgramsNotMigrated['PROGRAM_STAGE'].apply(lambda x: match_prefix(x, StageList))
    

    ProgramsNotMigrated = pd.merge(ProgramsNotMigrated, status_df, 
                                   left_on=['CURRENT_STATUS', 'PROGRAM_STAGE', 'PGM_TYPE'], 
                                   right_on=['Program_Status', 'PROGRAM_STAGE', 'TYPE'],
                                   how='left')

    ProgramsNotMigrated = pd.merge(ProgramsNotMigrated, plant_code_df, left_on=["SUPPLIER_NAME", "SUPPLIER_PLANT", "SUPPLIER_CITY"], right_on=["Supplier MIS", "Plant MIS", "City MIS"], how="left")
    ProgramsNotMigrated = ProgramsNotMigrated.rename(columns={"Supplier MIS":"sis_Supplier", "Plant MIS":"sis_Plant", "SIS Plant code":"sis_SIS_Plant_code"})

    ProgramsNotMigrated = pd.merge(ProgramsNotMigrated, plant_code_df, left_on=["AGENT_NAME", "AGENT_PLANT", "AGENT_CITY"], right_on=["Supplier MIS", "Plant MIS", "City MIS"], how="left")
    ProgramsNotMigrated = ProgramsNotMigrated.rename(columns={"Supplier MIS":"sis_Agent", "Plant MIS":"sis_A_Plant", "SIS Plant code":"sis_Agent_Plant_code"})

    ProgramsNotMigrated['SUPPLIER_NAME'] = ProgramsNotMigrated['Supplier Opcenter_x'].combine_first(ProgramsNotMigrated['SUPPLIER_NAME'])
    ProgramsNotMigrated['SUPPLIER_PLANT'] = ProgramsNotMigrated['Plant Opcenter_x'].combine_first(ProgramsNotMigrated['SUPPLIER_PLANT'])
    ProgramsNotMigrated['AGENT_NAME'] = ProgramsNotMigrated['Supplier Opcenter_y'].combine_first(ProgramsNotMigrated['AGENT_NAME'])
    ProgramsNotMigrated['AGENT_PLANT'] = ProgramsNotMigrated['Plant Opcenter_y'].combine_first(ProgramsNotMigrated['AGENT_PLANT'])

    ProgramsNotMigrated['REVISION_NO'] = ProgramsNotMigrated['REVISION_NO'].astype(str).str.replace('.0', '', regex=False)
    ProgramsNotMigrated['REVISION_NO'] = ProgramsNotMigrated['REVISION_NO'].apply(lambda x : int(x)+1 if x is not '' else 1)



    ProgramsWithSupAgent = ProgramsNotMigrated[
        (~ProgramsNotMigrated[['SUPPLIER_NAME', 'SUPPLIER_PLANT', 'AGENT_NAME', 'AGENT_PLANT', 'SUPPLIER_TRADENAME']].isin(['', None])).any(axis=1)
        & (~ProgramsNotMigrated['APPROVAL_PROCESS_NAME'].isin(['REVISIONS', 'SOURCE CHANGE', 'EMERGENCY']))
        # & (ProgramsNotMigrated['MATERIAL_CODE'].isin(mat_code_list) | ProgramsNotMigrated['EXP_CODE'].isin(mat_code_list))
        & (ProgramsNotMigrated['MATERIAL_GROUP'] == 'REINFORCEMENT')
        ]
    ProgramsNotMigrated['SUPPLIER_NAME'] = ProgramsNotMigrated['SUPPLIER_NAME'].str[:40]
    ProgramsNotMigrated['SUPPLIER_PLANT'] = ProgramsNotMigrated['SUPPLIER_PLANT'].str[:40]
    ProgramsNotMigrated['AGENT_NAME'] = ProgramsNotMigrated['AGENT_NAME'].str[:40]
    ProgramsNotMigrated['AGENT_PLANT'] = ProgramsNotMigrated['AGENT_PLANT'].str[:40]
    ProgramsWithSupAgent['SUPPLIER_TRADENAME'] = ProgramsWithSupAgent['SUPPLIER_TRADENAME'].str[:40]
    
    ProgramsWithSupAgent['Computed_MATERIAL_CODE'] = ProgramsWithSupAgent.apply(lambda x: x['EXP_CODE'] if (x['CURRENT_STATUS'] != 'Completed') else x['MATERIAL_CODE'], axis=1)
    ProgramsWithSupAgent['Computed_MATERIAL_CODE'] = ProgramsWithSupAgent.apply(update_material_code, axis=1)
    
    contextcols = ['Computed_MATERIAL_CODE','REVISION_NO', 'SUPPLIER_NAME', 'SUPPLIER_PLANT', 'AGENT_NAME', 'AGENT_PLANT', 'SUPPLIER_TRADENAME']
    ProgramsWithSupAgent = ProgramsWithSupAgent.sort_values(by='PGM_ID')
    ProgramsWithSupAgent = ProgramsWithSupAgent.drop_duplicates(contextcols, keep='last')
    
    SP_DF['SP_VALUE'] = ProgramsWithSupAgent['Computed_MATERIAL_CODE']
    SP_DF['SP_VERSION'] = ProgramsWithSupAgent['REVISION_NO']
    SP_DF['FR_SHORT_DESC'] = 'SupplierReinforcement'
    SP_DF['FR_VERSION'] = ''
    SP_DF['CONTEXT'] = 'Supplier'
    SP_DF['SP_KEY1'] = ProgramsWithSupAgent['SUPPLIER_NAME']
    SP_DF['SP_KEY2'] = ProgramsWithSupAgent['SUPPLIER_PLANT']
    SP_DF['SP_KEY3'] = ProgramsWithSupAgent['AGENT_NAME']
    SP_DF['SP_KEY4'] = ProgramsWithSupAgent['AGENT_PLANT']
    SP_DF['SP_KEY5'] = ProgramsWithSupAgent['SUPPLIER_TRADENAME']
    SP_DF['STYPE_VALUE'] = 'SRF'
    SP_DF['CREATED_BY'] = 'EventManager'
    SP_DF['LC_DESC'] = ProgramsWithSupAgent['Opcenter Lifecycle']
    SP_DF.loc[SP_DF['LC_DESC'].isnull() | (SP_DF['LC_DESC'] == ''), 'LC_DESC'] = 'Goodyear LC for Material Specifications'
    SP_DF['LC_VERSION'] = ''
    SP_DF['SS_DESC'] = ProgramsWithSupAgent['Opcenter Status Handle']
    SP_DF['CREATED_ON'] = date.today()
    SP_DF['EFFECTIVE_FROM'] = ''
    SP_DF['EFFECTIVE_TILL'] =''
    SP_DF['HAS_ADHOC_APPROVAL'] = ''

    for index, row in SP_DF.iterrows():
        Mat_Code = row['SP_VALUE']
        Version = row['SP_VERSION']
        SP_KEY1 = row['SP_KEY1']
        SP_KEY2 = row['SP_KEY2']
        SP_KEY3 = row['SP_KEY3']
        SP_KEY4 = row['SP_KEY4']
        SP_KEY5 = row['SP_KEY5']
        IsExist = SP_OP_DF[(SP_OP_DF['SP_VALUE'] == Mat_Code)
                   & (SP_OP_DF['SP_VERSION'] == Version)
                   & (SP_OP_DF['SP_KEY1'] == SP_KEY1)
                   & (SP_OP_DF['SP_KEY2'] == SP_KEY2)
                   & (SP_OP_DF['SP_KEY3'] == SP_KEY3)
                   & (SP_OP_DF['SP_KEY4'] == SP_KEY4)
                   & (SP_OP_DF['SP_KEY5'] == SP_KEY5)]

        if(len(IsExist)>0):
            SP_DF.drop(index, inplace=True)
            

    SP_OP_DF = pd.concat([SP_OP_DF, SP_DF], ignore_index=True)

    # considering only first 40 charecters
    SP_OP_DF['SP_KEY1'] = SP_OP_DF['SP_KEY1'].str[:40]
    SP_OP_DF['SP_KEY2'] = SP_OP_DF['SP_KEY2'].str[:40]
    SP_OP_DF['SP_KEY3'] = SP_OP_DF['SP_KEY3'].str[:40]
    SP_OP_DF['SP_KEY4'] = SP_OP_DF['SP_KEY4'].str[:40]
    SP_OP_DF['SP_KEY5'] = SP_OP_DF['SP_KEY5'].str[:40]

    SP_OP_DF = SP_OP_DF.drop_duplicates()

    SP_OP_DF['SP_VALUE'] = SP_OP_DF['SP_VALUE'].apply(lambda x: x if x.startswith('RF') else 'RF'+x)
    key_columns = ["SP_VALUE", "SP_VERSION", "FR_SHORT_DESC", "FR_VERSION", "CONTEXT", "SP_KEY1", "SP_KEY2", "SP_KEY3", "SP_KEY4", "SP_KEY5", "STYPE_VALUE", "CREATED_BY"]
    filtered_df_u = SP_OP_DF[SP_OP_DF["SS_DESC"] == "@U"].drop_duplicates(subset=key_columns, keep="first")
    filtered_df_ia = SP_OP_DF[SP_OP_DF["SS_DESC"] == "IA"].drop_duplicates(subset=key_columns, keep=False)
    remaining_df = SP_OP_DF[~SP_OP_DF["SS_DESC"].isin(["@U", "IA"])]
    combined_df = pd.concat([filtered_df_u, filtered_df_ia, remaining_df], ignore_index=True)
    SP_OP_DF = combined_df.drop_duplicates(subset=key_columns, keep="first")
    SP_OP_DF.drop_duplicates(inplace=True)
    SP_OP_DF.to_csv('Output_CSVs/SP.csv', index=False, escapechar='\\', doublequote=False)

    SP_DF = SP_DF.drop_duplicates(keep='last')
    ProgramsWithSupAgent = ProgramsWithSupAgent.drop_duplicates(keep='last')
    ProgramsWithSupAgent['PGM_REASON'] = (
    ProgramsWithSupAgent['PGM_REASON']
    .apply(lambda x: html.unescape(x) if isinstance(x, str) else x)
    .str.replace("\n", " ")
    .str.replace("\r", "")
    .str.strip()
)
    ProgramsWithSupAgent.to_csv('Input_CSVs/PGM_Data_Specs_mapping.csv', index=False, escapechar='\\', doublequote=False)
    
    return ProgramsWithSupAgent

print('AddNewSpecforProgramsNotMigrated()')
AddNewSpecforProgramsNotMigrated()
migrateMISScreens()
print('migrateMISScreens()')


# Add New base Specs for Programs

In [135]:
def createNewBaseSpecs():
    SP_OP_DF = pd.read_csv('Output_CSVs/SP.csv', na_filter=False, escapechar='\\')
    PGM_Specs_Mapping = AddNewSpecforProgramsNotMigrated()
    ExistingBaseSpecs = SP_OP_DF[SP_OP_DF['STYPE_VALUE'] == 'RF']
    ExistingMaterialCodes = ExistingBaseSpecs['SP_VALUE'].unique().tolist()
    BaseSpecsToBeCreated = SP_OP_DF[(SP_OP_DF['STYPE_VALUE'] == 'SRF')
                                    & (~SP_OP_DF['SP_VALUE'].isin(ExistingMaterialCodes))]
    
    SP_DF = pd.DataFrame()

    PGM_Specs_Mapping = PGM_Specs_Mapping[(PGM_Specs_Mapping['Computed_MATERIAL_CODE'].isin(BaseSpecsToBeCreated['SP_VALUE'].to_list()))
                                          & PGM_Specs_Mapping['MATERIAL_CODE'].isin(['', None])
                                          ]
    
    
    SP_DF['SP_VALUE'] = PGM_Specs_Mapping['Computed_MATERIAL_CODE']
    SP_DF['SP_VERSION'] = PGM_Specs_Mapping['REVISION_NO']
    SP_DF['FR_SHORT_DESC'] = '' 

    def fix_fr_short_desc(row):
      if 'FR_SHORT_DESC' in row.index:
        row['FR_SHORT_DESC'] = 'ReinforcementFabric' if pd.isnull(PGM_Specs_Mapping.loc[row.name, 'MAT_CLASS']) or PGM_Specs_Mapping.loc[row.name, 'CURRENT_STATUS'] == 'Cancelled' else 'ReinforcementFabric'
      return row

    # def fix_fr_short_desc(row):
    #   if 'FR_SHORT_DESC' in row.index:
    #     row['FR_SHORT_DESC'] = 'ReinforcementFabric' if pd.isnull(PGM_Specs_Mapping.loc[row.name, 'MAT_CLASS']) or PGM_Specs_Mapping.loc[row.name, 'CURRENT_STATUS'] == 'Cancelled' else row['FR_SHORT_DESC']
    #   return row

    SP_DF = SP_DF.apply(fix_fr_short_desc, axis=1)
    SP_DF['FR_VERSION'] = ''
    SP_DF['CONTEXT'] = ''
    SP_DF['SP_KEY1'] = ''
    SP_DF['SP_KEY2'] = ''
    SP_DF['SP_KEY3'] = ''
    SP_DF['SP_KEY4'] = ''
    SP_DF['SP_KEY5'] = ''
    SP_DF['STYPE_VALUE'] = 'RF'
    SP_DF['CREATED_BY'] = 'EventManager'
    SP_DF['LC_DESC'] = PGM_Specs_Mapping['Opcenter Lifecycle']
    SP_DF.loc[SP_DF['LC_DESC'].isnull() | (SP_DF['LC_DESC'] == ''), 'LC_DESC'] = 'Goodyear LC for Material Specifications'
    SP_DF['LC_VERSION'] = ''
    SP_DF['SS_DESC'] = 'IA'
    SP_DF['CREATED_ON'] = date.today()
    SP_DF['EFFECTIVE_FROM'] = ''
    SP_DF['EFFECTIVE_TILL'] =''
    SP_DF['HAS_ADHOC_APPROVAL'] = ''

    SP_OP_DF = pd.concat([SP_OP_DF, SP_DF], ignore_index=True)
    SP_OP_DF['SP_VALUE'] = SP_OP_DF['SP_VALUE'].apply(lambda x: x if x.startswith('RF') else 'RF'+x)
    key_columns = ["SP_VALUE", "SP_VERSION", "FR_SHORT_DESC", "FR_VERSION", "CONTEXT", "SP_KEY1", "SP_KEY2", "SP_KEY3", "SP_KEY4", "SP_KEY5", "STYPE_VALUE", "CREATED_BY"]
    filtered_df_u = SP_OP_DF[SP_OP_DF["SS_DESC"] == "@U"].drop_duplicates(subset=key_columns, keep="first")
    filtered_df_ia = SP_OP_DF[SP_OP_DF["SS_DESC"] == "IA"].drop_duplicates(subset=key_columns, keep=False)
    remaining_df = SP_OP_DF[~SP_OP_DF["SS_DESC"].isin(["@U", "IA"])]
    combined_df = pd.concat([filtered_df_u, filtered_df_ia, remaining_df], ignore_index=True)
    SP_OP_DF = combined_df.drop_duplicates(subset=key_columns, keep="first")
    SP_OP_DF = SP_OP_DF.drop_duplicates()
    SP_OP_DF.to_csv('Output_CSVs/SP.csv', index=False, escapechar='\\', doublequote=False)
    MAT_OP_DF = pd.read_csv('Output_CSVs/Material.csv', na_filter=False, escapechar='\\')
    MAT_DF = pd.DataFrame()

    MAT_DF['MA_VALUE'] = PGM_Specs_Mapping['Computed_MATERIAL_CODE']
    MAT_DF['DESCRIPTION'] = PGM_Specs_Mapping['RMI_DESC']
    MAT_DF['BASE_UOM'] = 'g'
    MAT_DF['MA_SOURCE'] = None
    MAT_DF['BASE_CONV_FACTOR'] = None
    MAT_DF['BASE_TO_UNIT'] = None
    MAT_DF['BASE_QUANTITY'] = None
    MAT_DF['DATE_IMPORTED'] = date.today()
    MAT_DF['ACTIVE'] = 1

    MAT_OP_DF = pd.concat([MAT_OP_DF, MAT_DF], ignore_index=True)
    MAT_OP_DF = MAT_OP_DF.drop_duplicates()
    MAT_OP_DF.to_csv('Output_CSVs/Material.csv', index=False, escapechar='\\', doublequote=False)
    return SP_DF

In [ ]:
def populateExtraInfocards():
    RM_BASIC = "Reinforcement - Basic"
    SAP_STATUS = 'Active (Non Development line-up)'
    infoFields = [
        {"IC_CAPTION": RM_BASIC, "II_CAPTION":"Construction Code - Fabric", "IE_SHORT_DESC":"cbConCodeFabric", "DB_COLUMN":"Fabric Construction Code"}
        , {"IC_CAPTION": RM_BASIC, "II_CAPTION":"Construction Code - Square Woven", "IE_SHORT_DESC":"cbConCodeSqWov", "DB_COLUMN":"Fabric Squarewoven Construction Code"}
        , {"IC_CAPTION": RM_BASIC, "II_CAPTION":"Construction Code - Wire/Cable", "IE_SHORT_DESC":"cbConCodeWirCab", "DB_COLUMN":"Cable Construction Code"}
        , {"IC_CAPTION": RM_BASIC, "II_CAPTION":"Processing Code", "IE_SHORT_DESC":"cbFabricProcCode", "DB_COLUMN":"Fabric Processing Code"}
        , {"IC_CAPTION": RM_BASIC, "II_CAPTION":"Cord Density Code", "IE_SHORT_DESC":"cbCordDensCode", "DB_COLUMN":"Cord Density Code"}
        , {"IC_CAPTION": RM_BASIC, "II_CAPTION":"EPI", "IE_SHORT_DESC":"cbEPI", "DB_COLUMN":"Calculated Ends per Inch"}
        , {"IC_CAPTION": RM_BASIC, "II_CAPTION":"Fill Cord (Type)", "IE_SHORT_DESC":"ddlFillCord", "DB_COLUMN":"Fill Cord"}
        , {"IC_CAPTION": RM_BASIC, "II_CAPTION":"Linear Fill Cord Weight (DEN)", "IE_SHORT_DESC":"efLinCorWt", "DB_COLUMN":"Linear Fill Cord Weight DEN"}
        , {"IC_CAPTION": RM_BASIC, "II_CAPTION":"Picks per Meter", "IE_SHORT_DESC":"efPicksPerMeter", "DB_COLUMN":"efPicksPerMeter"}
        , {"IC_CAPTION": RM_BASIC, "II_CAPTION":"SAP/SOS Description", "IE_SHORT_DESC":"efSAPSOSDesc", "DB_COLUMN":"Material description"}
        , {"IC_CAPTION": RM_BASIC, "II_CAPTION":"Update Spec Description (hidden)", "IE_SHORT_DESC":"ieSAPSpecDesc", "DB_COLUMN":""}
        , {"IC_CAPTION": RM_BASIC, "II_CAPTION":"Cable Bead Inside Diameter (mm)", "IE_SHORT_DESC":"efCBInsideDia", "DB_COLUMN":"Bead Inside Diameter"}
        , {"IC_CAPTION": RM_BASIC, "II_CAPTION":"Cable Bead Comp. Weight (kg)", "IE_SHORT_DESC":"efCBCompWeight", "DB_COLUMN":"Component Weight"}
        , {"IC_CAPTION": RM_BASIC, "II_CAPTION":"Cable Bead Gauge (mm)", "IE_SHORT_DESC":"efCBGauge", "DB_COLUMN":"Gauge"}
        , {"IC_CAPTION": RM_BASIC, "II_CAPTION":"Cable Bead Coating", "IE_SHORT_DESC":"ddlCBCoating", "DB_COLUMN":"Cable Bead Coating"}
        ]


    RF_GENERAL = "Reinforcement General Information"
    infoFieldsGeneral = [
        {"IC_CAPTION": RF_GENERAL, "II_CAPTION":"Material Class", "IE_SHORT_DESC":"ieMaterialClass", "DB_COLUMN":"MAT_CLASS"}
        , {"IC_CAPTION": RF_GENERAL, "II_CAPTION":"Material Subclass", "IE_SHORT_DESC":"ieMaterialSubclass", "DB_COLUMN":"MAT_SUBCLASS"}
        , {"IC_CAPTION": RF_GENERAL, "II_CAPTION":"Material Code", "IE_SHORT_DESC":"cbMaterialCode", "DB_COLUMN":"Computed_MATERIAL_CODE"}
        , {"IC_CAPTION": RF_GENERAL, "II_CAPTION":"SAP Material Type", "IE_SHORT_DESC":"cbGBSMatType", "DB_COLUMN":"MATERIAL_TYPE"}
        , {"IC_CAPTION": RF_GENERAL, "II_CAPTION":"SAP Secondary Material Type", "IE_SHORT_DESC":"cbGBSMatSecType", "DB_COLUMN":"MATERIAL_SUBTYPE"}
        , {"IC_CAPTION": RF_GENERAL, "II_CAPTION":"SAP Type of Component", "IE_SHORT_DESC":"cbGBSTypeComp", "DB_COLUMN":"COMPONENT_TYPE"}
        , {"IC_CAPTION": RF_GENERAL, "II_CAPTION":"Initial Exp Spec Code (hidden)", "IE_SHORT_DESC":"ieInitialExpSpCode", "DB_COLUMN":"EXP_CODE"}
        , {"IC_CAPTION": RF_GENERAL, "II_CAPTION":"", "IE_SHORT_DESC":"ieSAPStatusCode", "DB_COLUMN":"Y1"}
        , {"IC_CAPTION": RF_GENERAL, "II_CAPTION":"SAP Status", "IE_SHORT_DESC":"ddlSAPStatusDesc", "DB_COLUMN":"Active (Non Development line-up)"}
        ]
    pgm_mapping_df = pd.read_csv('Input_CSVs/PGM_Data_Specs_mapping.csv', na_filter=False, escapechar='\\')
    pgm_mapping_df['Computed_MATERIAL_CODE'] = pgm_mapping_df['Computed_MATERIAL_CODE'].apply(lambda x: x if x.startswith('RF') else 'RF' + x)
    pgm_mapping_df['STAINING'] = pgm_mapping_df['STAINING'].map({'Y':'Yes', 'N':'No'})
    pgm_mapping_df['COMPATIBLE'] = pgm_mapping_df['COMPATIBLE'].map({'Y':'Yes', 'N':'No'})
    pgm_mapping_df['MATERIAL_FORM'] = pgm_mapping_df['MATERIAL_FORM'].map({'S':'Solid', 'L':'Liquid', 'P':'Paste', 'G':'Gas'})

    SP_DF = createNewBaseSpecs()

    # Code to populate SP-AU for new base specs
    SP_AU_OP_DF = pd.read_csv('Output_CSVs/SP-AU.csv', na_filter=False, escapechar='\\')
    SP_AU_NEW = pd.DataFrame(columns=SP_AU_OP_DF.keys())
    SP_II_OP_DF = pd.read_csv('Output_CSVs/SP-II.csv', na_filter=False, escapechar='\\')
    SP_II_NEW = pd.DataFrame()
    for index, SP in SP_DF.iterrows():
        pgm_row = pgm_mapping_df[pgm_mapping_df['Computed_MATERIAL_CODE'] == SP['SP_VALUE']]
        AU_1 = [pgm_row['Computed_MATERIAL_CODE'].iloc[0],
                pgm_row['REVISION_NO'].iloc[0],
                None,
                None,
                None,
                None,
                None,
                None,
                'auExpSpecCode',
                None,
                None,
                pgm_row['EXP_CODE'].iloc[0]]

        
        SP_AU_NEW.loc[len(SP_AU_NEW)] = AU_1

        AU_2 = [pgm_row['Computed_MATERIAL_CODE'].iloc[0], pgm_row['REVISION_NO'].iloc[0], None, None, None, None, None, None,'auSpecType', None, None, 'Reinforcement']
        SP_AU_NEW.loc[len(SP_AU_NEW)] = AU_2

        AU_3 = [pgm_row['Computed_MATERIAL_CODE'].iloc[0], pgm_row['REVISION_NO'].iloc[0], None, None, None, None, None, None,'CodeMaskPrefix', None, None, 'N']
        SP_AU_NEW.loc[len(SP_AU_NEW)] = AU_3
        for field in infoFields:
            SP_II = pd.DataFrame()
            SP_II['SP_VALUE'] = pgm_row['Computed_MATERIAL_CODE']
            SP_II['SP_VERSION'] = pgm_row['REVISION_NO']
            SP_II['CONTEXT'] = ''
            SP_II['SP_KEY1'] = ''
            SP_II['SP_KEY2'] = ''
            SP_II['SP_KEY3'] = ''
            SP_II['SP_KEY4'] = ''
            SP_II['SP_KEY5'] = ''
            SP_II['IC_CAPTION'] = field['IC_CAPTION']
            SP_II['IC_ORDER'] = None
            SP_II['II_CAPTION'] = field['II_CAPTION']
            SP_II['IE_SHORT_DESC'] = field['IE_SHORT_DESC']
            SP_II['IE_VERSION'] = None
            SP_II['II_ORDER'] = None
            ExceptionList = ['ddlSAPCPIValue', 'ieSAPStatusCode','ddlSAPStatusDesc', 'ckThirdPartyConf']
            SP_II['IIVALUE'] = None

            if(field['IE_SHORT_DESC'] not in ExceptionList):
                SP_II['IIVALUE'] = field['DB_COLUMN']
            SP_II_NEW = pd.concat([SP_II_NEW, SP_II], ignore_index=True)
        
        for field in infoFieldsGeneral:
            SP_II = pd.DataFrame()
            SP_II['SP_VALUE'] = pgm_row['Computed_MATERIAL_CODE']
            SP_II['SP_VERSION'] = pgm_row['REVISION_NO']
            SP_II['CONTEXT'] = ''
            SP_II['SP_KEY1'] = ''
            SP_II['SP_KEY2'] = ''
            SP_II['SP_KEY3'] = ''
            SP_II['SP_KEY4'] = ''
            SP_II['SP_KEY5'] = ''
            SP_II['IC_CAPTION'] = field['IC_CAPTION']
            SP_II['IC_ORDER'] = None
            SP_II['II_CAPTION'] = field['II_CAPTION']
            SP_II['IE_SHORT_DESC'] = field['IE_SHORT_DESC']
            SP_II['IE_VERSION'] = None
            SP_II['II_ORDER'] = None
            ExceptionList = ['ddlSAPCPIValue', 'ieSAPStatusCode','ddlSAPStatusDesc', 'ckThirdPartyConf']
            SP_II['IIVALUE'] = None\
            
            if(field['IE_SHORT_DESC'] not in ExceptionList):
                SP_II['IIVALUE'] = pgm_row[field['DB_COLUMN']]
            else:
                if field['IE_SHORT_DESC'] == 'ddlSAPCPIValue':
                    CPI_df = pd.read_excel('Input_excels/CPI_logic.xlsx', na_filter=False, engine='openpyxl')
                    if (len(pgm_row) > 0) and (pgm_row['CLASS'].iloc[0] and pgm_row['SUB_CLASS'].iloc[0] != ''):
                        CPI_Rec = CPI_df[(CPI_df['Class'] == pgm_row['CLASS'].iloc[0]) & (CPI_df['Subclass'] == pgm_row['SUB_CLASS'].iloc[0])]
                        if(len(CPI_Rec)>0):
                            SP_II['IIVALUE'] = CPI_Rec['CPI Value']
                    else:
                        SP_II['IIVALUE'] = 1
                        
                if field['IE_SHORT_DESC'] == 'ieSAPStatusCode':
                    SP_II['IIVALUE'] = "Y1"
                if field['IE_SHORT_DESC'] == 'ddlSAPStatusDesc':
                    SP_II['IIVALUE'] = "Active (Non Development line-up)"
                if field['IE_SHORT_DESC'] == 'ckThirdPartyConf':
                    IIVALUE = not pgm_row['SUPPLIER_NAME'].isin(["GOODYEAR TIRE AND RUBBER"]).any()
                    if(IIVALUE):
                        SP_II['IIVALUE'] = 1
                    else:
                        SP_II['IIVALUE'] = 0
            SP_II_NEW = pd.concat([SP_II_NEW, SP_II], ignore_index=True)
            
    SP_AU_OP_DF = pd.concat([SP_AU_OP_DF, SP_AU_NEW], ignore_index=True)
    SP_AU_OP_DF['SP_VALUE'] = SP_AU_OP_DF['SP_VALUE'].apply(lambda x: x if x.startswith('RF') else 'RF'+x)
    SP_AU_OP_DF = SP_AU_OP_DF.drop_duplicates()
    SP_AU_OP_DF.to_csv('Output_CSVs/SP-AU.csv', index=False, escapechar='\\', doublequote=False)    
    SP_II_OP_DF = pd.concat([SP_II_OP_DF, SP_II_NEW])
    shortencontext(SP_II_OP_DF).drop_duplicates().to_csv('Output_CSVs/SP-II.csv', index=False, escapechar='\\', doublequote=False)

    return SP_II_NEW

print('populateExtraInfocards()')
populateExtraInfocards()

In [137]:
def shorten_filename(filename, max_length=80):
    parts = filename.split('_', 1)
    if len(parts) < 2:
        return filename  

    number = parts[0]  
    title_and_location = parts[1].rsplit('.', 1)  

    if len(title_and_location) < 2:
        return filename  

    title = title_and_location[0].strip()  
    extension = '.' + title_and_location[1]  
    new_filename = f"{number}_{title}{extension}"

    if len(new_filename) > max_length:
        available_length = max_length - len(number) - len(extension) - 1
        title = title[:available_length].rsplit(' ', 1)[0]  
        new_filename = f"{number}_{title}{extension}"

    return new_filename

In [ ]:
def PopulateSpIiforNewSupplierSpecs():
    Pgm_Sup_Df = pd.read_csv('Input_CSVs/PGM_Data_Specs_mapping.csv', na_filter=False, escapechar='\\')
    SOS_DF = pd.read_csv('Input_CSVs/SOS_SPECS.csv', na_filter=False, escapechar='\\')

    # mat_code_list_normalized = set(mat_code_list)
    # mat_code_list_normalized.update(['RF' + code for code in mat_code_list if not code.startswith('RF')])

    Pgm_Sup_Df = Pgm_Sup_Df[
        # Pgm_Sup_Df['MATERIAL_CODE'].isin(mat_code_list_normalized)
        Pgm_Sup_Df['MATERIAL_GROUP'].isin(["REINFORCEMENT"])
        & ~Pgm_Sup_Df['APPROVAL_PROCESS_NAME'].isin(['SOURCE CHANGE', 'EMERGENCY', 'REVISIONS'])
        ]
    Pgm_Sup_Df["APPROVAL_PROCESS_NAME_TC"] = Pgm_Sup_Df["APPROVAL_PROCESS_NAME"].apply(lambda x: x.title())
    IC_Caption = "Supplier/Agent Info"
    infoFields = [
        {"IC_CAPTION": IC_Caption, "II_CAPTION":"Initial Exp Spec Code (hidden)", "IE_SHORT_DESC":"ieInitialExpSpCode", "DB_COLUMN":"EXP_CODE"}
        , {"IC_CAPTION": IC_Caption, "II_CAPTION":"Material Code", "IE_SHORT_DESC":"cbMaterialCode", "DB_COLUMN":"Computed_MATERIAL_CODE"}
        , {"IC_CAPTION": IC_Caption, "II_CAPTION":"Material Description", "IE_SHORT_DESC":"ieMatDescSupplier", "DB_COLUMN":"RMI_DESC"}
        , {"IC_CAPTION": IC_Caption, "II_CAPTION":"Supplier", "IE_SHORT_DESC":"ieSupplier", "DB_COLUMN":"SUPPLIER_NAME"}
        , {"IC_CAPTION": IC_Caption, "II_CAPTION":"Agent", "IE_SHORT_DESC":"ieAgent", "DB_COLUMN":"AGENT_NAME"}
        , {"IC_CAPTION": IC_Caption, "II_CAPTION":"Supplier Plant", "IE_SHORT_DESC":"ieSupplierPlant", "DB_COLUMN":"SUPPLIER_PLANT"}
        , {"IC_CAPTION": IC_Caption, "II_CAPTION":"Agent Plant", "IE_SHORT_DESC":"ieAgentPlant", "DB_COLUMN":"AGENT_PLANT"}
        , {"IC_CAPTION": IC_Caption, "II_CAPTION":"Supplier Plant - City", "IE_SHORT_DESC":"ieSupplierPlantLocationCity", "DB_COLUMN":"SUPPLIER_CITY"}
        , {"IC_CAPTION": IC_Caption, "II_CAPTION":"Agent Plant - City", "IE_SHORT_DESC":"ieAgentPlantLocationCity", "DB_COLUMN":"AGENT_CITY"}
        , {"IC_CAPTION": IC_Caption, "II_CAPTION":"Supplier Plant - State", "IE_SHORT_DESC":"ieSupplierPlantLocationState", "DB_COLUMN":"SUPPLIER_STATE"}
        , {"IC_CAPTION": IC_Caption, "II_CAPTION":"Agent Plant - State", "IE_SHORT_DESC":"ieAgentPlantLocationState", "DB_COLUMN":"AGENT_STATE"}
        , {"IC_CAPTION": IC_Caption, "II_CAPTION":"Agent Plant - Country", "IE_SHORT_DESC":"ieAgentPlantLocationCountry", "DB_COLUMN":"AGENT_COUNTRY"}
        , {"IC_CAPTION": IC_Caption, "II_CAPTION":"Supplier Plant - Country", "IE_SHORT_DESC":"ieSupplierPlantLocationCountry", "DB_COLUMN":"SUPPLIER_COUNTRY"}
        , {"IC_CAPTION": IC_Caption, "II_CAPTION":"Plant Code", "IE_SHORT_DESC":"ieSISPlantCode", "DB_COLUMN":"sis_SIS_Plant_code"}
        , {"IC_CAPTION": IC_Caption, "II_CAPTION":"Plant Code", "IE_SHORT_DESC":"ieAgentPlantCode", "DB_COLUMN":"sis_Agent_Plant_code"}
        , {"IC_CAPTION": IC_Caption, "II_CAPTION":"Supplier Trade Name", "IE_SHORT_DESC":"ieSupplierTradeName", "DB_COLUMN":"SUPPLIER_TRADENAME"}
        , {"IC_CAPTION": IC_Caption, "II_CAPTION":"Agent to Break Bulk?", "IE_SHORT_DESC":"cbBreakBulk", "DB_COLUMN":"AGENT_BUILD_BREAK"}
    ]
    IC_PGM_INFO = 'Program Info'
    programInfoFields = [
        {"IC_CAPTION": IC_PGM_INFO, 'II_CAPTION':'Reason for Program', 'IE_SHORT_DESC':'mleReasProg', 'Conatins':'Reason for Program', "DB_COLUMN":"PGM_REASON"},
        {"IC_CAPTION": IC_PGM_INFO, 'II_CAPTION':'Program Type', 'IE_SHORT_DESC':'ddlProgramType', 'Conatins':'Program Type', "DB_COLUMN":"APPROVAL_PROCESS_NAME_TC"}, # important
        {"IC_CAPTION": IC_PGM_INFO, 'II_CAPTION':'Requested By', 'IE_SHORT_DESC':'cbReqBy', 'Conatins':'Requested By', "DB_COLUMN":"REQUESTER"}, # important
        {"IC_CAPTION": IC_PGM_INFO, 'II_CAPTION':'Assigned To', 'IE_SHORT_DESC':'mleAssignedTo', 'Conatins':'Assigned To', "DB_COLUMN":"ASSIGNEE"}, # important
        {"IC_CAPTION": IC_PGM_INFO, 'II_CAPTION':'NA', 'IE_SHORT_DESC':'cbRegionNA', 'Conatins':'NA', "DB_COLUMN":"REGION_NAME"},
        {"IC_CAPTION": IC_PGM_INFO, 'II_CAPTION':'LA', 'IE_SHORT_DESC':'cbRegionLA', 'Conatins':'LA', "DB_COLUMN":"REGION_NAME"},
        {"IC_CAPTION": IC_PGM_INFO, 'II_CAPTION':'EMEA', 'IE_SHORT_DESC':'cbRegionEMEA', 'Conatins':'EMEA', "DB_COLUMN":"REGION_NAME"},
        {"IC_CAPTION": IC_PGM_INFO, 'II_CAPTION':'Cooper Americas', 'IE_SHORT_DESC':'cbRegionCooperNA', 'Conatins':'Cooper Americas', "DB_COLUMN":"REGION_NAME"},
        {"IC_CAPTION": IC_PGM_INFO, 'II_CAPTION':'AP', 'IE_SHORT_DESC':'cbRegionAP', 'Conatins':'AP', "DB_COLUMN":"REGION_NAME"},
        {"IC_CAPTION": IC_PGM_INFO, 'II_CAPTION':'Cooper EMEA', 'IE_SHORT_DESC':'cbRegionCooperEMEA', 'Conatins':'Cooper EMEA', "DB_COLUMN":"REGION_NAME"},
        {"IC_CAPTION": IC_PGM_INFO, 'II_CAPTION':'Cooper AP', 'IE_SHORT_DESC':'cbRegionCooperAP', 'Conatins':'Cooper AP', "DB_COLUMN":"REGION_NAME"},
        {"IC_CAPTION": IC_PGM_INFO, 'II_CAPTION':'Proposed Plant for Trial', 'IE_SHORT_DESC':'cbProposedPlant', 'Conatins':'Proposed Plant for Trial', "DB_COLUMN":"PLANT_FOR_TRIAL"},
        {"IC_CAPTION": IC_PGM_INFO, 'II_CAPTION':'Akron', 'IE_SHORT_DESC':'cbLabAkron', 'Conatins':'Akron', "DB_COLUMN":"Computed_MATERIAL_CODE"}, # Exception
        {"IC_CAPTION": IC_PGM_INFO, 'II_CAPTION':'Lux', 'IE_SHORT_DESC':'cbLabLux', 'Conatins':'Lux', "DB_COLUMN":"Computed_MATERIAL_CODE"}, # Exception
        {"IC_CAPTION": IC_PGM_INFO, 'II_CAPTION':'Kunshan', 'IE_SHORT_DESC':'cbLabKunshan', 'Conatins':'Kunshan', "DB_COLUMN":"Computed_MATERIAL_CODE"}, # Exception
        {"IC_CAPTION": IC_PGM_INFO, 'II_CAPTION':'EHS Approval Region (hidden)', 'IE_SHORT_DESC':'efEHSApprovalReg', 'Conatins':'EHS Approval Region (hidden)', "DB_COLUMN":"Computed_MATERIAL_CODE"}, # Exception
        {"IC_CAPTION": IC_PGM_INFO, 'II_CAPTION':'SAP Status', 'IE_SHORT_DESC':'ddlSAPStatusDesc', 'Conatins':'SAP Status', "DB_COLUMN":"Computed_MATERIAL_CODE"}, # Exception
        {"IC_CAPTION": IC_PGM_INFO, 'II_CAPTION':'', 'IE_SHORT_DESC':'ieSAPStatusCode', 'Conatins':'.', "DB_COLUMN":"Computed_MATERIAL_CODE"} # Exception
        ]
    IC_GPS_SUP = 'GPS Full Supplier Approval Review'
    IP_SHORT_DESC_sup = 'ipFullApprovalInformation'
    IC_ORDER_sup = '7'
    GPSSuppInfoFields = [
        {"IC_CAPTION": IC_GPS_SUP, 'II_CAPTION':'NA', 'IE_SHORT_DESC':'cbRegionNA', 'Conatins':'NA', "DB_COLUMN":"REGION_NAME"},
        {"IC_CAPTION": IC_GPS_SUP, 'II_CAPTION':'LA', 'IE_SHORT_DESC':'cbRegionLA', 'Conatins':'LA', "DB_COLUMN":"REGION_NAME"},
        {"IC_CAPTION": IC_GPS_SUP, 'II_CAPTION':'AP', 'IE_SHORT_DESC':'cbRegionAP', 'Conatins':'AP', "DB_COLUMN":"REGION_NAME"},
        {"IC_CAPTION": IC_GPS_SUP, 'II_CAPTION':'EMEA', 'IE_SHORT_DESC':'cbRegionEMEA', 'Conatins':'EMEA', "DB_COLUMN":"REGION_NAME"},
        {"IC_CAPTION": IC_GPS_SUP, 'II_CAPTION':'Cooper Americas', 'IE_SHORT_DESC':'cbRegionCooperNA', 'Conatins':'Cooper Americas', "DB_COLUMN":"REGION_NAME"},
        {"IC_CAPTION": IC_GPS_SUP, 'II_CAPTION':'Cooper EMEA', 'IE_SHORT_DESC':'cbRegionCooperEMEA', 'Conatins':'Cooper Europe', "DB_COLUMN":"REGION_NAME"},
        {"IC_CAPTION": IC_GPS_SUP, 'II_CAPTION':'Cooper AP', 'IE_SHORT_DESC':'cbRegionCooperAP', 'Conatins':'Cooper Asia', "DB_COLUMN":"REGION_NAME"},
        {"IC_CAPTION": IC_GPS_SUP, 'II_CAPTION':'SOS Appr. Reg. for Layout', 'IE_SHORT_DESC':'efSOSApprovalReg', 'Conatins':'', "DB_COLUMN":"APPROVAL_REGIONS"},
        {"IC_CAPTION": IC_GPS_SUP, 'II_CAPTION':None, 'IE_SHORT_DESC':'mlePlantMatProh', 'Conatins':None, "DB_COLUMN":"RESTRICTED_PLANTS"},
        {"IC_CAPTION": IC_GPS_SUP, 'II_CAPTION':'Full Approval Prefix Code (hidden)', 'IE_SHORT_DESC':'efFullAppCode', 'Conatins':'', "DB_COLUMN":""}
    ]
    
    IC_AGENT_GPS = 'GPS Full Agent Approval Review'
    IP_SHORT_DESC = 'ipFullAgentApprovalInformation'
    IC_ORDER = '5'
    GPSAgentInfoFields = [
        {"IC_CAPTION": IC_AGENT_GPS, 'II_CAPTION':'NA', 'IE_SHORT_DESC':'cbRegionNA', 'Conatins':'NA', "DB_COLUMN":"REGION_NAME"},
        {"IC_CAPTION": IC_AGENT_GPS, 'II_CAPTION':'LA', 'IE_SHORT_DESC':'cbRegionLA', 'Conatins':'LA', "DB_COLUMN":"REGION_NAME"},
        {"IC_CAPTION": IC_AGENT_GPS, 'II_CAPTION':'AP', 'IE_SHORT_DESC':'cbRegionAP', 'Conatins':'AP', "DB_COLUMN":"REGION_NAME"},
        {"IC_CAPTION": IC_AGENT_GPS, 'II_CAPTION':'EMEA', 'IE_SHORT_DESC':'cbRegionEMEA', 'Conatins':'EMEA', "DB_COLUMN":"REGION_NAME"},
        {"IC_CAPTION": IC_AGENT_GPS, 'II_CAPTION':'Cooper Americas', 'IE_SHORT_DESC':'cbRegionCooperNA', 'Conatins':'Cooper Americas', "DB_COLUMN":"REGION_NAME"},
        {"IC_CAPTION": IC_AGENT_GPS, 'II_CAPTION':'Cooper EMEA', 'IE_SHORT_DESC':'cbRegionCooperEMEA', 'Conatins':'Cooper Europe', "DB_COLUMN":"REGION_NAME"},
        {"IC_CAPTION": IC_AGENT_GPS, 'II_CAPTION':'Cooper AP', 'IE_SHORT_DESC':'cbRegionCooperAP', 'Conatins':'Cooper Asia', "DB_COLUMN":"REGION_NAME"},
        {"IC_CAPTION": IC_AGENT_GPS, 'II_CAPTION':'SOS Appr. Reg. for Layout - Agent', 'IE_SHORT_DESC':'efSOSAgentApprovalReg', 'Conatins':'', "DB_COLUMN":"APPROVAL_REGIONS"},
        {"IC_CAPTION": IC_AGENT_GPS, 'II_CAPTION':None, 'IE_SHORT_DESC':'mlePlantMatProh', 'Conatins':'', "DB_COLUMN":"RESTRICTED_PLANTS"},
    ]

    IC_DOCUMENTS = 'Documents'
    DocumentsInfoFields = [
        {"IC_CAPTION": IC_DOCUMENTS, 'II_CAPTION':'General Documents', 'IE_SHORT_DESC':'docDocuments', "DB_COLUMN":"REGION_NAME"}

    ]

    ExceptionList = ['cbLabAkron', 'cbLabLux', 'cbLabKunshan', 'efEHSApprovalReg', 'ddlSAPStatusDesc', 'ieSAPStatusCode']
    regions = ['cbRegionNA', 'cbRegionLA', 'cbRegionEMEA', 'cbRegionCooperNA', 'cbRegionAP', 'cbRegionCooperEMEA', 'cbRegionCooperAP']
    
    SP_II_OP_DF = pd.read_csv('Output_CSVs/SP-II.csv', na_filter=False, escapechar='\\')
    SP_II_OP_DF = SP_II_OP_DF.drop_duplicates()
    
    SP_IC_DF = pd.read_csv('Output_CSVs/SP-IC.csv', na_filter=False, escapechar='\\')
    SP_IC_DF = SP_IC_DF.drop_duplicates()

    PGM_REG_DF = pd.read_csv('Raw_Data/GPS_Regions_Plants.csv', na_filter=False, escapechar='\\')


    SP_II_DOCUMENT = pd.read_csv('Output_CSVs/SP-II-Document.csv', na_filter=False, escapechar='\\')
    SP_II_DOCUMENT = SP_II_DOCUMENT.drop_duplicates()


    PGM_DOCUMENT_DF = pd.read_csv('Raw_Data/MIS_PGM_DOCUMENTS.csv', na_filter=False, escapechar='\\')
    PGM_DOCUMENT_DF['FILE_NAME'] = PGM_DOCUMENT_DF['FILE_PATH'].str.split('/').str[-1]
    PGM_DOCUMENT_DF = PGM_DOCUMENT_DF.drop_duplicates()
    Files_not_fit = pd.DataFrame(columns=['PGM_ID', 'OLD_NAME', 'NEW_NAME'])
    

    plant_code_df = pd.read_excel('Input_excels/Supplier_Plant_SISCode-241122.xlsx', na_filter=False, engine='openpyxl')

    merge_df3 = pd.merge(PGM_REG_DF, plant_code_df, left_on= 'PROH_PLANT_NAME', right_on='Plant MIS')
    PGM_REG_DF['PROH_PLANT_NAME'] = merge_df3['Plant MIS'].combine_first(PGM_REG_DF['PROH_PLANT_NAME'])

    SP_AU_OP_DF = pd.read_csv('Output_CSVs/SP-AU.csv', na_filter=False, escapechar='\\')
    SP_AU_NEW = pd.DataFrame(columns=SP_AU_OP_DF.keys())

    SP_II_NEW = pd.DataFrame(columns=SP_II_OP_DF.keys())
    

    Pgm_Sup_Df['Computed_MATERIAL_CODE'] = Pgm_Sup_Df['Computed_MATERIAL_CODE'].apply(lambda x: x if x.startswith('RF') else 'RF'+x)
    for index, Pgm_row in Pgm_Sup_Df.iterrows():
        SP = SOS_DF[
            (SOS_DF['SP_VALUE'] == Pgm_row['Computed_MATERIAL_CODE'])
            & (SOS_DF['SP_VERSION'] == Pgm_row['REVISION_NO'])
            & (SOS_DF['CONTEXT'] == 'Supplier')
            & (SOS_DF['SP_KEY1'] == Pgm_row['SUPPLIER_NAME'])
            & (SOS_DF['SP_KEY2'] == Pgm_row['SUPPLIER_PLANT'])
            & (SOS_DF['SP_KEY3'] == Pgm_row['AGENT_NAME'])
            & (SOS_DF['SP_KEY4'] == Pgm_row['AGENT_PLANT'])
            & (SOS_DF['SP_KEY5'] == Pgm_row['SUPPLIER_TRADENAME'])
            ]
        if len(SP) > 0:
            Pgm_Sup_Df = Pgm_Sup_Df.drop(index)
        else:
            
            AU_1 = [Pgm_row['Computed_MATERIAL_CODE'],
                    Pgm_row['REVISION_NO'],
                    'Supplier',
                    Pgm_row['SUPPLIER_NAME'],
                    Pgm_row['SUPPLIER_PLANT'],
                    Pgm_row['AGENT_NAME'],
                    Pgm_row['AGENT_PLANT'],
                    Pgm_row['SUPPLIER_TRADENAME'],
                    'auExpSpecCode',
                    None,
                    None,
                    Pgm_row['EXP_CODE']]

            
            SP_AU_NEW.loc[len(SP_AU_NEW)] = AU_1

            AU_2 = [Pgm_row['Computed_MATERIAL_CODE'], Pgm_row['REVISION_NO'], 'Supplier', Pgm_row['SUPPLIER_NAME'], Pgm_row['SUPPLIER_PLANT'], Pgm_row['AGENT_NAME'], Pgm_row['AGENT_PLANT'], Pgm_row['SUPPLIER_TRADENAME'],'auSpecType', None, None, 'Supplier Reinforcement']
            SP_AU_NEW.loc[len(SP_AU_NEW)] = AU_2

            AU_3 = [Pgm_row['Computed_MATERIAL_CODE'], Pgm_row['REVISION_NO'], 'Supplier', Pgm_row['SUPPLIER_NAME'], Pgm_row['SUPPLIER_PLANT'], Pgm_row['AGENT_NAME'], Pgm_row['AGENT_PLANT'], Pgm_row['SUPPLIER_TRADENAME'],'CodeMaskPrefix', None, None, 'N']
            SP_AU_NEW.loc[len(SP_AU_NEW)] = AU_3




            for field in infoFields:
                II_VALUE = ''
                if (field['IE_SHORT_DESC'] == 'cbMaterialCode') and (Pgm_row['MATERIAL_CODE'] != ''):
                    II_VALUE = Pgm_row['MATERIAL_CODE']
                else:
                    II_VALUE = Pgm_row[field['DB_COLUMN']]
                
                values = [
                    Pgm_row['Computed_MATERIAL_CODE'],
                    Pgm_row['REVISION_NO'],
                    'Supplier',
                    Pgm_row['SUPPLIER_NAME'],
                    Pgm_row['SUPPLIER_PLANT'],
                    Pgm_row['AGENT_NAME'],
                    Pgm_row['AGENT_PLANT'],
                    Pgm_row['SUPPLIER_TRADENAME'],
                    field['IC_CAPTION'],
                    None,
                    field['II_CAPTION'],
                    field['IE_SHORT_DESC'],
                    None,
                    None,
                    II_VALUE
                ]
                SP_II_NEW.loc[len(SP_II_NEW)] = values


            # Populating Program Info infocard
            for field in programInfoFields:
                II_VALUE = None
                values = None
                if field['IE_SHORT_DESC'] in ExceptionList:
                    if field['IE_SHORT_DESC'] == 'ddlSAPStatusDesc':
                        II_VALUE = 'See base specification'
                    elif field['IE_SHORT_DESC'] == 'ieSAPStatusCode':
                        II_VALUE = 'ZZ'
                    elif field['IE_SHORT_DESC'] == 'efEHSApprovalReg':
                        II_VALUE = ''
                    else:
                        II_VALUE = 0
                elif field['IE_SHORT_DESC'] in regions:
                    PGM_REG = PGM_REG_DF[PGM_REG_DF['PGM_ID'] == Pgm_row['PGM_ID']]
                    if field['Conatins'] in PGM_REG['REGION_NAME'].unique():
                        II_VALUE = 1
                    else:
                        II_VALUE = 0
                else:
                    II_VALUE = Pgm_row[field['DB_COLUMN']]
                values = [
                    Pgm_row['Computed_MATERIAL_CODE'],
                    Pgm_row['REVISION_NO'],
                    'Supplier',
                    Pgm_row['SUPPLIER_NAME'],
                    Pgm_row['SUPPLIER_PLANT'],
                    Pgm_row['AGENT_NAME'],
                    Pgm_row['AGENT_PLANT'],
                    Pgm_row['SUPPLIER_TRADENAME'],
                    field['IC_CAPTION'],
                    None,
                    field['II_CAPTION'],
                    field['IE_SHORT_DESC'],
                    None,
                    None,
                    II_VALUE
                ]
                SP_II_NEW.loc[len(SP_II_NEW)] = values
            
            # Populate GPS supplier Approval Review Infocard
            if Pgm_row['SUPPLIER_NAME'] != '':
                IC = [
                    Pgm_row['Computed_MATERIAL_CODE'],
                    Pgm_row['REVISION_NO'],
                    'Supplier',
                    Pgm_row['SUPPLIER_NAME'],
                    Pgm_row['SUPPLIER_PLANT'],
                    Pgm_row['AGENT_NAME'],
                    Pgm_row['AGENT_PLANT'],
                    Pgm_row['SUPPLIER_TRADENAME'],
                    IC_GPS_SUP,
                    None,
                    IP_SHORT_DESC_sup,
                    IC_ORDER_sup
                ]
                SP_IC_DF.loc[len(SP_IC_DF)] = IC
                
            for field in GPSSuppInfoFields:
                II_VALUE = None
                if field['IE_SHORT_DESC'] in regions:
                    PGM_REG = PGM_REG_DF[PGM_REG_DF['PGM_ID'] == Pgm_row['PGM_ID']]
                    if field['Conatins'] in PGM_REG['REGION_NAME'].unique():
                        II_VALUE = 1
                    else:
                        II_VALUE = 0
                elif field['IE_SHORT_DESC'] == 'mlePlantMatProh':
                    plants = str(Pgm_row['RESTRICTED_PLANTS']).split(', ')
                    plants_string = '\n'.join(plants)
                    II_VALUE = plants_string
                elif field['IE_SHORT_DESC'] == 'efFullAppCode':
                    II_VALUE = ''
                else:
                    II_VALUE = Pgm_row[field['DB_COLUMN']]                
                    
                values = [
                    Pgm_row['Computed_MATERIAL_CODE'],
                    Pgm_row['REVISION_NO'],
                    'Supplier',
                    Pgm_row['SUPPLIER_NAME'],
                    Pgm_row['SUPPLIER_PLANT'],
                    Pgm_row['AGENT_NAME'],
                    Pgm_row['AGENT_PLANT'],
                    Pgm_row['SUPPLIER_TRADENAME'],
                    field['IC_CAPTION'],
                    None,
                    field['II_CAPTION'],
                    field['IE_SHORT_DESC'],
                    None,
                    None,
                    II_VALUE
                ]
                SP_II_NEW.loc[len(SP_II_NEW)] = values

            # Populating GPS Full Supplier Approval Review infocard
            if Pgm_row['AGENT_NAME'] != '':
                IC = [
                    Pgm_row['Computed_MATERIAL_CODE'],
                    Pgm_row['REVISION_NO'],
                    'Supplier',
                    Pgm_row['SUPPLIER_NAME'],
                    Pgm_row['SUPPLIER_PLANT'],
                    Pgm_row['AGENT_NAME'],
                    Pgm_row['AGENT_PLANT'],
                    Pgm_row['SUPPLIER_TRADENAME'],
                    IC_AGENT_GPS,
                    None,
                    IP_SHORT_DESC,
                    IC_ORDER
                ]
                SP_IC_DF.loc[len(SP_IC_DF)] = IC
                
                for field in GPSAgentInfoFields:
                    II_VALUE = None
                    if field['IE_SHORT_DESC'] in regions:
                        PGM_REG = PGM_REG_DF[PGM_REG_DF['PGM_ID'] == Pgm_row['PGM_ID']]
                        if field['Conatins'] in PGM_REG['REGION_NAME'].unique():
                            II_VALUE = 1
                        else:
                            II_VALUE = 0
                    elif field['IE_SHORT_DESC'] == 'mlePlantMatProh':
                        plants = str(Pgm_row['RESTRICTED_PLANTS']).split(', ')
                        plants_string = '\n'.join(plants)
                        II_VALUE = plants_string
                    else:
                        II_VALUE = Pgm_row[field['DB_COLUMN']]
                        
                    values = [
                        Pgm_row['Computed_MATERIAL_CODE'],
                        Pgm_row['REVISION_NO'],
                        'Supplier',
                        Pgm_row['SUPPLIER_NAME'],
                        Pgm_row['SUPPLIER_PLANT'],
                        Pgm_row['AGENT_NAME'],
                        Pgm_row['AGENT_PLANT'],
                        Pgm_row['SUPPLIER_TRADENAME'],
                        field['IC_CAPTION'],
                        IC_ORDER,
                        field['II_CAPTION'],
                        field['IE_SHORT_DESC'],
                        None,
                        None,
                        II_VALUE
                    ]
                    SP_II_NEW.loc[len(SP_II_NEW)] = values
            
            # Populating Documents info Card
            PGM_DOC = pd.DataFrame(columns=SP_II_DOCUMENT.keys())

            PGM_Documents = PGM_DOCUMENT_DF[PGM_DOCUMENT_DF['PGM_ID'] == Pgm_row['PGM_ID']]
            for D_index, Document_row in PGM_Documents.iterrows():
                file = Document_row['FILE_NAME']
                if(len(file) > 80):
                    old_file_name = file
                    new_file_name = shorten_filename(file)
                    notFit = [Pgm_row['PGM_ID'], old_file_name, new_file_name]
                    Files_not_fit.loc[len(Files_not_fit)] = notFit
                    file = new_file_name
                
                Document = [
                    Pgm_row['Computed_MATERIAL_CODE'],
                    Pgm_row['REVISION_NO'],
                    'Supplier',
                    Pgm_row['SUPPLIER_NAME'],
                    Pgm_row['SUPPLIER_PLANT'],
                    Pgm_row['AGENT_NAME'],
                    Pgm_row['AGENT_PLANT'],
                    Pgm_row['SUPPLIER_TRADENAME'],
                    IC_DOCUMENTS,
                    None,
                    'General Documents',
                    'docDocuments',
                    None,
                    None,
                    file
                ]

                PGM_DOC.loc[len(PGM_DOC)] = Document  
    if os.path.isfile('Output_CSVs/Documents-Not-Fit.csv'):
        Files_not_fit_df = pd.read_csv('Output_CSVs/Documents-Not-Fit.csv', na_filter=False, escapechar='\\')
        Files_not_fit = pd.concat([Files_not_fit_df, Files_not_fit], ignore_index=True)
    Files_not_fit.to_csv('Output_CSVs/Documents-Not-Fit.csv', index=False, escapechar='\\', doublequote=False)    
    SP_AU_OP_DF = pd.concat([SP_AU_OP_DF, SP_AU_NEW], ignore_index=True)
    SP_AU_OP_DF['SP_VALUE'] = SP_AU_OP_DF['SP_VALUE'].apply(lambda x: x if x.startswith('RF') else 'RF'+x)
    shortencontext(SP_AU_OP_DF).drop_duplicates().to_csv('Output_CSVs/SP-AU.csv', index=False, escapechar='\\', doublequote=False)
    SP_II_DOCUMENT = pd.concat([SP_II_DOCUMENT, PGM_DOC], ignore_index=True)
    SP_II_DOCUMENT['SP_VALUE'] = SP_II_DOCUMENT['SP_VALUE'].apply(lambda x: x if x.startswith('RF') else 'RF'+x)
    SP_II_DOCUMENT = SP_II_DOCUMENT.drop_duplicates()
    SP_II_DOCUMENT.to_csv('Output_CSVs/SP-II-Document.csv', index=False, escapechar='\\', doublequote=False)
    shortencontext(SP_II_DOCUMENT).drop_duplicates().to_csv('Output_CSVs/SP-II-Document.csv', index=False, escapechar='\\', doublequote=False)
    SP_IC_DF['SP_VALUE'] = SP_IC_DF['SP_VALUE'].apply(lambda x: x if x.startswith('RF') else 'RF'+x)
    shortencontext(SP_IC_DF).drop_duplicates().to_csv('Output_CSVs/SP-IC.csv', index=False, escapechar='\\', doublequote=False)
    SP_II_OP_DF = pd.concat([SP_II_OP_DF, SP_II_NEW], ignore_index=True)
    SP_II_OP_DF = SP_II_OP_DF[SP_II_OP_DF['II_CAPTION'] != 'Proposed region for trial']
    SP_II_OP_DF = SP_II_OP_DF[SP_II_OP_DF['II_CAPTION'] != 'Proposed Plant for Trial']
    SP_II_OP_DF['SP_VALUE'] = SP_II_OP_DF['SP_VALUE'].apply(lambda x: x if x.startswith('RF') else 'RF' + x)
    SP_II_OP_DF['IIVALUE'] = SP_II_OP_DF.apply(
    lambda row: row['IIVALUE'] if row['IE_SHORT_DESC'] != 'cbMaterialCode' or str(row['IIVALUE']).startswith('RF') 
    else 'RF' + str(row['IIVALUE']), axis=1)
    shortencontext(SP_II_OP_DF).drop_duplicates().to_csv('Output_CSVs/SP-II.csv', index=False, escapechar='\\', doublequote=False)

    return SP_II_NEW
print('PopulateSpIiforNewSupplierSpecs()')
PopulateSpIiforNewSupplierSpecs()

In [ ]:
import pandas as pd

def UpdateMaterialDescription():
    material_df = pd.read_csv('Output_CSVs/Material.csv', na_filter=False, escapechar='\\')
 
    sp_ii_df = pd.read_csv('Output_CSVs/SP-II.csv', na_filter=False, escapechar='\\')
    
    ef_sapsos_desc_df = sp_ii_df[sp_ii_df['IE_SHORT_DESC'] == 'efSAPSOSDesc']
    
    ef_sapsos_desc_dict = dict(zip(ef_sapsos_desc_df['SP_VALUE'], ef_sapsos_desc_df['IIVALUE']))
    
    cb_material_code_dict = dict(zip(sp_ii_df[sp_ii_df['IE_SHORT_DESC'] == 'cbMaterialCode']['SP_VALUE'], 
                                    sp_ii_df[sp_ii_df['IE_SHORT_DESC'] == 'cbMaterialCode']['IIVALUE']))
    
    material_df['cbMaterialCode'] = material_df['MA_VALUE'].apply(lambda x: cb_material_code_dict.get(x, ''))
    
    material_df['DESCRIPTION'] = material_df['cbMaterialCode'].apply(lambda x: ef_sapsos_desc_dict.get(x, ''))
    
    material_df['DESCRIPTION'] = material_df['DESCRIPTION'].str.slice(0, 80)
    
    material_df = material_df.drop('cbMaterialCode', axis=1)
    
    material_df.to_csv('Output_CSVs/Material.csv', header=True, index=False, escapechar='\\', doublequote=False)
    return material_df

print('UpdateMaterialCsv()')
UpdateMaterialDescription()

In [ ]:
def CreateNewMaterials():
    MAT_DF = pd.read_csv('Output_CSVs/Material.csv', na_filter=False, escapechar='\\')
    SP_DF = pd.read_csv('Output_CSVs/SP.csv', na_filter=False, escapechar='\\')
    SP_II_DF = pd.read_csv('Output_CSVs/SP-II.csv', na_filter=False, escapechar='\\')
    PGM_DF = pd.read_csv('Input_CSVs/PGM_Data_Specs_mapping.csv', na_filter=False, escapechar='\\')
    spv = SP_II_DF.loc[(SP_II_DF['IE_SHORT_DESC']=='cbMaterialCode'), 'SP_VALUE']
    cbmc = SP_II_DF.loc[(SP_II_DF['IE_SHORT_DESC']=='cbMaterialCode'), 'IIVALUE']
    spvdict = dict(zip(spv, cbmc))
    descdict = dict(zip(MAT_DF['MA_VALUE'], MAT_DF['DESCRIPTION']))

    MA_VALUES = MAT_DF['MA_VALUE']
    MAT_NOT_EXIST = SP_DF[~SP_DF['SP_VALUE'].isin(MA_VALUES)]
    MAT_NOT_EXIST = MAT_NOT_EXIST['SP_VALUE'].to_list()

    PGM_DF = PGM_DF[PGM_DF['Computed_MATERIAL_CODE'].isin(MAT_NOT_EXIST)]

    MA_OP_DF = pd.DataFrame(columns=MAT_DF.keys())
    MA_OP_DF['MA_VALUE'] = PGM_DF['Computed_MATERIAL_CODE']
    MA_OP_DF['DESCRIPTION'] = PGM_DF['RMI_DESC'].str.rstrip('\r\n').str.slice(0,80).replace('', np.nan)
    MA_OP_DF['DESCRIPTION'] = MA_OP_DF['DESCRIPTION'].fillna(MA_OP_DF['MA_VALUE'].map(spvdict).map(descdict))
    MA_OP_DF['MA_SOURCE'] = 'RDL'
    MA_OP_DF['ACTIVE'] = 1
    MA_OP_DF['BASE_UOM'] = 'g'
    MA_OP_DF['BASE_CONV_FACTOR'] = None
    MA_OP_DF['BASE_TO_UNIT'] = None
    MA_OP_DF['BASE_QUANTITY'] = None
    MA_OP_DF['DATE_IMPORTED'] = date.today()

    MAT_DF = pd.concat([MAT_DF, MA_OP_DF], ignore_index=True)
    MAT_DF['DESCRIPTION'] = MAT_DF['DESCRIPTION'].replace('', np.nan)
    MAT_DF['DESCRIPTION'] = MAT_DF['DESCRIPTION'].fillna('Empty')
    MAT_DF['MA_SOURCE'] = MAT_DF['MA_SOURCE'].replace('', "RDL")
    MAT_DF = MAT_DF.drop_duplicates()
    MAT_DF.to_csv('Output_CSVs/Material.csv', index=False, escapechar='\\', doublequote=False)
print('CreateNewMaterials()')
CreateNewMaterials()
print('Finished')

In [ ]:
def add_xcodes_from_pgm_documents():
    spau = pd.read_csv('./Output_CSVs/SP-AU.csv', na_filter=False, escapechar='\\')
    spau['VALUE'] = spau['VALUE'].replace('', np.nan)
    spau.to_csv('Output_CSVs/SP-AU_before_x_code_fix.csv', index=False, escapechar='\\', doublequote=False)
    # with open('misdbsecrets.json') as secrets_file:
    #     missecrets = json.load(secrets_file)
    # hostname =  missecrets['hostname']
    # port =   missecrets['port']
    # username = missecrets['username']
    # password = missecrets['password']
    # service_name= missecrets['service_name']
    
    tin = 'MIS_AP_REINFO_PGM_DTL'
    
    dsn = cx_Oracle.makedsn(hostname,port, service_name);
    con = cx_Oracle.connect(username, password, dsn);
    cursor = con.cursor();
    cursor.execute('SELECT * FROM '+str(tin))
    columns = [col[0] for col in cursor.description]
    result = cursor.fetchall()
    rmidf = pd.DataFrame(result, columns=columns)
    cursor.close()
    con.close()
    dbxc = set(rmidf['EXP_CODE'].dropna())
    spxc = set(spau.loc[spau['AU_SHORT_DESC']=='auExpSpecCode', 'VALUE'].dropna())
    missingxc = (dbxc-spxc)
    
    missingpgm = rmidf.loc[rmidf['EXP_CODE'].isin(missingxc), 'PGM_ID'].drop_duplicates()
    tin = 'MIS_AP_PROGRAM_MST'
    
    dsn = cx_Oracle.makedsn(hostname,port, service_name);
    con = cx_Oracle.connect(username, password, dsn);
    cursor = con.cursor();
    cursor.execute('SELECT PGM_ID, APPROVAL_PROCESS_ID,	MATERIAL_GROUP FROM '+str(tin))
    columns = [col[0] for col in cursor.description]
    result = cursor.fetchall()
    pgmdf = pd.DataFrame(result, columns=columns)
    cursor.close()
    con.close()
    
    procdict = {'1': 'NEW SOURCE',
            '2': 'NEW MATERIAL',
            '3': 'OTHER MATERIAL',
            '4': 'AGENT APPROVAL',
            '5': 'REVISIONS',
            '6': 'ADMIN',
            '7': 'REPORTS',
            '8': 'SOURCE CHANGE',
            '9': 'EMERGENCY',
            '10': 'SOS CHANGES'
            }
    pgmdf['APPROVAL_PROCESS'] = pgmdf['APPROVAL_PROCESS_ID'].astype(str).map(procdict)
    
    pgmlist = list(pgmdf.loc[pgmdf['PGM_ID'].isin(missingpgm) & pgmdf['APPROVAL_PROCESS'].isin(['NEW SOURCE', 'NEW MATERIAL', 'OTHER MATERIAL', 'AGENT APPROVAL']), 'PGM_ID'])
    xcdict = dict(zip(rmidf['PGM_ID'].astype(str), rmidf['EXP_CODE'].astype(str)))
    # list of exp codes
    # rmidf.loc[rmidf['PGM_ID'].isin(pgmlist),
    #           ['PGM_ID', 'EXP_CODE', 'REINGMI_DESC', 'MAT_CLASS', 'MAT_SUBCLASS']
    #         ]
    pgmdoc = pd.read_csv('./Output_CSVs/SP-II-Documents(PGM).csv', na_filter=False, escapechar='\\')
    emptyxc = spau[(spau['AU_SHORT_DESC']=='auExpSpecCode') &(spau['VALUE'].isna()) &(spau['CONTEXT']=='Supplier')].copy()
    emptyxc['cstr'] = (emptyxc['SP_VALUE'].fillna('na').astype(str)
                  +emptyxc['SP_VERSION'].fillna('na').astype(str)
                  +emptyxc['CONTEXT'].fillna('na').astype(str)
                  +emptyxc['SP_KEY1'].fillna('na').astype(str)
                  +emptyxc['SP_KEY2'].fillna('na').astype(str)
                  +emptyxc['SP_KEY3'].fillna('na').astype(str)
                  +emptyxc['SP_KEY4'].fillna('na').astype(str)
                  +emptyxc['SP_KEY5'].fillna('na').astype(str))
    pgmdoc['PGM_ID'] = pgmdoc['FILE_NAME'].str.split('_', expand=True)[1].str.lstrip('0')
    pgmdoc['cstr'] = (pgmdoc['SP_VALUE'].fillna('na').astype(str)
                  +pgmdoc['SP_VERSION'].fillna('na').astype(str)
                  +pgmdoc['CONTEXT'].fillna('na').astype(str)
                  +pgmdoc['SP_KEY1'].fillna('na').astype(str)
                  +pgmdoc['SP_KEY2'].fillna('na').astype(str)
                  +pgmdoc['SP_KEY3'].fillna('na').astype(str)
                  +pgmdoc['SP_KEY4'].fillna('na').astype(str)
                  +pgmdoc['SP_KEY5'].fillna('na').astype(str))
    pgmdoc['xccandidate'] = pgmdoc['PGM_ID'].map(xcdict)
    fixer = pgmdoc[pgmdoc['cstr'].isin(emptyxc['cstr']) & ~pgmdoc['xccandidate'].isna()].drop_duplicates(['cstr', 'PGM_ID']).sort_values('PGM_ID')
    mxcdict = dict(zip(fixer['cstr'], fixer['xccandidate']))
    spau['cstr'] = (spau['SP_VALUE'].fillna('na').astype(str)
                  +spau['SP_VERSION'].fillna('na').astype(str)
                  +spau['CONTEXT'].fillna('na').astype(str)
                  +spau['SP_KEY1'].fillna('na').astype(str)
                  +spau['SP_KEY2'].fillna('na').astype(str)
                  +spau['SP_KEY3'].fillna('na').astype(str)
                  +spau['SP_KEY4'].fillna('na').astype(str)
                  +spau['SP_KEY5'].fillna('na').astype(str))
    spau.loc[(spau['AU_SHORT_DESC']=='auExpSpecCode') &(spau['VALUE'].isna()) &(spau['CONTEXT']=='Supplier'),
            'VALUE'] = spau.loc[(spau['AU_SHORT_DESC']=='auExpSpecCode') &(spau['VALUE'].isna()) &(spau['CONTEXT']=='Supplier'),
                                'cstr'].map(mxcdict)
    spau[['SP_VALUE', 'SP_VERSION', 'CONTEXT', 'SP_KEY1', 'SP_KEY2', 'SP_KEY3',
       'SP_KEY4', 'SP_KEY5', 'AU_SHORT_DESC', 'AU_VERSION', 'AUSEQ', 'VALUE']].to_csv('Output_CSVs/SP-AU.csv', index=False)
    # spii = pd.read_csv('Output_CSVs/SP-II.csv', na_filter=False, escapechar='\\')
    # spii['cstr'] = (spii['SP_VALUE'].fillna('na').astype(str)
    #               +spii['SP_VERSION'].fillna('na').astype(str)
    #               +spii['CONTEXT'].fillna('na').astype(str)
    #               +spii['SP_KEY1'].fillna('na').astype(str)
    #               +spii['SP_KEY2'].fillna('na').astype(str)
    #               +spii['SP_KEY3'].fillna('na').astype(str)
    #               +spii['SP_KEY4'].fillna('na').astype(str)
    #               +spii['SP_KEY5'].fillna('na').astype(str))
    # spiifilt = (spii['IE_SHORT_DESC']=='ieInitialExpSpCode') & (spii['CONTEXT']=='Supplier')
    # spaufilt = (spau['AU_SHORT_DESC']=='auExpSpecCode') & (spau['CONTEXT']=='Supplier')
    # spiixc = dict(zip(spii.loc[spiifilt, 'cstr'],
    #                   spii.loc[spiifilt, 'IIVALUE'].fillna('N/A')))
    # spauxc = dict(zip(spau.loc[spaufilt, 'cstr'],
    #                   spau.loc[spaufilt, 'VALUE'].fillna('N/A')))
    # for context in spauxc.keys():
    #     if context in spiixc.keys():
    #         if spiixc[context]!=spauxc[context]:
    #             if spauxc[context]=='N/A':
    #                 spauxc[context]=spiixc[context]
    #             else:
    #                 spiixc[context]=spauxc[context]
    #             print(context, spiixc[context], spauxc[context])
    #     else:
    #         print(context, ' not in spii')
    # spii.loc[spiifilt, 'IIVALUE'] = spii.loc[spiifilt, 'cstr'].map(spiixc).replace('N/A', np.nan)
    # spau.loc[spaufilt, 'VALUE'] = spau.loc[spaufilt, 'cstr'].map(spauxc).replace('N/A', np.nan)
    
    
    # spii[['SP_VALUE', 'SP_VERSION', 'CONTEXT', 'SP_KEY1', 'SP_KEY2', 'SP_KEY3',
    #    'SP_KEY4', 'SP_KEY5', 'IC_CAPTION', 'IC_ORDER', 'II_CAPTION',
    #    'IE_SHORT_DESC', 'IE_VERSION', 'II_ORDER', 'IIVALUE']].to_csv('Output_CSVs/SP-II-iisync.csv', index=False, escapechar='\\', doublequote=False)
    # spau[['SP_VALUE', 'SP_VERSION', 'CONTEXT', 'SP_KEY1', 'SP_KEY2', 'SP_KEY3',
    #    'SP_KEY4', 'SP_KEY5', 'AU_SHORT_DESC', 'AU_VERSION', 'AUSEQ', 'VALUE']].to_csv('Output_CSVs/SP-AU-iisync.csv', index=False, escapechar='\\', doublequote=False)
    return True
add_xcodes_from_pgm_documents()

In [142]:
# import pandas as pd

# SP_OP_DF = pd.read_csv(
#     'Output_CSVs\\SP-II-Documents(PGM).csv'
#     ,na_filter=False, escapechar='\\'
# )

# key_columns = ['SP_KEY1', 'SP_KEY2', 'SP_KEY3', 'SP_KEY4', 'SP_KEY5']
# for col in key_columns:
#     SP_OP_DF[col] = SP_OP_DF[col].astype(str).str[:40]

# SP_OP_DF.to_csv(
#     'Output_CSVs\\SP-II-Documents(PGM).csv'
#     , index=False, escapechar='\\', doublequote=False
# )

In [143]:
# def filter_material_codes(testlist,file_paths,output_files):
   
#     filtered_data=[]
   
    
 
#     for file_path in file_paths:
#         df=pd.read_csv(file_path,na_filter=False, escapechar='\\')
#         column_name=None
#         if 'SP_VALUE' in df.columns:
#             column_name='SP_VALUE'
#         elif 'MA_VALUE' in df.columns:
#             column_name='MA_VALUE'
#         filtered_df=df[df[column_name].isin(testlist)]
 
                  
       
 
#         filtered_data.append(filtered_df)  
   
 
#     for filtered_df,output_file in zip(filtered_data,output_files):
#         if filtered_df is not None:
#             filtered_df=filtered_df.drop_duplicates(keep='first')
#             filtered_df.to_csv(output_file, index=False, escapechar='\\', doublequote=False)
 
 

# # testlist=['RFTV27CN', 'RFNN', 'RFTU01CU', 'RFBH', 'RFWY', 'RF139N', 'RF0047828', 'RF0047235', 'RF396D33', 'RFRD4024']

# # testlist=['RFNN',
# #  'RFBH',
# #  'RF139N',
# #  'RFWY',
# #  'RFTV27CN',
# #  'RF396D33',
# #  'RFTU01CU',
# #  'RF0047235',
# #  'RF0040259',
# #  'RFNC0005',
# #  'RFY00040',
# #  'RF0041806',
# #  'RF0041819',
# #  'RF0041921',
# #  'RF0045665',
# #  'RF0044055',
# #  'RF0043057',
# #  'RFV00144',
# #  'RFW00012',
# #  'RFV00141',
# #  'RF0044054',
# #  'RF0044051',
# #  'RF0044052',
# #  'RF0044053',
# #  'RFV00159',
# #  'RFV00160',
# #  'RFGL3310',
# #  'RFGL3332',
# #  'RFGL2127',
# #  'RFGL3329',
# #  'RFV00163',
# #  'RFV00166',
# #  'RF0047758']

# testlist=[
#     "RFP01L",
#     "RFNT26QA",
#     "RFJ35ZS",
#     "RFB05L",
#     "RF9496",
#     "RFLH23MA",
#     "RFKC15JR",
#     "RFF07Q",
#     "RFCH",
#     "RFNC16KR",
#     "RFNT30UA",
#     "RFNJ27JF",
#     "RFLV19JA",
#     "RFFP",
#     "RFQV28",
#     "RFC17GAZ",
#     "RFS01Q",
#     "RFE09A",
#     "RFBL01WM",
#     "RFQU26PF",
#     "RFQ09R",
#     "RFZ02BI30GF",
#     "RFU07L",
#     "RFQC32PF",
#     "RFLC16SD",
#     "RFS03N",
#     "RFM14LP19JA",
#     "RFTQ26HF",
#     "RFJOAX_FDC",
#     "RFTEST1",
#     "RFRW31SH",
#     "RF40HK2B",
#     "RFZ01NG30GA",
#     "RFFA01WY",
#     "RFFC01FR",
#     "RFN00001",
#     "RFS00004",
#     "RF0040084",
#     "RFGL2060",
#     "RF0040253",
#     "RF0040652",
#     "RFS00047",
#     "RF0041590",
#     "RF0041619",
#     "RF0041582",
#     "RF0041826",
#     "RF0041817",
#     "RF0045664",
#     "RF0043059",
#     "RF0044389",
#     "RFS00090",
#     "RF0046962",
#     "RFW00044"
# ]

# file_paths=['Output_CSVs\\Material.csv','Output_CSVs\\SP.csv','Output_CSVs\\SP-II.csv','Output_CSVs\\SP-IC.csv',
#              'Output_CSVs\\SP-AU.csv','Output_CSVs\\SP-II-Document.csv','Output_CSVs\\SP-II-Documents(PGM).csv','Output_CSVS\\SP-II(GPS).csv','Output_CSVS\\SP-II(Program Info).csv'
#             ]

# output_files=["Filtered_Output_CSVS\\Material.csv","Filtered_Output_CSVS\\SP.csv","Filtered_Output_CSVS\\SP-II.csv","Filtered_Output_CSVS\\SP-IC.csv",
#                   "Filtered_Output_CSVS\\SP-AU.csv","Filtered_Output_CSVS\\SP-II-Document.csv","Filtered_Output_CSVS\\SP-II-Documents(PGM).csv","Filtered_Output_CSVS\\SP-II(GPS).csv",
#                   "Filtered_Output_CSVS\\SP-II(Program Info).csv"]

# filter_material_codes(testlist,file_paths,output_files)

In [144]:
# import pandas as pd
# def replace_na(input_file):
#     df=pd.read_csv(input_file, escapechar='\\', na_filter=False)
#     df.replace('N/A','',inplace=True)
#     df.to_csv(input_file, index=False, escapechar='\\', doublequote=False)
 
# file_path='Filtered_Output_CSVs\\SP-II.csv'
# replace_na(file_path)